# FAERS HS Machine Learning PipelineReproducible notebook for drug-associated hidradenitis suppurativa (HS) signal detection using FAERS pharmacovigilance data.

## Section 0 — Setup & ConfigurationColab-specific setup: mount Drive, copy data for fast I/O, install dependencies.

In [ ]:
# ==============================================================================
# 0A. COLAB SETUP (run once — skip if not on Colab)
# ==============================================================================
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/FAERS Files")
FAERS_DIR = Path("/content/FAERS_Files")

FAERS_DIR.mkdir(parents=True, exist_ok=True)
print("Copying FAERS files from Drive -> /content (faster I/O)...")
os.system(f'cp -r "{DRIVE_DIR}/." "{FAERS_DIR}/"')
print("Using FAERS_DIR =", FAERS_DIR)

In [ ]:
# ==============================================================================
# 0B. INSTALL DEPENDENCIES (run once)
# ==============================================================================
!pip install shap adjustText betacal openpyxl -q

## Section 0C — ImportsAll imports consolidated here for reproducibility.

In [ ]:
# ==============================================================================
# 0C. IMPORTS
# ==============================================================================
import pandas as pd
import numpy as np
import re
import math
import warnings
import pickle
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.stats import norm, fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, brier_score_loss
)
from collections import Counter

import shap

warnings.filterwarnings("ignore", category=FutureWarning)
csv_kwargs = {"engine": "c", "on_bad_lines": "skip"}
print("All imports loaded successfully.")
# --- Additional dependencies ---
try:
    from betacal import BetaCalibration
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "betacal", "-q"])
    from betacal import BetaCalibration

try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "xgboost", "-q"])
    import xgboost as xgb



In [ ]:
# Path fix — paste BELOW 0C Imports, ABOVE Section 1 Data Loading. No re-copy.
import os
from pathlib import Path
_FAERS_ROOT = Path("/content/FAERS_Files")
class _SmartDir:
    def __init__(self, root):
        self.root = str(root); self._idx = {}
        for dp, _, fns in os.walk(self.root):
            for fn in fns:
                self._idx.setdefault(fn.lower(), os.path.join(dp, fn))
    def __truediv__(self, name):
        name = str(name)
        return self._idx.get(name.lower(), os.path.join(self.root, name))
    def __fspath__(self): return self.root
    def __str__(self): return self.root
    def rebuild(self): self.__init__(self.root)
FAERS_DIR = _SmartDir(_FAERS_ROOT)
print("resolver active ->", FAERS_DIR / "sanitized_demographics.csv")
print("resolver active ->", FAERS_DIR / "table_of_indications.csv")

In [ ]:
# =====================================================================
# FAERS DATA DIAGNOSTIC  —  paste into a Colab cell and run BEFORE the pipeline.
# Never raises: every check is guarded. Prints a final verdict + the exact fix.
# =====================================================================
import os, traceback
from pathlib import Path
try:
    import pandas as pd
except Exception:
    os.system("pip -q install pandas"); import pandas as pd

DRIVE_DIR = Path("/content/drive/MyDrive/FAERS Files")   # <- what your notebook uses
FAERS_DIR = Path("/content/FAERS_Files")                 # <- where it copies to / reads from

# filename -> (var_name, all_required_cols, one_of_cols)
REQUIRED = {
 "sanitized_demographics.csv":        ("demo",             ["compositeid","sex","event_dt"], ["age","age_yr"]),
 "sanitized_drug_data.csv":           ("drug",             ["compositeid","role_cod","drug_name","drug_seq"], []),
 "sanitized_outcome_data.csv":        ("outcome",          ["compositeid","outcome_concept_id"], []),
 "sanitized_outcome_dictionary.csv":  ("outcome_dict",     [], []),
 "sanitized_ther.csv":                ("ther",             ["compositeid","drug_seq","start_dt"], []),
 "standard_case_indication.csv":      ("indic",            ["indication_concept_id","indi_drug_seq"], ["primaryid","isr"]),
 "standard_case_outcome_category.csv":("outcome_category", [], []),
 "table_of_indications.csv":          ("indic_table",      ["indication_concept_id"], []),
 "Updated_Drug_Dictionary_Fullnames.csv":("drug_dict",     [], []),
 "drug_tnfi_cohort.csv":              ("tnfi_drug",        ["compositeid","drug_name","role_cod"], []),
}
HS_CONCEPT_ID = 37320281

def hr(t): print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)

# ---------------------------------------------------------------------
hr("1. ENVIRONMENT & MOUNT")
if not Path("/content/drive").exists():
    try:
        from google.colab import drive; drive.mount("/content/drive"); print("  Mounted Google Drive.")
    except Exception as e:
        print("  Could NOT mount Drive:", e)
print("  /content/drive mounted :", Path("/content/drive").exists())
print("  DRIVE_DIR              :", DRIVE_DIR, "| exists:", DRIVE_DIR.exists())
print("  FAERS_DIR              :", FAERS_DIR, "| exists:", FAERS_DIR.exists())

def listdir(p, n=80):
    try: return sorted(os.listdir(p))[:n]
    except Exception as e: return [f"<cannot list: {e}>"]
print("  DRIVE_DIR contents     :", listdir(DRIVE_DIR))
print("  FAERS_DIR contents     :", listdir(FAERS_DIR))

# ---------------------------------------------------------------------
hr("2. LOCATE EACH REQUIRED FILE (recursive, case-insensitive)")
SEARCH_ROOTS = ["/content/FAERS_Files", str(DRIVE_DIR), "/content/drive/MyDrive", "/content"]
MAX_DIRS = 40000   # safety cap so a huge Drive doesn't hang the walk

def build_index(roots):
    """Map lowercased filename -> list of full paths, with a visit cap."""
    idx, seen, visited = {}, set(), 0
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, dns, fns in os.walk(root):
            visited += 1
            if visited > MAX_DIRS:
                print(f"  [!] Walk cap ({MAX_DIRS} dirs) hit; results may be partial."); break
            for fn in fns:
                full = os.path.join(dp, fn)
                if full in seen: continue
                seen.add(full)
                idx.setdefault(fn.lower(), []).append(full)
        else:
            continue
        break
    return idx

index = build_index(SEARCH_ROOTS)
found_paths = {}     # required filename -> chosen path
for name in REQUIRED:
    hits = index.get(name.lower(), [])
    if hits:
        found_paths[name] = hits[0]
        extra = f"  (+{len(hits)-1} more)" if len(hits) > 1 else ""
        print(f"  FOUND    {name:42} -> {hits[0]}{extra}")
    else:
        print(f"  MISSING  {name:42} -> not found anywhere under search roots")

# recommend the directory that holds the most required files
from collections import Counter
dir_cov = Counter(os.path.dirname(p) for p in found_paths.values())
best_dir = dir_cov.most_common(1)[0][0] if dir_cov else None
print(f"\n  Directory with the most required files: {best_dir}"
      f"  ({dir_cov[best_dir] if best_dir else 0}/{len(REQUIRED)})" if best_dir else "\n  No files found.")

# ---------------------------------------------------------------------
hr("3. PER-FILE: SHAPE, COLUMNS, DTYPES, SAMPLE + REQUIRED-COLUMN CHECK")
def read_head(path, nrows=5):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, nrows=nrows, encoding=enc, low_memory=False), enc, None
        except Exception as e:
            last = e
    return None, None, last

def count_rows(path, limit_mb=250):
    try:
        if os.path.getsize(path) > limit_mb * 1e6:
            return f"~ (skipped; {os.path.getsize(path)/1e6:.0f} MB)"
        with open(path, "rb") as f:
            return sum(1 for _ in f) - 1
    except Exception as e:
        return f"<count failed: {e}>"

problems = []
file_cols = {}
for name, (var, req, one_of) in REQUIRED.items():
    print(f"\n--- {name}   (loaded as `{var}`) ---")
    path = found_paths.get(name)
    if not path:
        print("   STATUS: MISSING"); problems.append(f"{name}: file not found"); continue
    print(f"   path : {path}")
    print(f"   size : {os.path.getsize(path)/1e6:.2f} MB | rows : {count_rows(path)}")
    head, enc, err = read_head(path)
    if head is None:
        print(f"   [!] could not read: {err}"); problems.append(f"{name}: unreadable ({err})"); continue
    cols = list(head.columns); file_cols[name] = cols
    print(f"   encoding: {enc}")
    print(f"   columns ({len(cols)}): {cols}")
    print(f"   dtypes  : {dict(head.dtypes.astype(str))}")
    try: print(f"   sample  :\n{head.head(2).to_string(index=False)}")
    except Exception: pass
    low = {c.lower(): c for c in cols}
    miss = [c for c in req if c.lower() not in low]
    if miss:
        print(f"   [!] MISSING required columns: {miss}")
        problems.append(f"{name}: missing columns {miss}")
    if one_of and not any(c.lower() in low for c in one_of):
        print(f"   [!] needs at least one of: {one_of} (none present)")
        problems.append(f"{name}: none of {one_of} present")
    # case/whitespace hints
    for c in req:
        if c.lower() in low and low[c.lower()] != c:
            print(f"   [hint] '{c}' present but named '{low[c.lower()]}' (case/space differs)")

# ---------------------------------------------------------------------
hr("4. CONTENT SANITY CHECKS (values the pipeline depends on)")
def try_check(desc, fn):
    try: print(f"  {desc}: {fn()}")
    except Exception as e: print(f"  {desc}: <check failed: {e}>")

if "sanitized_demographics.csv" in found_paths:
    p = found_paths["sanitized_demographics.csv"]
    try_check("demo age column present (age/age_yr)",
              lambda: [c for c in pd.read_csv(p, nrows=1).columns if c in ("age","age_yr")] or "NONE")
    try_check("demo event_dt sample",
              lambda: pd.read_csv(p, usecols=lambda c: c=="event_dt", nrows=5)["event_dt"].tolist())
if "sanitized_drug_data.csv" in found_paths:
    p = found_paths["sanitized_drug_data.csv"]
    try_check("drug role_cod contains 'PS'",
              lambda: "PS" in set(pd.read_csv(p, usecols=["role_cod"])["role_cod"].dropna().unique()))
if "sanitized_outcome_data.csv" in found_paths:
    p = found_paths["sanitized_outcome_data.csv"]
    try_check(f"outcome has HS concept id {HS_CONCEPT_ID}",
              lambda: bool((pd.read_csv(p, usecols=["outcome_concept_id"])["outcome_concept_id"]==HS_CONCEPT_ID).any()))
if "standard_case_indication.csv" in found_paths:
    p = found_paths["standard_case_indication.csv"]
    try_check(f"indications reference HS concept id {HS_CONCEPT_ID}",
              lambda: bool((pd.read_csv(p, usecols=["indication_concept_id"])["indication_concept_id"]==HS_CONCEPT_ID).any()))
if "table_of_indications.csv" in found_paths:
    p = found_paths["table_of_indications.csv"]
    def _namecol():
        t = pd.read_csv(p, nrows=50)
        return [c for c in t.columns if c != "indication_concept_id" and t[c].dtype == object] or "NONE (no text name column!)"
    try_check("indic_table name column(s) [used for eFig3 labels]", _namecol)

# ---------------------------------------------------------------------
hr("5. VERDICT + EXACT FIX")
if not problems:
    print("  ✅ All required files found and columns look correct.")
    if best_dir and str(FAERS_DIR) != best_dir:
        print(f"  ⚠️  But your files live in:\n        {best_dir}\n     while the notebook uses FAERS_DIR = {FAERS_DIR}")
        print(f"     FIX — set this at the top of the notebook and skip the cp step:\n"
              f"        FAERS_DIR = Path(r'{best_dir}')")
    else:
        print("  You can run the pipeline as-is.")
else:
    print("  ❌ Problems found:")
    for pr in problems: print("     -", pr)
    if best_dir:
        print(f"\n  Most files are in: {best_dir}")
        print(f"  RECOMMENDED FIX — point the notebook there directly (no copy needed):\n"
              f"        from pathlib import Path\n"
              f"        FAERS_DIR = Path(r'{best_dir}')")
        missing_here = [n for n in REQUIRED if os.path.dirname(found_paths.get(n,'')) != best_dir]
        if missing_here:
            print(f"  Files NOT in that folder (locate/copy these): {missing_here}")
    else:
        print("\n  No files found at all. Likely causes:")
        print("   - Google Drive not mounted (run drive.mount).")
        print("   - The Drive folder name differs from 'FAERS Files' (check exact spelling/case above).")
        print("   - Files are inside a subfolder — see the recursive search in section 2.")
print("\nDiagnostic complete.")

## Section 1 — Data LoadingLoad all raw FAERS tables.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_kwargs = {"engine": "c", "on_bad_lines": "skip"}
print("Using pandas C CSV engine")

print("Loading raw FAERS files...")
demo = pd.read_csv(
    FAERS_DIR / "sanitized_demographics.csv",
    dtype={"compositeid": "string", "sex": "category"},
    low_memory=False, **csv_kwargs
)
drug = pd.read_csv(
    FAERS_DIR / "sanitized_drug_data.csv",
    dtype={"compositeid": "string", "role_cod": "category", "drug_name": "string", "drug_seq": "Int32"},
    low_memory=False, **csv_kwargs
)
outcome = pd.read_csv(
    FAERS_DIR / "sanitized_outcome_data.csv",
    dtype={"compositeid": "string", "outcome_concept_id": "Int64"},
    low_memory=False, **csv_kwargs
)
outcome_dict = pd.read_csv(FAERS_DIR / "sanitized_outcome_dictionary.csv", low_memory=False, **csv_kwargs)
ther = pd.read_csv(FAERS_DIR / "sanitized_ther.csv", dtype={"compositeid": "string"}, low_memory=False, **csv_kwargs)
indic = pd.read_csv(
    FAERS_DIR / "standard_case_indication.csv",
    dtype={"primaryid": "string", "isr": "string", "indication_concept_id": "Int64", "indi_drug_seq": "Int32"},
    low_memory=False, **csv_kwargs
)
outcome_category = pd.read_csv(FAERS_DIR / "standard_case_outcome_category.csv", low_memory=False, **csv_kwargs)
indic_table = pd.read_csv(FAERS_DIR / "table_of_indications.csv", dtype={"indication_concept_id": "Int64"}, low_memory=False, **csv_kwargs)
drug_dict = pd.read_csv(FAERS_DIR / "Updated_Drug_Dictionary_Fullnames.csv", low_memory=False, **csv_kwargs)
print("Loaded shapes:")
print("  demo   ", demo.shape)
print("  drug   ", drug.shape)
print("  outcome ", outcome.shape)
print("  indic  ", indic.shape)

In [ ]:
!pip install -q betacal xgboost shap

from google.colab import drive
drive.mount('/content/drive')

import pickle, os
import numpy as np
import pandas as pd
import betacal
import xgboost as xgb

save_dir = "/content/drive/MyDrive/FAERS Files/Checkpoints"

print("Restoring variables from checkpoint...")

df_path = f"{save_dir}/checkpoint_dataframes_v2.pkl"
if os.path.exists(df_path):
    with open(df_path, "rb") as f:
        df_checkpoint = pickle.load(f)
    for name, obj in df_checkpoint.items():
        globals()[name] = obj
    print(f"  ✅ Loaded {len(df_checkpoint)} dataframes")
else:
    print(f"  ❌ Checkpoint not found at {df_path}")

model_path = f"{save_dir}/checkpoint_models_v2.pkl"
if os.path.exists(model_path):
    with open(model_path, "rb") as f:
        model_checkpoint = pickle.load(f)
    for name, obj in model_checkpoint.items():
        if name == "_thresholds":
            for k, v in obj.items():
                globals()[k] = v
        else:
            globals()[name] = obj
    print(f"  ✅ Loaded {len(model_checkpoint)} model objects")
else:
    print(f"  ❌ Checkpoint not found at {model_path}")

# Ensure aliases exist for subsequent cells
if 'iso_model' in globals():
    calibration_model = iso_model
if 'test_preds_isotonic' in globals():
    test_preds_final = test_preds_isotonic

print("\nRestoration complete. Variables are now in global scope.")

## Section 2 — Composite IDs & Cohort Definitions

Builds the `compositeid` key (`pid.`/`isr.`) for the indications table, then defines the HS cohorts: cases (HS as a reported outcome, concept `37320281`) vs. history (HS listed as an indication), and the **drug-worsened** (case + history) vs **drug-induced** (case, no history) strata used throughout.

In [ ]:
# ==============================================================================
# 2. BUILD COMPOSITE IDs FOR INDICATIONS TABLE (robust)
# ==============================================================================
print("Building composite IDs for indications...")

indic = indic.copy()

for col in ["primaryid", "isr"]:
    indic[col] = indic[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}).astype("string")
    indic[col] = indic[col].str.strip().replace({"": pd.NA})
    indic[col] = indic[col].str.replace(r"\.0$", "", regex=True)

indications_compositeid = indic.copy()
indications_compositeid["compositeid"] = pd.Series(pd.NA, index=indications_compositeid.index, dtype="string")

mask_pid = indications_compositeid["primaryid"].notna()
mask_isr = (~mask_pid) & indications_compositeid["isr"].notna()

indications_compositeid.loc[mask_pid, "compositeid"] = "pid." + indications_compositeid.loc[mask_pid, "primaryid"]
indications_compositeid.loc[mask_isr, "compositeid"] = "isr." + indications_compositeid.loc[mask_isr, "isr"]

print(f"  indications_compositeid: {indications_compositeid.shape}")
print(f"  NAs in compositeid: {indications_compositeid['compositeid'].isna().sum()}")
print(f"  Example IDs: {indications_compositeid['compositeid'].dropna().head(5).tolist()}")

# ==============================================================================
# 3. DEFINE COHORTS (FAST)
# ==============================================================================
print("Building cohorts (fast)...")
HS_CONCEPT_ID = 37320281

ids_case_true = set(
    outcome.loc[outcome["outcome_concept_id"].eq(HS_CONCEPT_ID), "compositeid"].dropna().unique()
)
ids_history_true = set(
    indications_compositeid.loc[indications_compositeid["indication_concept_id"].eq(HS_CONCEPT_ID), "compositeid"].dropna().unique()
)

print(f"  HS cases (outcome): {len(ids_case_true)}")
print(f"  HS history (indication): {len(ids_history_true)}")
print(f"  Overlap (case & history): {len(ids_case_true & ids_history_true)}")

## Section 3 — ROR Statistics (Haldane-corrected + BH/FDR)

Computes Haldane-corrected reporting odds ratios (RORs) with Wald 95% CIs and Benjamini-Hochberg FDR-adjusted P values for each drug within HS-history × sex strata → **`stats_combined`**, the master disproportionality table feeding the volcano and mirrored-signal plots.

In [ ]:
# ==============================================================================
# 4. ROR STATISTICS (Haldane-corrected + BH/FDR within subgroup)
# ==============================================================================
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

print("Computing ROR statistics...")

dt_drug_optimized = (
    drug.loc[drug["role_cod"].eq("PS"), ["compositeid", "drug_name"]]
    .dropna(subset=["compositeid", "drug_name"])
    .drop_duplicates()
)

ids_F = set(demo.loc[demo["sex"].astype(str).eq("F"), "compositeid"].dropna().unique())
ids_M = set(demo.loc[demo["sex"].astype(str).eq("M"), "compositeid"].dropna().unique())

ids_F_NoHist = ids_F - ids_history_true
ids_M_NoHist = ids_M - ids_history_true
ids_F_Hist = ids_F & ids_history_true
ids_M_Hist = ids_M & ids_history_true

def run_batch(ids, hist_lbl, sex_lbl):
    ids = set(ids)

    cohort = dt_drug_optimized.loc[
        dt_drug_optimized["compositeid"].isin(ids)
    ].copy()

    cohort["is_case"] = cohort["compositeid"].isin(ids_case_true)

    N_db = cohort["compositeid"].nunique()
    N_cases = cohort.loc[cohort["is_case"], "compositeid"].nunique()

    stats = (
        cohort.groupby("drug_name", sort=False)
        .agg(
            a=("is_case", "sum"),
            total_users=("compositeid", "nunique"),
        )
        .reset_index()
    )

    stats = stats.loc[stats["a"] >= 3].copy()
    stats["N_db"] = N_db
    stats["N_cases"] = N_cases
    stats["history_group"] = hist_lbl
    stats["sex_group"] = sex_lbl

    return stats

# Run subgroup analyses
res_F_no = run_batch(ids_F_NoHist, "No HS Indication", "Female")
res_M_no = run_batch(ids_M_NoHist, "No HS Indication", "Male")
res_T_no = run_batch(ids_F_NoHist | ids_M_NoHist, "No HS Indication", "Total")

res_F_hi = run_batch(ids_F_Hist, "Has HS Indication", "Female")
res_M_hi = run_batch(ids_M_Hist, "Has HS Indication", "Male")
res_T_hi = run_batch(ids_F_Hist | ids_M_Hist, "Has HS Indication", "Total")

stats_combined = pd.concat(
    [res_F_no, res_M_no, res_T_no, res_F_hi, res_M_hi, res_T_hi],
    ignore_index=True
)

# 2x2 table counts
stats_combined["b"] = stats_combined["total_users"] - stats_combined["a"]
stats_combined["c"] = stats_combined["N_cases"] - stats_combined["a"]
stats_combined["d"] = (
    (stats_combined["N_db"] - stats_combined["N_cases"]) - stats_combined["b"]
)

# Haldane correction only when any cell is zero
needs_cor = stats_combined[["a", "b", "c", "d"]].eq(0).any(axis=1)

for col in ["a", "b", "c", "d"]:
    stats_combined[f"{col}_adj"] = np.where(
        needs_cor,
        stats_combined[col] + 0.5,
        stats_combined[col]
    )

# ROR + SE + CI + raw p
stats_combined["ror"] = (
    stats_combined["a_adj"] * stats_combined["d_adj"]
) / (
    stats_combined["b_adj"] * stats_combined["c_adj"]
)

stats_combined["ror_se"] = np.sqrt(
    1 / stats_combined["a_adj"] +
    1 / stats_combined["b_adj"] +
    1 / stats_combined["c_adj"] +
    1 / stats_combined["d_adj"]
)

log_ror = np.log(stats_combined["ror"])
z = log_ror / stats_combined["ror_se"]

stats_combined["ror_ci_lower"] = np.exp(log_ror - 1.96 * stats_combined["ror_se"])
stats_combined["ror_ci_upper"] = np.exp(log_ror + 1.96 * stats_combined["ror_se"])
stats_combined["p_value"] = 2 * norm.sf(np.abs(z))

# BH/FDR correction within each subgroup
# subgroup = each unique history_group x sex_group combination
stats_combined["p_fdr_bh"] = (
    stats_combined
    .groupby(["history_group", "sex_group"])["p_value"]
    .transform(lambda p: multipletests(p, method="fdr_bh")[1])
)

print(f"  stats_combined: {stats_combined.shape}")

## Section 4 — Primary Suspect Indications (No-History group)

Extracts and ranks the recorded indications for the primary-suspect drug among drug-induced (no-HS-history) reports → `matched_indications` / `final_indication_list` (input to eFigure 3).

In [ ]:
# ==============================================================================
# 5. PRIMARY SUSPECT INDICATIONS (No History group)
# ==============================================================================
print("Building primary suspect indications for No-History group...")

target_ids = ids_case_true - ids_history_true

ps_drugs_details = drug.loc[
    drug["compositeid"].isin(target_ids) & drug["role_cod"].eq("PS"),
    ["compositeid", "drug_seq", "drug_name"]
].dropna(subset=["compositeid", "drug_seq", "drug_name"])

matched_indications = ps_drugs_details.merge(
    indications_compositeid,
    left_on=["compositeid", "drug_seq"],
    right_on=["compositeid", "indi_drug_seq"],
    how="inner"
)[["compositeid", "drug_name", "indication_concept_id"]].dropna(subset=["indication_concept_id"])

indication_counts = (
    matched_indications.groupby("indication_concept_id", sort=False)
    .size()
    .reset_index(name="N_Indication")
    .sort_values("N_Indication", ascending=False)
)

final_indication_list = indication_counts.merge(indic_table, on="indication_concept_id", how="left")
final_indication_list["Percent"] = final_indication_list["N_Indication"] / max(len(target_ids), 1) * 100
final_indication_list["Label"] = (
    final_indication_list["N_Indication"].astype(str) + " (" +
    final_indication_list["Percent"].round(1).astype(str) + "%)"
)
final_indication_list = final_indication_list.sort_values("N_Indication", ascending=False)

final_indication_list.to_excel("Primary_Suspect_Indications_NoHist.xlsx", index=False)
print(f"  Exported: Primary_Suspect_Indications_NoHist.xlsx")
print(f"  Target IDs (case, no history): {len(target_ids)}")

## Section 5 — Comorbidity Matrix

Maps curated OMOP concept-ID sets (eTable 1) onto each report's indication fields to build per-report comorbidity flags across all 14 comorbidity categories (`comor_lookup` + comorbidity matrix).

In [ ]:
# ==============================================================================
# 6. COMORBIDITY MATRIX
# ==============================================================================
print("Building comorbidity matrix...")

comor_defs = {
    "Smoking": [37522261, 36919130, 37420593, 788265],
    "Diabetes_T2": [35506609, 35506622, 35506632, 35532172, 35532077, 35506521, 35506537, 35506506],
    "Metabolic_Obesity": [36416534, 37522010, 36467967, 36416535, 36416526, 36416682, 35506634, 36617314],
    "Depression_Anxiety": [36978967, 36918942, 36918950, 36978958, 788432, 36919033, 36919058, 36818810, 36978953, 36919235, 36919149, 36918853, 36918905, 36978773, 36918911, 36918909, 36918860, 36918906, 36919054],
    "Follicular_Tetrad": [37384215, 36110675, 37320258],
    "PCOS": [35532028, 35506596, 35506582, 35506560, 35531939, 35531986],
    "IBD": [35708073, 35708063, 35708044, 35737791, 35737911, 35708072, 35708091],
    "Arthropathies": [36009797, 42893183, 36516787, 36516828, 36009833, 36516788, 35708048, 36516977, 36516786],
    "Hyperlipidemia": [36468282, 36416641, 36416634, 36468259, 36314073, 36468272, 36313985, 36313982, 36361931, 35329545],
    "Psoriasis": [37320205],
    "Dermatologic_Conditions": [37320257, 36009768, 36009851, 36110026, 37320260, 36009735, 37320141, 37320143, 37320176, 37320203, 37320318],
    "Multiple_Sclerosis": [36718111, 36718116, 1197926],
    "Malignancy_Hematologic": [35104351, 35104339, 35104349, 35104378, 35104397, 35104465, 35104252, 35104667, 35104235, 35104242, 35104336, 35104341, 35104342, 35104354, 35104392, 35104405, 35104420, 35104618, 35124346],
    "Malignancy_Solid": [35708417, 36617647, 35708539, 36617162, 36617920, 35506808, 36617488, 36617644, 36617375, 36617537, 36617657, 36617828, 36617836, 43053818, 43054036, 45886111],
}

comor_lookup = pd.DataFrame(
    [{"concept_id": cid, "category": cat} for cat, cids in comor_defs.items() for cid in cids]
).astype({"concept_id": "Int64", "category": "string"})

comor_long = (
    indications_compositeid.loc[
        indications_compositeid["indication_concept_id"].isin(comor_lookup["concept_id"]) &
        indications_compositeid["compositeid"].notna(),
        ["compositeid", "indication_concept_id"]
    ]
    .merge(comor_lookup, left_on="indication_concept_id", right_on="concept_id", how="inner")
    [["compositeid", "category"]]
    .drop_duplicates()
)

comor_matrix = (
    comor_long.assign(flag=1)
    .pivot_table(index="compositeid", columns="category", values="flag", fill_value=0)
    .reset_index()
)
comor_matrix.columns = ["compositeid"] + [f"Hist_{c}" for c in comor_matrix.columns[1:]]
comor_matrix.columns.name = None

print(f"  comor_matrix: {comor_matrix.shape}")

## Section 6 — Demographics Standardization

Standardizes demographics into model-ready fields (`demo_std`): numeric age, a 9-level `age_group`, missingness indicators (`Age_missing`, `Sex_Unknown`), and parsed event dates.

In [ ]:
# ==============================================================================
# 7. STANDARDIZE DEMOGRAPHICS (demo_std)
# ==============================================================================
print("Standardizing demographics...")

age_col = "age" if "age" in demo.columns else ("age_yr" if "age_yr" in demo.columns else None)
if age_col is None:
    raise ValueError("demo missing age column. Expected one of: age, age_yr")
print(f"  Resolved age column: {age_col}")

demo_std = demo[["compositeid", age_col, "sex", "event_dt"]].copy()
demo_std = demo_std.rename(columns={age_col: "age"})
demo_std["age"] = pd.to_numeric(demo_std["age"], errors="coerce")

print(f"  demo_std: {demo_std.shape}")
print(f"  Age summary:\n{demo_std['age'].describe()}")
print(f"  Sex values: {sorted(demo_std['sex'].astype(str).unique())}")

## Section 7 — Audit Checks

Consistency/sanity audits on the assembled cohorts and fields (row counts, key coverage, case/history overlap) before modeling.

In [ ]:
# ==============================================================================
# 8. AUDIT CHECKS
# ==============================================================================
print("=" * 60)
print("AUDIT SUMMARY")
print("=" * 60)

print(f"  ids_case_true: {len(ids_case_true)} unique")
print(f"  ids_history_true: {len(ids_history_true)} unique")
print(f"  demo_std: {demo_std.shape[0]} rows, {demo_std.shape[1]} cols")
print(f"  comor_matrix: {comor_matrix.shape[0]} rows, {comor_matrix.shape[1]} cols")
print(f"  Hist_ columns: {sum(1 for c in comor_matrix.columns if c.startswith('Hist_'))}")

demo_ids = set(demo_std["compositeid"].dropna().unique())
comor_ids = set(comor_matrix["compositeid"].dropna().unique())

print(f"\n  ids_case_true in demo: {100 * len(ids_case_true & demo_ids) / max(len(ids_case_true), 1):.1f}%")
print(f"  ids_case_true in comor_matrix: {100 * len(ids_case_true & comor_ids) / max(len(ids_case_true), 1):.1f}%")
print(f"  ids_history_true in demo: {100 * len(ids_history_true & demo_ids) / max(len(ids_history_true), 1):.1f}%")

evt = demo_std["event_dt"].astype(str).replace({"nan": "", "None": "", "<NA>": ""})
evt = evt[evt != ""]
is_8digit = evt.str.match(r"^\d{8}$")
print(f"  event_dt 8-digit format: {100 * is_8digit.mean() if len(is_8digit) else 0:.1f}% (among non-missing)")

print("\n✅ Upstream data prep complete.")
print("Objects ready: demo_std, ids_case_true, ids_history_true, comor_matrix, stats_combined")

## Section 8 — Cohort Definition & Feature Engineering

Assembles the modeling frame **`model_df_full`** over all TNFi-exposed reports: HS outcome label, TNFi-agent flags, indication-group flags, comorbidity and top concomitant-drug indicators, and standardized demographics (`tnfi_drug` / `drug_patient_counts` support the drug features).

In [ ]:
# ==============================================================================
# SECTIONS 1–2: COHORT DEFINITION & FEATURE ENGINEERING
# ==============================================================================
# Requires: demo_std, ids_case_true, ids_history_true, comor_matrix, drug
# Produces: model_df_full, model_df_model
# ==============================================================================
import re

TNF_INHIBITORS = [
    "adalimumab", "infliximab", "etanercept",
    "golimumab", "certolizumab pegol",
]

def make_clean_name(s):
    """Equivalent to R's make.names()."""
    s = str(s).strip()
    s = re.sub(r"[^A-Za-z0-9.]", ".", s)
    s = re.sub(r"\.+", ".", s).strip(".")
    if s and s[0].isdigit():
        s = "X" + s
    return s

# ------------------------------------------------------------------------------
# 1. COHORT DEFINITION
# ------------------------------------------------------------------------------
print("=" * 60)
print("SECTION 1 — COHORT DEFINITION")
print("=" * 60)

tnfi_drug = pd.read_csv(
    FAERS_DIR / "drug_tnfi_cohort.csv",
    dtype={"compositeid": "string", "drug_name": "string", "role_cod": "category"},
    low_memory=False
)
print(f"tnfi_drug: {len(tnfi_drug):,} rows")

tnf_user_ids = set(tnfi_drug["compositeid"].unique())
target_cohort_ids = tnf_user_ids - ids_history_true
print(f"Target cohort: {len(target_cohort_ids):,} TNFi users after excluding HS indication")

tnfi_drug = tnfi_drug[tnfi_drug["compositeid"].isin(target_cohort_ids)].copy()

# ------------------------------------------------------------------------------
# 2. FEATURE MATRICES
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SECTION 2 — FEATURE MATRICES")
print("=" * 60)

cohort_size = len(target_cohort_ids)
min_count_1pct = cohort_size * 0.01

# --- A. Concomitant drugs (1% prevalence filter) ---
print("\n--- A. Concomitant drugs ---")

concom = tnfi_drug[~tnfi_drug["drug_name"].isin(TNF_INHIBITORS)].copy()
concom["drug_name_clean"] = "Drug_" + concom["drug_name"].str.strip().apply(make_clean_name)

drug_patient_counts = concom.groupby("drug_name_clean")["compositeid"].nunique()
drugs_passing = drug_patient_counts[drug_patient_counts >= min_count_1pct].index
print(f"  Drugs passing 1% threshold: {len(drugs_passing)}")

drug_features = (
    concom[concom["drug_name_clean"].isin(drugs_passing)][["compositeid", "drug_name_clean"]]
    .drop_duplicates()
    .assign(value=1)
    .pivot_table(index="compositeid", columns="drug_name_clean", values="value", fill_value=0)
    .reset_index()
)
drug_features.columns.name = None
print(f"  drug_features: {drug_features.shape}")

del concom

# --- B. TNFi identity ---
print("\n--- B. TNFi identity ---")

tnfi_ps = tnfi_drug[
    (tnfi_drug["drug_name"].isin(TNF_INHIBITORS)) &
    (tnfi_drug["role_cod"] == "PS")
].copy()
tnfi_ps["tnfi_col"] = "TNFi_" + tnfi_ps["drug_name"].str.strip().apply(make_clean_name)

tnfi_identity = (
    tnfi_ps[["compositeid", "tnfi_col"]]
    .drop_duplicates()
    .assign(value=1)
    .pivot_table(index="compositeid", columns="tnfi_col", values="value", fill_value=0)
    .reset_index()
)
tnfi_identity.columns.name = None
print(f"  tnfi_identity: {tnfi_identity.shape}")

del tnfi_drug, tnfi_ps

# --- C. Demographics subset ---
print("\n--- C. Demographics ---")

demo_target = demo_std[demo_std["compositeid"].isin(target_cohort_ids)][
    ["compositeid", "age", "sex", "event_dt"]
].copy()
print(f"  demo_target: {demo_target.shape}")

# --- D. Assemble model dataframe ---
print("\n--- D. Assembling model dataframe ---")

model_df_full = pd.DataFrame({"compositeid": list(target_cohort_ids)})
model_df_full["outcome_hs"] = np.where(
    model_df_full["compositeid"].isin(ids_case_true), "Yes", "No"
)

# Join demographics
model_df_full = model_df_full.merge(demo_target, on="compositeid", how="left")
del demo_target

# Parse report year (event_dt is float like 20030815.0)
model_df_full["event_dt_str"] = model_df_full["event_dt"].astype(str).str.strip()
model_df_full["report_year"] = np.where(
    model_df_full["event_dt_str"].str.match(r"^\d{8}\.0$"),
    model_df_full["event_dt_str"].str[:4].astype(float),
    np.where(
        model_df_full["event_dt_str"].str.match(r"^\d{8}$"),
        model_df_full["event_dt_str"].str[:4].astype(float),
        np.nan
    )
)
model_df_full.drop(columns=["event_dt", "event_dt_str"], inplace=True)

# Join comorbidity matrix (with overlap guard)
overlap_cols = set(model_df_full.columns) & set(comor_matrix.columns) - {"compositeid"}
if overlap_cols:
    print(f"  Dropping overlapping columns from comor_matrix: {overlap_cols}")
    comor_clean = comor_matrix.drop(columns=list(overlap_cols))
else:
    comor_clean = comor_matrix
model_df_full = model_df_full.merge(comor_clean, on="compositeid", how="left")
del comor_clean

# Join drug features
model_df_full = model_df_full.merge(drug_features, on="compositeid", how="left")
del drug_features

# Join TNFi identity
model_df_full = model_df_full.merge(tnfi_identity, on="compositeid", how="left")
del tnfi_identity

print(f"  model_df_full after all joins: {model_df_full.shape}")

# --- E. Handle missingness ---
print("\n--- E. Handling missingness ---")

UNKNOWN_SEX = {"", "UNK", "UNKNOWN", "NS", "I", "P", "T", "NAN", "NONE"}
model_df_full["sex_clean"] = model_df_full["sex"].astype(str).str.upper().str.strip()
model_df_full["sex_clean"] = model_df_full["sex_clean"].where(
    ~model_df_full["sex_clean"].isin(UNKNOWN_SEX), "UNKNOWN"
)
model_df_full["Sex_F"] = (model_df_full["sex_clean"] == "F").astype(int)
model_df_full["Sex_M"] = (model_df_full["sex_clean"] == "M").astype(int)
model_df_full["Sex_Unknown"] = (~model_df_full["sex_clean"].isin(["F", "M"])).astype(int)
model_df_full.drop(columns=["sex", "sex_clean"], inplace=True)

# Age: median imputation + missingness indicator
n_age_missing = model_df_full["age"].isna().sum()
pct_age_missing = 100 * n_age_missing / len(model_df_full)
print(f"  Age missing: {n_age_missing:,} rows ({pct_age_missing:.1f}%)")

median_age = model_df_full["age"].median()
model_df_full["Age_missing"] = model_df_full["age"].isna().astype(int)
model_df_full["age"].fillna(median_age, inplace=True)

# --- F. Fill NAs in binary columns ---
bin_cols = [c for c in model_df_full.columns if c.startswith(("Drug_", "Hist_", "TNFi_"))]
na_total = model_df_full[bin_cols].isna().sum().sum()
print(f"  Binary cols: {len(bin_cols)} | Total NAs to fill: {na_total:,}")
model_df_full[bin_cols] = model_df_full[bin_cols].fillna(0).astype(int)

# Report year missingness
n_year_missing = model_df_full["report_year"].isna().sum()
print(f"  report_year missing: {n_year_missing:,} ({100 * n_year_missing / len(model_df_full):.1f}%)")

# --- G. Safety checks ---
xy_cols = [c for c in model_df_full.columns if c.endswith((".x", ".y", "_x", "_y"))]
if xy_cols:
    print(f"  ⚠️  Merge suffix columns: {xy_cols[:10]}")
else:
    print("  ✅ No merge suffix columns.")

# --- H. Build model-ready dataframe ---
model_df_model = model_df_full.drop(columns=["compositeid", "report_year"])

# Ensure unique column names
if model_df_model.columns.duplicated().any():
    model_df_model = model_df_model.loc[:, ~model_df_model.columns.duplicated()]

n_yes = (model_df_model["outcome_hs"] == "Yes").sum()
n_no = (model_df_model["outcome_hs"] == "No").sum()
print(f"\n  Class balance: {n_yes} Yes ({100 * n_yes / (n_yes + n_no):.2f}%) / "
      f"{n_no} No ({100 * n_no / (n_yes + n_no):.2f}%)")
print(f"  model_df_full: {model_df_full.shape}")
print(f"  model_df_model: {model_df_model.shape}")
print(f"  Memory: {model_df_full.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print("\n✅ Sections 1–2 complete.")

## Section 9 — Train/Test Split & Feature Selection (10-Fold CV)

70/30 stratified split, then 10-fold cross-validated, RF-ranked feature selection (features retained in ≥5/10 folds) → 29 predictors; **`feature_stability`** records per-feature fold-selection frequency (manuscript eTable 12).

In [ ]:
# ==============================================================================
# SECTIONS 3–5: CASE WEIGHTS, TRAIN/TEST SPLIT, FEATURE SELECTION
# ==============================================================================
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from collections import Counter
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ==============================================================================
# 3. CASE WEIGHTS FUNCTION
# ==============================================================================
def compute_sample_weights(y, cap=50):
    """Return per-sample weights: minority class gets weight = min(N_maj/N_min, cap)."""
    n_yes = (y == 1).sum()
    n_no = (y == 0).sum()
    if n_yes == 0:
        raise ValueError("No positive cases — cannot compute weights.")
    ratio_raw = n_no / n_yes
    ratio = min(ratio_raw, cap)
    print(f"  Case weight ratio: {ratio:.1f} (raw {ratio_raw:.1f}, cap {cap})")
    return np.where(y == 1, ratio, 1.0)

# ==============================================================================
# 4. TRAIN / TEST SPLIT (70/30 Stratified)
# ==============================================================================
print("=" * 60)
print("SECTION 4 — TRAIN / TEST SPLIT")
print("=" * 60)

feature_cols = [c for c in model_df_model.columns if c != "outcome_hs"]
X = model_df_model[feature_cols].values
y = (model_df_model["outcome_hs"] == "Yes").astype(int).values
feature_names = feature_cols

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=123, stratify=y
)

print(f"Train: {len(y_train)} ({y_train.sum()} Yes) | Test: {len(y_test)} ({y_test.sum()} Yes)")

# ==============================================================================
# 5. FEATURE SELECTION (10-Fold CV — Full Training Data, No Subsampling)
# ==============================================================================
# CHANGES FROM R (memory-constrained) VERSION:
#   - No subsampling of controls (was capped at 100K in R)
#   - n_estimators=500 (was 100 in R; 500 is standard for stable importance)
#   - min_samples_leaf=5 (was 20 in R; 1-5 is standard in literature)
#   - n_jobs=-1 (was 1 in R to avoid memory spikes)
#   - K_TOP=30 and STABILITY_THRESHOLD=5 unchanged (methodologically sound)
# ==============================================================================
print("\n" + "=" * 60)
print("SECTION 5 — FEATURE SELECTION (10-Fold CV)")
print("=" * 60)

K_TOP = 30
STABILITY_THRESHOLD = 5

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=456)
top_features_per_fold = []

for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"  Fold {fold_i}/10 ...", end=" ")

    X_fold = X_train[train_idx]
    y_fold = y_train[train_idx]
    w_fold = compute_sample_weights(y_fold)

    rf = RandomForestClassifier(
        n_estimators=500,        # was 100 (memory); 500 standard for stable rankings
        max_features="sqrt",     # standard for classification RF
        min_samples_leaf=5,      # was 20 (memory); 1-5 is literature standard
        max_depth=None,          # fully grown trees (R default); was implicitly limited
        class_weight=None,       # using manual sample_weight instead
        random_state=1000 + fold_i,
        n_jobs=-1,               # was 1 in R; use all cores
        oob_score=True,
    )
    rf.fit(X_fold, y_fold, sample_weight=w_fold)

    importances = rf.feature_importances_
    top_k_idx = np.argsort(importances)[::-1][:K_TOP]
    top_k_names = [feature_names[i] for i in top_k_idx]
    top_features_per_fold.append(top_k_names)

    print(f"OOB accuracy: {rf.oob_score_:.3f}")
    del rf, X_fold, y_fold, w_fold

# --- Stability selection ---
all_selected = [feat for fold_list in top_features_per_fold for feat in fold_list]
feature_counts = Counter(all_selected)

stable_features = sorted(
    [feat for feat, count in feature_counts.items() if count >= STABILITY_THRESHOLD],
    key=lambda f: -feature_counts[f]
)

if len(stable_features) == 0:
    raise ValueError(
        "No stable features selected. Consider lowering STABILITY_THRESHOLD "
        "or increasing K_TOP."
    )

print(f"\nStable features selected: {len(stable_features)} "
      f"(of {len(feature_names)} total predictors)")

feature_stability = pd.DataFrame([
    {"feature": feat, "folds_selected": count}
    for feat, count in feature_counts.items()
]).sort_values(["folds_selected", "feature"], ascending=[False, True]).reset_index(drop=True)

print("\nTop 15 most stable features:")
print(feature_stability.head(15).to_string(index=False))

# --- Subset train/test to stable features ---
stable_idx = [feature_names.index(f) for f in stable_features]
X_train_sel = X_train[:, stable_idx]
X_test_sel = X_test[:, stable_idx]

print(f"\nX_train_sel: {X_train_sel.shape}")
print(f"X_test_sel:  {X_test_sel.shape}")
print("\n✅ Sections 3–5 complete. Ready for final model training.")

## Section 10 — Final Model Training & Permutation Importance

Fits the final class-weighted random forest with out-of-fold Platt scaling and computes test-set **`perm_importance`** (AUROC-drop).

In [ ]:
# ==============================================================================
# SECTIONS 6–7: FINAL MODEL TRAINING + OUT-OF-FOLD PLATT SCALING
# ==============================================================================
# CHANGES FROM R (memory-constrained) VERSION:
#   Section 6:
#   - n_estimators=1000 (was 150; 500-1000 standard for probability forests)
#   - min_samples_leaf=5 (was 25; 1-5 is literature standard)
#   - No fallback subsampling — full training set used directly
#   - No separate permutation-importance model — sklearn computes it natively
#   - n_jobs=-1 (was 1)
#   Section 7:
#   - No subsampling of calibration set — full training data used
#   - Calibration RF uses 500 trees (was 100) and min_samples_leaf=5 (was 20)
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

# ==============================================================================
# 6. FINAL MODEL TRAINING (full training set, selected features)
# ==============================================================================
print("=" * 60)
print("SECTION 6 — FINAL MODEL TRAINING")
print("=" * 60)

print(f"X_train_sel: {X_train_sel.shape}")
print(f"X_test_sel:  {X_test_sel.shape}")

# --- A. Final predictive model (full training set) ---
train_weights = compute_sample_weights(y_train)

final_rf = RandomForestClassifier(
    n_estimators=500,       # was 150 (memory); 500-1000 standard for stable probabilities
    max_features="sqrt",
    min_samples_leaf=5,      # was 25 (memory); 1-5 is literature standard
    max_depth=None,          # fully grown trees
    random_state=789,
    n_jobs=-1,               # was 1 in R
    oob_score=True,
)
final_rf.fit(X_train_sel, y_train, sample_weight=train_weights)

print(f"Final model OOB accuracy: {final_rf.oob_score_:.4f}")
print(f"Final model trained on full training set ({len(y_train):,} rows)")

# --- B. Permutation importance (full training set, no separate subsampled model) ---
print("\nComputing permutation importance (this may take a few minutes)...")

perm_result = permutation_importance(
    final_rf,
    X_test_sel,          # evaluate on TEST set (honest importance)
    y_test,
    n_repeats=10,        # was effectively 1 in R's ranger; 10 is standard
    random_state=792,
    n_jobs=-1,
    scoring="roc_auc",   # importance measured by AUC drop, not accuracy
)

perm_importance = pd.DataFrame({
    "feature": stable_features,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("\nTop 20 features by permutation importance (AUC drop):")
print(perm_importance.head(20).to_string(index=False))


# ==============================================================================
# 7. OUT-OF-FOLD PLATT SCALING (honest calibration)
# ==============================================================================
# Full training data, no subsampling. 5-fold OOF predictions to avoid
# information leakage in calibration.
# ==============================================================================
print("\n" + "=" * 60)
print("SECTION 7 — OUT-OF-FOLD PLATT SCALING")
print("=" * 60)

oof_probs = np.full(len(y_train), np.nan)

skf_cal = StratifiedKFold(n_splits=5, shuffle=True, random_state=800)

for fold_i, (train_idx, val_idx) in enumerate(skf_cal.split(X_train_sel, y_train), 1):
    print(f"  Platt fold {fold_i}/5 ...", end=" ")

    X_cal_train = X_train_sel[train_idx]
    y_cal_train = y_train[train_idx]
    X_cal_val = X_train_sel[val_idx]

    w_cal = compute_sample_weights(y_cal_train)

    rf_cal = RandomForestClassifier(
        n_estimators=500,        # was 100 (memory); 500 for stable calibration scores
        max_features="sqrt",
        min_samples_leaf=5,      # was 20 (memory)
        max_depth=None,
        random_state=850 + fold_i,
        n_jobs=-1,
    )
    rf_cal.fit(X_cal_train, y_cal_train, sample_weight=w_cal)

    oof_probs[val_idx] = rf_cal.predict_proba(X_cal_val)[:, 1]
    print("done")
    del rf_cal, X_cal_train, y_cal_train, X_cal_val, w_cal

# Safety check
if np.isnan(oof_probs).any():
    raise ValueError("OOF Platt predictions contain NaN — a fold failed.")

# Clamp probabilities away from 0/1 for numerical stability
eps = 1e-6
oof_probs_clamped = np.clip(oof_probs, eps, 1 - eps)



## Section 11 — Calibration (Isotonic Regression)

Calibrates predicted risk by out-of-fold isotonic regression, compared against raw, Platt, and beta calibration; isotonic is selected as primary (rank-preserving, so AUPRC/enrichment are unchanged).

In [ ]:
# ==============================================================================
# 11A. RAW PREDICTIONS
# ==============================================================================
test_preds_raw = final_rf.predict_proba(X_test_sel)[:, 1]
print(f"test_preds_raw: {len(test_preds_raw)}, range: [{test_preds_raw.min():.4f}, {test_preds_raw.max():.4f}]")

# ==============================================================================
# 11B. ISOTONIC REGRESSION CALIBRATION
# ==============================================================================
eps = 1e-8
oof_probs_clamped = np.clip(oof_probs, eps, 1 - eps)

iso_model = IsotonicRegression(out_of_bounds="clip")
iso_model.fit(oof_probs_clamped, y_train)

test_preds_isotonic = iso_model.predict(np.clip(test_preds_raw, eps, 1 - eps))

print("\nIsotonic Regression calibration:")
print(f"  Calibrated range: [{test_preds_isotonic.min():.6f}, {test_preds_isotonic.max():.6f}]")
print(f"  Cases mean:    {test_preds_isotonic[y_test == 1].mean():.6f}")
print(f"  Controls mean: {test_preds_isotonic[y_test == 0].mean():.6f}")
print(f"  Ratio (cases/controls): {test_preds_isotonic[y_test == 1].mean() / test_preds_isotonic[y_test == 0].mean():.1f}x")

# ==============================================================================
# 11C. PLATT SCALING ON BALANCED SUBSAMPLE (for comparison)
# ==============================================================================
np.random.seed(800)
cal_case_idx = np.where(y_train == 1)[0]
cal_ctrl_idx = np.random.choice(np.where(y_train == 0)[0], size=min(50000, (y_train == 0).sum()), replace=False)
cal_balanced_idx = np.concatenate([cal_case_idx, cal_ctrl_idx])

platt_balanced = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000, random_state=42)
platt_balanced.fit(
    oof_probs_clamped[cal_balanced_idx].reshape(-1, 1),
    y_train[cal_balanced_idx]
)

test_preds_platt_bal = platt_balanced.predict_proba(
    np.clip(test_preds_raw, eps, 1 - eps).reshape(-1, 1)
)[:, 1]

print(f"\nPlatt on balanced subsample:")
print(f"  Calibrated range: [{test_preds_platt_bal.min():.6f}, {test_preds_platt_bal.max():.6f}]")

# ==============================================================================
# 11D. BETA CALIBRATION (3-parameter)
# ==============================================================================
from betacal import BetaCalibration

beta_cal = BetaCalibration(parameters="abm")
beta_cal.fit(oof_probs_clamped, y_train)

test_preds_beta = beta_cal.predict(np.clip(test_preds_raw, eps, 1 - eps))
print(f"\nBeta calibration (3-param):")
print(f"  Calibrated range: [{test_preds_beta.min():.6f}, {test_preds_beta.max():.6f}]")
print(f"  Cases mean:    {test_preds_beta[y_test == 1].mean():.6f}")
print(f"  Controls mean: {test_preds_beta[y_test == 0].mean():.6f}")

# ==============================================================================
# 11E. FINAL CALIBRATION CHOICE: ISOTONIC
# ==============================================================================
calibration_model = iso_model
test_preds_final = test_preds_isotonic

print(f"\nAUC comparison (should all be similar):")
print(f"  Raw RF probs:      {roc_auc_score(y_test, test_preds_raw):.4f}")
print(f"  Isotonic:          {roc_auc_score(y_test, test_preds_isotonic):.4f}")
print(f"  Platt (balanced):  {roc_auc_score(y_test, test_preds_platt_bal):.4f}")
print(f"  Beta (3-param):    {roc_auc_score(y_test, test_preds_beta):.4f}")

print("\n✅ Calibration complete. Using isotonic regression as primary.")


## Section 12 — Evaluation on Holdout Test Set

Holdout evaluation: discrimination (AUROC/AUPRC), Brier scores, a 4-method calibration comparison (`cal_summary`), and an operating-point/threshold table (`threshold_table`); exports calibration figures and `calibration_summary_4methods.csv`.

In [ ]:
# ==============================================================================
# SECTION 8: EVALUATION ON HOLDOUT TEST SET
# ==============================================================================
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, brier_score_loss
)
import matplotlib.pyplot as plt

print("=" * 60)
print("SECTION 8 — EVALUATION ON HOLDOUT TEST SET")
print("=" * 60)

# --- A. ROC-AUC with bootstrap CI ---
base_auc = roc_auc_score(y_test, test_preds_raw)

np.random.seed(42)
n_boot = 2000
boot_aucs = []
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    if len(np.unique(y_test[idx])) < 2:
        continue
    boot_aucs.append(roc_auc_score(y_test[idx], test_preds_raw[idx]))

auc_ci_lower = np.percentile(boot_aucs, 2.5)
auc_ci_upper = np.percentile(boot_aucs, 97.5)
print(f"\nROC-AUC: {base_auc:.4f} (95% CI: {auc_ci_lower:.4f}–{auc_ci_upper:.4f})")

# --- B. PR-AUC ---
precision_arr, recall_arr, _ = precision_recall_curve(y_test, test_preds_raw)
pr_auc = auc(recall_arr, precision_arr)
print(f"PR-AUC: {pr_auc:.4f}")

# --- C. Optimal thresholds ---
fpr, tpr, thresholds_roc = roc_curve(y_test, test_preds_raw)

# Youden's J
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
youden_thresh = thresholds_roc[best_j_idx]
youden_sens = tpr[best_j_idx]
youden_spec = 1 - fpr[best_j_idx]
print(f"\nYouden threshold: {youden_thresh:.4f} | Sens: {youden_sens:.3f} | Spec: {youden_spec:.3f}")

# High-sensitivity threshold (target >= 90%)
high_sens_idx = np.where(tpr >= 0.90)[0]
if len(high_sens_idx) > 0:
    # Pick the one with highest specificity among those with >=90% sensitivity
    best_hs_idx = high_sens_idx[np.argmax(1 - fpr[high_sens_idx])]
    hs_thresh = thresholds_roc[best_hs_idx]
    hs_sens = tpr[best_hs_idx]
    hs_spec = 1 - fpr[best_hs_idx]
else:
    print("Could not achieve 90% sensitivity. Using Youden as fallback.")
    hs_thresh, hs_sens, hs_spec = youden_thresh, youden_sens, youden_spec

print(f"High-sens threshold: {hs_thresh:.4f} | Sens: {hs_sens:.3f} | Spec: {hs_spec:.3f}")

# --- D. Classification metrics at both thresholds ---
prevalence = y_test.mean()

def report_threshold(y_true, y_prob, threshold, label):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    print(f"\n--- {label} (threshold={threshold:.4f}) ---")
    print(f"Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")
    print(f"PPV: {ppv:.4f} | NPV: {npv:.4f}")
    print(f"Confusion matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

    return {"Label": label, "Threshold": threshold,
            "Sensitivity": sens, "Specificity": spec,
            "PPV": ppv, "NPV": npv, "TP": tp, "FP": fp, "FN": fn, "TN": tn}

r1 = report_threshold(y_test, test_preds_raw, youden_thresh, "Youden")
r2 = report_threshold(y_test, test_preds_raw, hs_thresh, "High-Sensitivity (90%)")

threshold_table = pd.DataFrame([r1, r2])
print(f"\n{threshold_table.to_string(index=False)}")

# --- E. Brier scores ---
brier_raw = brier_score_loss(y_test, test_preds_raw)
brier_cal = brier_score_loss(y_test, test_preds_final)  # isotonic-calibrated
print(f"\nBrier Score — Raw: {brier_raw:.4f} | Isotonic-calibrated: {brier_cal:.4f}")

# --- F. Bootstrap CIs for sensitivity/specificity/PPV/NPV at Youden ---
print("\nBootstrapping CIs for classification metrics at Youden threshold...")
np.random.seed(42)
boot_metrics = {"sens": [], "spec": [], "ppv": [], "npv": []}
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    y_b = y_test[idx]
    p_b = (test_preds_raw[idx] >= youden_thresh).astype(int)
    cm_b = confusion_matrix(y_b, p_b, labels=[0, 1])
    tn_b, fp_b, fn_b, tp_b = cm_b.ravel()
    boot_metrics["sens"].append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
    boot_metrics["spec"].append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
    boot_metrics["ppv"].append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0)
    boot_metrics["npv"].append(tn_b / (tn_b + fn_b) if (tn_b + fn_b) > 0 else 0)

print(f"  Sensitivity: {np.percentile(boot_metrics['sens'], 2.5):.3f}–{np.percentile(boot_metrics['sens'], 97.5):.3f}")
print(f"  Specificity: {np.percentile(boot_metrics['spec'], 2.5):.3f}–{np.percentile(boot_metrics['spec'], 97.5):.3f}")
print(f"  PPV:         {np.percentile(boot_metrics['ppv'], 2.5):.4f}–{np.percentile(boot_metrics['ppv'], 97.5):.4f}")
print(f"  NPV:         {np.percentile(boot_metrics['npv'], 2.5):.4f}–{np.percentile(boot_metrics['npv'], 97.5):.4f}")

# --- G. ROC Curve Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
axes[0].plot(fpr, tpr, color="#B2182B", lw=2, label=f"AUC = {base_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].scatter([1 - youden_spec], [youden_sens], color="blue", s=80, zorder=5, label=f"Youden ({youden_thresh:.3f})")
axes[0].set_xlabel("1 - Specificity (FPR)")
axes[0].set_ylabel("Sensitivity (TPR)")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")

# PR curve
axes[1].plot(recall_arr, precision_arr, color="#2166AC", lw=2, label=f"PR-AUC = {pr_auc:.3f}")
axes[1].axhline(y=prevalence, color="grey", linestyle="--", alpha=0.5, label=f"Prevalence = {prevalence:.4f}")
axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision (PPV)")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(loc="upper right")

# Calibration curve (raw vs isotonic)
def calibration_curve_custom(y_true, y_prob, n_bins=10):
    bin_edges = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    mean_pred, obs_rate, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if i == len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            obs_rate.append(y_true[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_pred), np.array(obs_rate), np.array(counts)

mp_raw, or_raw, ct_raw = calibration_curve_custom(y_test, test_preds_raw)
mp_cal, or_cal, ct_cal = calibration_curve_custom(y_test, test_preds_final)

axes[2].plot(mp_raw, or_raw, "o-", color="#B2182B", label="Raw RF", markersize=5)
axes[2].plot(mp_cal, or_cal, "s-", color="#2166AC", label="Isotonic", markersize=5)
axes[2].plot([0, max(mp_raw.max(), mp_cal.max())],
             [0, max(mp_raw.max(), mp_cal.max())], "k--", alpha=0.3)
axes[2].set_xlabel("Mean Predicted Probability")
axes[2].set_ylabel("Observed HS Rate")
axes[2].set_title("Calibration Curve")
axes[2].legend()

plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ Section 8 complete.")

In [ ]:
# --- G. Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# =========================
# 1. ROC CURVE
# =========================
axes[0].plot(fpr, tpr, color="#B2182B", lw=2, label=f"AUC = {base_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].scatter(
    [1 - youden_spec],
    [youden_sens],
    color="blue",
    s=80,
    zorder=5,
    label=f"Youden ({youden_thresh:.3f})"
)
axes[0].set_xlabel("1 - Specificity (FPR)")
axes[0].set_ylabel("Sensitivity (TPR)")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")

# =========================
# 2. PR CURVE
# =========================
axes[1].plot(
    recall_arr,
    precision_arr,
    color="#2166AC",
    lw=2,
    label=f"PR-AUC = {pr_auc:.3f}"
)
axes[1].axhline(
    y=prevalence,
    color="grey",
    linestyle="--",
    alpha=0.5,
    label=f"Prevalence = {prevalence:.4f}"
)
axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision (PPV)")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(loc="upper right")

# =========================
# 3. CALIBRATION CURVE
# =========================
def calibration_curve_custom(y_true, y_prob, n_bins=10):
    bin_edges = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)

    mean_pred, obs_rate, counts = [], [], []

    for i in range(len(bin_edges) - 1):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])

        if i == len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])

        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            obs_rate.append(y_true[mask].mean())
            counts.append(mask.sum())

    return np.array(mean_pred), np.array(obs_rate), np.array(counts)

mp_raw, or_raw, ct_raw = calibration_curve_custom(y_test, test_preds_raw)
mp_cal, or_cal, ct_cal = calibration_curve_custom(y_test, test_preds_final)

# Full observed range across both curves
full_lim = max(
    mp_raw.max(), mp_cal.max(),
    or_raw.max(), or_cal.max()
) * 1.08

# Tight isotonic-focused zoom
iso_lim = max(
    mp_cal.max(), or_cal.max()
) * 1.15

axes[2].plot(mp_raw, or_raw, "o-", color="#B2182B", label="Raw RF", markersize=6)
axes[2].plot(mp_cal, or_cal, "s-", color="#2166AC", label="Isotonic", markersize=6)
axes[2].plot([0, full_lim], [0, full_lim], "k--", alpha=0.3, label="Ideal")
axes[2].axhline(
    y=prevalence,
    color="grey",
    linestyle=":",
    alpha=0.5,
    label=f"Event rate = {prevalence:.4f}"
)
axes[2].set_xlim(0, full_lim)
axes[2].set_ylim(0, full_lim)
axes[2].set_aspect("equal", adjustable="box")
axes[2].set_xlabel("Mean Predicted Probability")
axes[2].set_ylabel("Observed HS Rate")
axes[2].set_title("Calibration (Full Observed Range)")
axes[2].legend()

plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Also save standalone 2-panel calibration figure for manuscript ---
fig2, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: full range
ax_full.plot(mp_raw, or_raw, "o-", color="#B2182B", label="Raw RF", markersize=6)
ax_full.plot(mp_cal, or_cal, "s-", color="#2166AC", label="Isotonic", markersize=6)
ax_full.plot([0, full_lim], [0, full_lim], "k--", alpha=0.3, label="Ideal")
ax_full.axhline(
    y=prevalence,
    color="grey",
    linestyle=":",
    alpha=0.5,
    label=f"Event rate = {prevalence:.4f}"
)
ax_full.set_xlim(0, full_lim)
ax_full.set_ylim(0, full_lim)
ax_full.set_aspect("equal", adjustable="box")
ax_full.set_xlabel("Mean Predicted Probability")
ax_full.set_ylabel("Observed HS Rate")
ax_full.set_title("A. Full Observed Range")
ax_full.legend()

# Panel B: isotonic-focused zoom
ax_zoom.plot(mp_raw, or_raw, "o-", color="#B2182B", alpha=0.35, label="Raw RF", markersize=6)
ax_zoom.plot(mp_cal, or_cal, "s-", color="#2166AC", label="Isotonic", markersize=6)
ax_zoom.plot([0, iso_lim], [0, iso_lim], "k--", alpha=0.3, label="Ideal")
ax_zoom.axhline(
    y=prevalence,
    color="grey",
    linestyle=":",
    alpha=0.5,
    label=f"Event rate = {prevalence:.4f}"
)
ax_zoom.set_xlim(0, iso_lim)
ax_zoom.set_ylim(0, iso_lim)
ax_zoom.set_aspect("equal", adjustable="box")
ax_zoom.set_xlabel("Mean Predicted Probability")
ax_zoom.set_ylabel("Observed HS Rate")
ax_zoom.set_title("B. Isotonic-Focused Zoom")
ax_zoom.legend()

plt.tight_layout()
plt.savefig("calibration_2panel.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ Section 8 complete.")

In [ ]:
# --- Calibration Curve Comparison Plot (4 methods) ---
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

strategies = [
    (test_preds_raw,         "Raw RF",            "#888888", "o"),
    (test_preds_isotonic,    "Isotonic",           "#B2182B", "s"),
    (test_preds_platt_bal,   "Platt (balanced)",   "#2166AC", "^"),
    (test_preds_beta,        "Beta (3-param)",     "#4DAF4A", "D"),
]

for ax, n_bins, strategy_label, xlim in [
    (axes[0], 10, "Quantile bins (n=10)",  None),
    (axes[1], 10, "Uniform bins (n=10)",   None),
    (axes[2], 10, "Uniform bins (n=10)",   (0, 0.15)),  # zoomed
]:
    for probs, label, color, marker in strategies:
        try:
            frac_pos, mean_pred = calibration_curve(
                y_test, probs, n_bins=n_bins,
                strategy='quantile' if 'Quantile' in strategy_label else 'uniform'
            )
            ax.plot(mean_pred, frac_pos, marker=marker, color=color,
                    lw=2, markersize=6, label=label)
        except Exception as e:
            print(f"Skipping {label} on {strategy_label}: {e}")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect calibration")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Observed HS Rate")
    ax.set_title(strategy_label)
    ax.legend(fontsize=8)
    if xlim:
        ax.set_xlim(xlim)
        ax.set_ylim(0, 0.15)

plt.suptitle("Calibration Curves — Raw RF vs Isotonic vs Platt vs Beta",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("calibration_comparison_4methods.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Calibration comparison plot saved (4 methods).")


In [ ]:
# --- Calibration Metrics (4 methods) ---
from sklearn.metrics import brier_score_loss
import pandas as pd

print("\n" + "=" * 60)
print("CALIBRATION METRICS COMPARISON")
print("=" * 60)

def calibration_metrics(y_true, y_prob, label, n_bins=10):
    brier = brier_score_loss(y_true, y_prob)

    # ECE — uniform bins
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    mce = 0.0
    n = len(y_true)
    bin_details = []

    for i in range(n_bins):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if i == n_bins - 1:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_n = mask.sum()
        bin_conf = y_prob[mask].mean()
        bin_acc = y_true[mask].mean()
        bin_err = abs(bin_conf - bin_acc)
        ece += (bin_n / n) * bin_err
        mce = max(mce, bin_err)
        bin_details.append({
            "Bin": f"{bin_edges[i]:.2f}–{bin_edges[i+1]:.2f}",
            "N": bin_n, "Mean Pred": bin_conf, "Obs Rate": bin_acc, "Error": bin_err
        })

    # ECE — quantile bins
    bin_edges_q = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges_q = np.unique(bin_edges_q)
    ece_q = 0.0
    for i in range(len(bin_edges_q) - 1):
        mask = (y_prob >= bin_edges_q[i]) & (y_prob < bin_edges_q[i + 1])
        if i == len(bin_edges_q) - 2:
            mask = (y_prob >= bin_edges_q[i]) & (y_prob <= bin_edges_q[i + 1])
        if mask.sum() == 0:
            continue
        bin_n = mask.sum()
        bin_conf = y_prob[mask].mean()
        bin_acc = y_true[mask].mean()
        ece_q += (bin_n / n) * abs(bin_conf - bin_acc)

    mean_pred = y_prob.mean()
    mean_obs = y_true.mean()
    calibration_ratio = mean_pred / mean_obs if mean_obs > 0 else np.nan

    print(f"\n--- {label} ---")
    print(f"  Brier Score:              {brier:.6f}  (lower=better; null={y_true.mean()*(1-y_true.mean()):.6f})")
    print(f"  ECE (uniform bins):       {ece:.6f}  (lower=better)")
    print(f"  ECE (quantile bins):      {ece_q:.6f}  (lower=better)")
    print(f"  Max Calibration Error:    {mce:.6f}")
    print(f"  Mean predicted prob:      {mean_pred:.6f}")
    print(f"  Mean observed rate:       {mean_obs:.6f}")
    print(f"  Calibration ratio (P/O):  {calibration_ratio:.4f}  (1.0 = perfect)")
    print(f"  AUC:                      {roc_auc_score(y_true, y_prob):.4f}")

    return {
        "Model": label, "Brier": brier,
        "ECE_uniform": ece, "ECE_quantile": ece_q, "MCE": mce,
        "Mean_Pred": mean_pred, "Mean_Obs": mean_obs,
        "P/O Ratio": calibration_ratio,
        "AUC": roc_auc_score(y_true, y_prob)
    }, bin_details

results = []
r1, bd1 = calibration_metrics(y_test, test_preds_raw,       "Raw RF")
r2, bd2 = calibration_metrics(y_test, test_preds_isotonic,   "Isotonic")
r3, bd3 = calibration_metrics(y_test, test_preds_platt_bal,  "Platt (balanced)")
r4, bd4 = calibration_metrics(y_test, test_preds_beta,       "Beta (3-param)")

results = [r1, r2, r3, r4]
cal_summary = pd.DataFrame(results).set_index("Model")

print("\n" + "=" * 60)
print("SUMMARY TABLE")
print("=" * 60)
print(cal_summary.to_string())

# Bin-level detail for Platt (uniform bins)
print("\n--- Bin-level detail for Platt (uniform bins) ---")
print(pd.DataFrame(bd3).to_string(index=False))

# Bin-level detail for Beta (uniform bins)
print("\n--- Bin-level detail for Beta (uniform bins) ---")
print(pd.DataFrame(bd4).to_string(index=False))

cal_summary.to_csv("calibration_summary_4methods.csv")
print("\n✅ Calibration metrics saved (4 methods).")


## Section 12B — Top-Risk Enrichment Analysis
Evaluate enrichment of true HS cases in the top 1%, 5%, and 10% of predicted risk.
This bridges model discrimination to clinical interpretability.

In [ ]:
# ==============================================================================
# SECTION 12B: TOP-RISK ENRICHMENT ANALYSIS
# ==============================================================================
# Moved early: this is the clinical bridge between discrimination and utility.
# Compute enrichment for isotonic-calibrated RF predictions on the held-out test set.
# ==============================================================================
print("=" * 60)
print("SECTION 12B — TOP-RISK ENRICHMENT")
print("=" * 60)

def top_risk_enrichment(y_true, y_prob, percentiles=[1, 5, 10], n_boot=2000, seed=42):
    """
    Compute enrichment statistics for top-risk percentile groups.
    Returns a DataFrame with one row per percentile.
    """
    n = len(y_true)
    total_pos = y_true.sum()
    overall_rate = y_true.mean()
    sorted_idx = np.argsort(y_prob)[::-1]

    rows = []
    for pct in percentiles:
        k = max(1, int(np.ceil(n * pct / 100)))
        top_idx = sorted_idx[:k]
        n_pos_in_bucket = y_true[top_idx].sum()
        obs_rate = y_true[top_idx].mean()
        fold_enrichment = obs_rate / overall_rate if overall_rate > 0 else np.nan
        pct_positives_captured = n_pos_in_bucket / total_pos * 100 if total_pos > 0 else 0

        np.random.seed(seed)
        boot_enrichments = []
        boot_captures = []
        for _ in range(n_boot):
            b_idx = np.random.choice(n, size=n, replace=True)
            b_y = y_true[b_idx]
            b_p = y_prob[b_idx]
            b_sorted = np.argsort(b_p)[::-1]
            b_k = max(1, int(np.ceil(len(b_y) * pct / 100)))
            b_top = b_sorted[:b_k]
            b_rate = b_y[b_top].mean()
            b_overall = b_y.mean()
            b_enrich = b_rate / b_overall if b_overall > 0 else np.nan
            b_cap = b_y[b_top].sum() / max(b_y.sum(), 1) * 100
            boot_enrichments.append(b_enrich)
            boot_captures.append(b_cap)

        rows.append({
            "Top_Pct": f"Top {pct}%",
            "N_in_bucket": k,
            "N_positives": int(n_pos_in_bucket),
            "Observed_rate": obs_rate,
            "Overall_rate": overall_rate,
            "Fold_enrichment": fold_enrichment,
            "Fold_enrich_CI_lo": np.percentile(boot_enrichments, 2.5),
            "Fold_enrich_CI_hi": np.percentile(boot_enrichments, 97.5),
            "Pct_positives_captured": pct_positives_captured,
            "Pct_captured_CI_lo": np.percentile(boot_captures, 2.5),
            "Pct_captured_CI_hi": np.percentile(boot_captures, 97.5),
        })

    return pd.DataFrame(rows)

# --- RF (isotonic-calibrated) enrichment ---
print("\n--- RF (Isotonic-Calibrated) Enrichment ---")
enrich_rf_iso = top_risk_enrichment(y_test, test_preds_final, [1, 5, 10])
enrich_rf_iso.insert(0, "Model", "RF_isotonic")
print(enrich_rf_iso.to_string(index=False))

for _, r in enrich_rf_iso.iterrows():
    print(f"  → {r['Top_Pct']}: {r['Fold_enrichment']:.1f}× enrichment "
          f"(95% CI: {r['Fold_enrich_CI_lo']:.1f}–{r['Fold_enrich_CI_hi']:.1f}), "
          f"captures {r['Pct_positives_captured']:.1f}% of cases")

# --- RF (raw) enrichment for comparison ---
print("\n--- RF (Raw) Enrichment ---")
enrich_rf_raw = top_risk_enrichment(y_test, test_preds_raw, [1, 5, 10])
enrich_rf_raw.insert(0, "Model", "RF_raw")
print(enrich_rf_raw.to_string(index=False))

# Combined
enrich_combined_rf = pd.concat([enrich_rf_iso, enrich_rf_raw], ignore_index=True)
enrich_combined_rf.to_csv("top_risk_enrichment_rf.csv", index=False)

print("\n✅ Section 12B: Top-risk enrichment complete.")

## Section 12C — XGBoost Benchmark
Weighted gradient-boosted tree model as additional nonlinear benchmark.
Same train/test split, same features, same evaluation framework as RF and LR.
Includes OOF calibration, SHAP, and top-risk enrichment for direct comparison.

In [ ]:
# ==============================================================================
# SECTION 12C: XGBOOST BENCHMARK
# ==============================================================================
# Adds a gradient-boosted tree model alongside RF and LR.
# Uses scale_pos_weight (capped at 50, same as RF case weight cap).
# OOF predictions for calibration. Full evaluation on held-out test set.
# ==============================================================================
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, brier_score_loss

print("=" * 60)
print("SECTION 12C — XGBOOST BENCHMARK")
print("=" * 60)

# --- A. Class weight (same cap as RF) ---
raw_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb_scale_pos_weight = min(raw_ratio, 50.0)
print(f"scale_pos_weight: {xgb_scale_pos_weight:.1f} (raw {raw_ratio:.1f}, cap 50)")

# --- B. XGBoost model ---
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "scale_pos_weight": xgb_scale_pos_weight,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "n_estimators": 500,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

xgb_model = xgb.XGBClassifier(**xgb_params)

# --- C. OOF predictions for calibration ---
print("\nGenerating OOF predictions for XGBoost calibration...")
xgb_oof_probs = np.full(len(y_train), np.nan)
skf_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=900)

for fold_i, (tr_idx, val_idx) in enumerate(skf_xgb.split(X_train_sel, y_train), 1):
    print(f"  XGB OOF fold {fold_i}/5 ...", end=" ")
    X_f_tr, y_f_tr = X_train_sel[tr_idx], y_train[tr_idx]
    X_f_val = X_train_sel[val_idx]

    xgb_fold = xgb.XGBClassifier(**xgb_params)
    xgb_fold.fit(X_f_tr, y_f_tr)
    xgb_oof_probs[val_idx] = xgb_fold.predict_proba(X_f_val)[:, 1]
    print("done")

# --- D. Fit final XGBoost on full training set ---
print("\nFitting final XGBoost on full training set...")
xgb_model.fit(X_train_sel, y_train)
xgb_test_preds_raw = xgb_model.predict_proba(X_test_sel)[:, 1]
print(f"  Raw test preds range: [{xgb_test_preds_raw.min():.4f}, {xgb_test_preds_raw.max():.4f}]")

# --- E. Calibrate XGBoost (isotonic + beta) ---
eps = 1e-8
xgb_oof_clamped = np.clip(xgb_oof_probs, eps, 1 - eps)

# Isotonic
xgb_iso_model = IsotonicRegression(out_of_bounds="clip")
xgb_iso_model.fit(xgb_oof_clamped, y_train)
xgb_test_preds_iso = xgb_iso_model.predict(np.clip(xgb_test_preds_raw, eps, 1 - eps))

# Beta
from betacal import BetaCalibration
xgb_beta_cal = BetaCalibration(parameters="abm")
xgb_beta_cal.fit(xgb_oof_clamped, y_train)
xgb_test_preds_beta = xgb_beta_cal.predict(np.clip(xgb_test_preds_raw, eps, 1 - eps))

print(f"\n  XGB isotonic range: [{xgb_test_preds_iso.min():.6f}, {xgb_test_preds_iso.max():.6f}]")
print(f"  XGB beta range:    [{xgb_test_preds_beta.min():.6f}, {xgb_test_preds_beta.max():.6f}]")

# --- F. Evaluation metrics ---
print("\n" + "=" * 60)
print("XGBOOST EVALUATION ON HOLDOUT TEST SET")
print("=" * 60)

# ROC-AUC with bootstrap CI
xgb_auc = roc_auc_score(y_test, xgb_test_preds_raw)
np.random.seed(42)
xgb_boot_aucs = []
for _ in range(2000):
    idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    if len(np.unique(y_test[idx])) < 2:
        continue
    xgb_boot_aucs.append(roc_auc_score(y_test[idx], xgb_test_preds_raw[idx]))

xgb_auc_ci_lo = np.percentile(xgb_boot_aucs, 2.5)
xgb_auc_ci_hi = np.percentile(xgb_boot_aucs, 97.5)
print(f"\nROC-AUC: {xgb_auc:.4f} (95% CI: {xgb_auc_ci_lo:.4f}–{xgb_auc_ci_hi:.4f})")

# PR-AUC
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, xgb_test_preds_raw)
xgb_prauc = auc(rec_xgb, prec_xgb)
print(f"PR-AUC: {xgb_prauc:.4f}")

# Brier
xgb_brier_raw = brier_score_loss(y_test, xgb_test_preds_raw)
xgb_brier_iso = brier_score_loss(y_test, xgb_test_preds_iso)
print(f"Brier (raw): {xgb_brier_raw:.6f}")
print(f"Brier (isotonic): {xgb_brier_iso:.6f}")

# Calibration metrics for XGBoost
def xgb_cal_metrics(y_true, y_prob, label):
    brier = brier_score_loss(y_true, y_prob)
    mean_pred = y_prob.mean()
    mean_obs = y_true.mean()
    po_ratio = mean_pred / mean_obs if mean_obs > 0 else np.nan
    auroc = roc_auc_score(y_true, y_prob)
    return {"Model": label, "Brier": brier, "Mean_Pred": mean_pred,
            "Mean_Obs": mean_obs, "P/O Ratio": po_ratio, "AUC": auroc}

xgb_cal_rows = [
    xgb_cal_metrics(y_test, xgb_test_preds_raw, "XGB Raw"),
    xgb_cal_metrics(y_test, xgb_test_preds_iso, "XGB Isotonic"),
    xgb_cal_metrics(y_test, xgb_test_preds_beta, "XGB Beta"),
]
xgb_cal_df = pd.DataFrame(xgb_cal_rows)
print("\nXGBoost Calibration Summary:")
print(xgb_cal_df.to_string(index=False))

# --- G. Thresholds (Youden + high-sensitivity) ---
from sklearn.metrics import roc_curve
fpr_xgb, tpr_xgb, thresh_xgb = roc_curve(y_test, xgb_test_preds_raw)
j_xgb = tpr_xgb - fpr_xgb
best_j_xgb = np.argmax(j_xgb)
xgb_youden_thresh = thresh_xgb[best_j_xgb]
xgb_youden_sens = tpr_xgb[best_j_xgb]
xgb_youden_spec = 1 - fpr_xgb[best_j_xgb]

print(f"\nYouden threshold: {xgb_youden_thresh:.4f} | Sens: {xgb_youden_sens:.3f} | Spec: {xgb_youden_spec:.3f}")

# --- H. Top-risk enrichment ---
print("\n--- XGBoost Top-Risk Enrichment (Isotonic-Calibrated) ---")
enrich_xgb = top_risk_enrichment(y_test, xgb_test_preds_iso, [1, 5, 10])
enrich_xgb.insert(0, "Model", "XGB_isotonic")
print(enrich_xgb.to_string(index=False))

for _, r in enrich_xgb.iterrows():
    print(f"  → {r['Top_Pct']}: {r['Fold_enrichment']:.1f}× enrichment "
          f"(95% CI: {r['Fold_enrich_CI_lo']:.1f}–{r['Fold_enrich_CI_hi']:.1f}), "
          f"captures {r['Pct_positives_captured']:.1f}% of cases")

# --- I. SHAP for XGBoost ---
print("\n" + "=" * 60)
print("XGBOOST SHAP VALUES")
print("=" * 60)

import shap
xgb_explainer = shap.TreeExplainer(xgb_model)
# Use same SHAP sample size as RF (2000)
np.random.seed(42)
shap_sample_idx = np.random.choice(len(X_test_sel), size=min(2000, len(X_test_sel)), replace=False)
X_shap_sample = X_test_sel[shap_sample_idx]

xgb_shap_values = xgb_explainer.shap_values(X_shap_sample)
xgb_shap_importance = pd.DataFrame({
    "feature": stable_features,
    "mean_abs_shap": np.abs(xgb_shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("\nTop 20 XGBoost features by mean |SHAP|:")
print(xgb_shap_importance.head(20).to_string(index=False))

# --- J. Three-model comparison table (RF + XGB; LR appended after Section 15) ---
print("\n" + "=" * 60)
print("THREE-MODEL COMPARISON (RF + XGB — LR added after Section 15)")
print("=" * 60)

rf_auc = roc_auc_score(y_test, test_preds_raw)
prec_rf_arr, rec_rf_arr, _ = precision_recall_curve(y_test, test_preds_raw)
rf_prauc = auc(rec_rf_arr, prec_rf_arr)
rf_brier_raw = brier_score_loss(y_test, test_preds_raw)
rf_brier_iso = brier_score_loss(y_test, test_preds_isotonic)
rf_po_raw = test_preds_raw.mean() / y_test.mean()
rf_po_iso = test_preds_isotonic.mean() / y_test.mean()

comparison_rows = [
    {
        "Model": "Random Forest",
        "ROC-AUC": rf_auc,
        "PR-AUC": rf_prauc,
        "Brier (raw)": rf_brier_raw,
        "Brier (isotonic)": rf_brier_iso,
        "P/O (raw)": rf_po_raw,
        "P/O (isotonic)": rf_po_iso,
    },
    {
        "Model": "XGBoost",
        "ROC-AUC": xgb_auc,
        "PR-AUC": xgb_prauc,
        "Brier (raw)": xgb_brier_raw,
        "Brier (isotonic)": xgb_brier_iso,
        "P/O (raw)": xgb_test_preds_raw.mean() / y_test.mean(),
        "P/O (isotonic)": xgb_test_preds_iso.mean() / y_test.mean(),
    },
]

three_model_comparison_partial = pd.DataFrame(comparison_rows)
print(three_model_comparison_partial.to_string(index=False))

# Combined enrichment table
enrich_combined_all = pd.concat([enrich_rf_iso, enrich_xgb], ignore_index=True)
enrich_combined_all.to_csv("top_risk_enrichment_rf_xgb.csv", index=False)

print("\n✅ Section 12C: XGBoost benchmark complete.")

## Section 13 — SHAP Values & Interaction Analysis

**Methodological notes:**
- TreeSHAP computes exact Shapley values for tree-based models in polynomial time.
- With correlated binary features (common in FAERS drug indicators), SHAP distributes importance unpredictably across collinear features. Interpret for directional signal, not causal inference.
- The SHAP correlation proxy (Section 13B) measures co-directionality of marginal SHAP values — not true pairwise interaction decomposition.

**Also runs in this section:** the formal indication × TNFi-agent interaction regression (`results_df`; full output + interaction-term ORs), marginally standardized adjusted probabilities (`pred_grid`), the SHAP pairwise-interaction screen (`pairs_df`), and the SHAP beeswarm/bar plus interaction forest and adjusted-probability heatmap figures.

In [ ]:
# ==============================================================================
# SECTION 13A: SHAP VALUES (TreeSHAP — exact, fast for tree models)
# ==============================================================================
# NOTE: SHAP with correlated binary features (common in FAERS drug indicators)
# distributes importance unpredictably across collinear features. Interpret for
# directional signal, not causal inference.
# ==============================================================================
import shap
print("=" * 60)
print("SECTION 13 — SHAP VALUES")
print("=" * 60)

# --- A. Compute SHAP values ---
shap_sample_size = min(len(y_test), 2000)
np.random.seed(101)
shap_idx = np.random.choice(len(y_test), shap_sample_size, replace=False)
X_shap = X_test_sel[shap_idx]
y_shap = y_test[shap_idx]

print(f"Computing TreeSHAP on {shap_sample_size} test observations...")
explainer = shap.TreeExplainer(final_rf)
shap_values = explainer.shap_values(X_shap)

# Handle both old and new SHAP output formats
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # class 1
elif shap_values.ndim == 3:
    shap_vals = shap_values[:, :, 1]
else:
    shap_vals = shap_values

print(f"SHAP values shape: {shap_vals.shape}")
assert shap_vals.shape[1] == len(stable_features), \
    f"SHAP feature count ({shap_vals.shape[1]}) != stable_features ({len(stable_features)})"

# --- B. Mean |SHAP| importance table ---
shap_importance = pd.DataFrame({
    "feature": stable_features,
    "mean_abs_shap": np.abs(shap_vals).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\nTop 20 features by mean |SHAP|:")
print(shap_importance.head(20).to_string(index=False))

# --- C. SHAP beeswarm plot ---
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=stable_features,
    max_display=15, show=False,
)
plt.title("SHAP Value Distribution — Top 15 Features")
plt.tight_layout()
plt.savefig("shap_beeswarm_top15.png", dpi=300, bbox_inches="tight")
plt.show()

# --- D. SHAP bar plot ---
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=stable_features,
    plot_type="bar", max_display=20, show=False,
)
plt.title("Mean |SHAP| — Top 20 Features")
plt.tight_layout()
plt.savefig("shap_bar_top20.png", dpi=300, bbox_inches="tight")
plt.show()

# --- E. Export ---
shap_importance.to_csv("shap_importance_all_features.csv", index=False)
print("\nExported: shap_importance_all_features.csv")
print("\n✅ Section 13A complete.")

In [ ]:
# ==============================================================================
# SECTION 13B: SHAP CORRELATION PROXY & DEPENDENCE PLOTS
# ==============================================================================
# The SHAP correlation proxy computes pairwise Pearson correlation of marginal
# SHAP values. This detects co-directionality of SHAP effects but does NOT
# isolate the true synergistic interaction component. Two features can show
# correlated SHAP values simply because they are both independently important
# for the same patients, without any synergistic effect.
# ==============================================================================
print("=" * 60)
print("SECTION 13B — SHAP CORRELATION PROXY & DEPENDENCE PLOTS")
print("=" * 60)

# --- A. SHAP correlation proxy ---
shap_corr = np.corrcoef(shap_vals.T)
np.fill_diagonal(shap_corr, 0)

pairs = []
for i in range(len(stable_features)):
    for j in range(i + 1, len(stable_features)):
        pairs.append((stable_features[i], stable_features[j],
                       shap_corr[i, j]))

pairs_df = pd.DataFrame(pairs, columns=["feat_1", "feat_2", "shap_corr"])
pairs_df["abs_shap_corr"] = pairs_df["shap_corr"].abs()
pairs_df = pairs_df.sort_values("abs_shap_corr", ascending=False)

print("Top 15 feature pairs by |SHAP correlation| (proxy — not true interaction):")
print(pairs_df.head(15).to_string(index=False))
pairs_df.to_csv("shap_interaction_screening.csv", index=False)

# --- B. SHAP dependence plots: Hist_IBD × 3 interaction features ---
# These test the clinical hypothesis that IBD history interacts with specific
# TNFi agents and sex in determining HS risk.
interact_features = [
    ("TNFi_adalimumab",  "TNFi_adalimumab"),
    ("TNFi_infliximab",  "TNFi_infliximab"),
    ("Sex_F",            "Sex_F"),
]

x_feat = "Hist_IBD"

if x_feat not in stable_features:
    print(f"  ⚠️  '{x_feat}' not in stable_features — skipping dependence plots.")
else:
    for interact_feat, interact_label in interact_features:
        if interact_feat not in stable_features:
            print(f"  ⚠️  '{interact_feat}' not in stable_features — skipping.")
            continue

        interact_idx = stable_features.index(interact_feat)
        fig, ax = plt.subplots(figsize=(8, 5))
        shap.dependence_plot(
            x_feat,
            shap_vals,
            X_shap,
            feature_names=stable_features,
            interaction_index=interact_idx,
            show=False,
            ax=ax,
        )
        plt.title(f"SHAP: {x_feat}  (color = {interact_label})")
        plt.tight_layout()
        safe_name = f"shap_dep_{x_feat}_color_{interact_label}"
        plt.savefig(f"{safe_name}.png", dpi=300, bbox_inches="tight")
        plt.show()
        print(f"  Saved: {safe_name}.png")

print("\n✅ Section 13B complete.")

In [ ]:
# ==============================================================================
# SECTION 14A: FACTOR VARIABLE CONSTRUCTION
# ==============================================================================
# Build categorical factor variables from existing binary columns in model_df_full.
# These are used for the formal interaction regression (epidemiologic, not predictive).
# ==============================================================================
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

print("=" * 60)
print("SECTION 14 — FACTOR-LEVEL INTERACTION REGRESSION")
print("=" * 60)

# Work on a copy to avoid modifying model_df_full
int_df = model_df_full.copy()

# --- A. tnfi_agent_group ---
tnfi_cols = ["TNFi_adalimumab", "TNFi_infliximab", "TNFi_etanercept",
             "TNFi_certolizumab.pegol", "TNFi_golimumab"]
# Map column names to clean agent labels
tnfi_labels = {
    "TNFi_adalimumab": "adalimumab",
    "TNFi_infliximab": "infliximab",
    "TNFi_etanercept": "etanercept",
    "TNFi_certolizumab.pegol": "certolizumab_pegol",
    "TNFi_golimumab": "golimumab",
}

def assign_tnfi_group(row):
    active = [tnfi_labels[c] for c in tnfi_cols if row.get(c, 0) == 1]
    if len(active) == 0:
        return "None"
    elif len(active) == 1:
        return active[0]
    else:
        return "Multiple_TNFi"

int_df["tnfi_agent_group"] = int_df.apply(assign_tnfi_group, axis=1)
print("\ntnfi_agent_group distribution:")
print(int_df["tnfi_agent_group"].value_counts().to_string())

# --- B. indication_group ---
indic_cols = ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]
indic_labels = {
    "Hist_Arthropathies": "Arthropathies",
    "Hist_IBD": "IBD",
    "Hist_Psoriasis": "Psoriasis",
}

def assign_indication_group(row):
    active = [indic_labels[c] for c in indic_cols if row.get(c, 0) == 1]
    if len(active) == 0:
        return "None"
    elif len(active) == 1:
        return active[0]
    else:
        return "Multiple_Indications"

int_df["indication_group"] = int_df.apply(assign_indication_group, axis=1)
print("\nindication_group distribution:")
print(int_df["indication_group"].value_counts().to_string())

# --- C. sex_group ---
def assign_sex_group(row):
    if row.get("Sex_F", 0) == 1:
        return "Female"
    elif row.get("Sex_M", 0) == 1:
        return "Male"
    else:
        return "Unknown"

int_df["sex_group"] = int_df.apply(assign_sex_group, axis=1)
print("\nsex_group distribution:")
print(int_df["sex_group"].value_counts().to_string())

# --- D. age_group (factor with Missing as its own level) ---
# Bins: <20, 20-29, 30-39, 40-49, 50-59, 60-69, 70-79, 80+, Missing
# This replaces continuous age + Age_missing in the interaction regression,
# avoiding median-imputation artifacts and allowing non-linear age effects.

def assign_age_group(row):
    if row.get("Age_missing", 0) == 1 or pd.isna(row.get("age")):
        return "Missing"
    a = row["age"]
    if a < 20:
        return "<20"
    elif a < 30:
        return "20-29"
    elif a < 40:
        return "30-39"
    elif a < 50:
        return "40-49"
    elif a < 60:
        return "50-59"
    elif a < 70:
        return "60-69"
    elif a < 80:
        return "70-79"
    else:
        return "80+"

int_df["age_group"] = int_df.apply(assign_age_group, axis=1)
print("\nage_group distribution:")
print(int_df["age_group"].value_counts().sort_index().to_string())

# --- E. Binary outcome ---
int_df["hs_binary"] = (int_df["outcome_hs"] == "Yes").astype(int)
print(f"\nHS cases in interaction df: {int_df['hs_binary'].sum()} / {len(int_df)}")

print("\n✅ Section 14A: Factor variables constructed.")


In [ ]:
# ==============================================================================
# SECTION 14B: SPARSE CELL AUDIT
# ==============================================================================
# Cross-tabulate indication_group × tnfi_agent_group with total counts and HS cases.
# Auto-collapse any cells with < 3 total patients.
# ==============================================================================
print("=" * 60)
print("SECTION 14B — SPARSE CELL AUDIT")
print("=" * 60)

# --- Total count cross-tab ---
ct_total = pd.crosstab(int_df["indication_group"], int_df["tnfi_agent_group"], margins=True)
print("\nTotal patients per cell:")
print(ct_total.to_string())

# --- HS case count cross-tab ---
ct_cases = pd.crosstab(
    int_df["indication_group"],
    int_df["tnfi_agent_group"],
    values=int_df["hs_binary"],
    aggfunc="sum",
    margins=True
).fillna(0).astype(int)
print("\nHS cases per cell:")
print(ct_cases.to_string())

# --- Flag and auto-collapse sparse cells (< 3 total patients) ---
sparse_cells = []
for idx_name in ct_total.index:
    if idx_name == "All":
        continue
    for col_name in ct_total.columns:
        if col_name == "All":
            continue
        n = ct_total.loc[idx_name, col_name]
        if n < 3:
            sparse_cells.append((idx_name, col_name, n))

if sparse_cells:
    print(f"\n⚠️  Found {len(sparse_cells)} cells with < 3 patients — collapsing:")
    for ind, agent, n in sparse_cells:
        print(f"    {ind} × {agent}: n={n} → collapsing to 'Other'")

    # Collapse sparse indication levels to "Other"
    sparse_indications = set(ind for ind, agent, n in sparse_cells)
    sparse_agents = set(agent for ind, agent, n in sparse_cells)

    # Only collapse if a level is sparse across MOST agent groups
    for ind in sparse_indications:
        ind_total = ct_total.loc[ind, "All"] if ind in ct_total.index else 0
        if ind_total < 3:
            int_df.loc[int_df["indication_group"] == ind, "indication_group"] = "Other"
            print(f"    → Collapsed indication '{ind}' to 'Other' (total n={ind_total})")

    for agent in sparse_agents:
        agent_total = ct_total.loc["All", agent] if agent in ct_total.columns else 0
        if agent_total < 3:
            int_df.loc[int_df["tnfi_agent_group"] == agent, "tnfi_agent_group"] = "Other"
            print(f"    → Collapsed agent '{agent}' to 'Other' (total n={agent_total})")

    # Reprint after collapsing
    ct_total_post = pd.crosstab(int_df["indication_group"], int_df["tnfi_agent_group"], margins=True)
    print("\nPost-collapse total patients per cell:")
    print(ct_total_post.to_string())
else:
    print("\n✅ No sparse cells detected (all ≥ 3 patients).")

print("\n✅ Section 14B: Sparse cell audit complete.")


In [ ]:
# ==============================================================================
# SECTION 14C: FORMAL INTERACTION REGRESSION
# ==============================================================================
# Logistic regression: HS ~ indication_group * tnfi_agent_group + covariates
# Fit on full model_df_full (epidemiologic/inferential analysis).
# Concomitant drug covariates: top 15 Drug_* features by mean |SHAP| importance.
# ==============================================================================
print("=" * 60)
print("SECTION 14C — FORMAL INTERACTION REGRESSION")
print("=" * 60)

# --- A. Select top 15 Drug_* covariates by SHAP importance ---
drug_shap = shap_importance[shap_importance["feature"].str.startswith("Drug_")].head(15)
top_drug_covariates = drug_shap["feature"].tolist()
print(f"Top {len(top_drug_covariates)} concomitant drug covariates (by mean |SHAP|):")
for i, d in enumerate(top_drug_covariates, 1):
    shap_val = drug_shap.loc[drug_shap["feature"] == d, "mean_abs_shap"].values[0]
    print(f"  {i:2d}. {d}  (mean |SHAP| = {shap_val:.6f})")

# --- B. Ensure covariates exist in int_df ---
# Some Drug_ features may use dots from make_clean_name; statsmodels needs backticks
available_drug_covs = [c for c in top_drug_covariates if c in int_df.columns]
missing_covs = set(top_drug_covariates) - set(available_drug_covs)
if missing_covs:
    print(f"\n⚠️  Missing from int_df (skipping): {missing_covs}")
print(f"Using {len(available_drug_covs)} drug covariates in regression.")

# --- C. Build formula ---
# Backtick-wrap column names that contain dots or special characters
def bt(col):
    """Wrap in backticks for statsmodels if needed."""
    if any(c in col for c in "./ -"):
        return f"Q('{col}')"
    return col

drug_terms = " + ".join(bt(c) for c in available_drug_covs)

formula = (
    "hs_binary ~ "
    "C(indication_group, Treatment(reference='None')) * "
    "C(tnfi_agent_group, Treatment(reference='None')) + "
    "C(age_group, Treatment(reference='50-59')) + "
    "C(sex_group, Treatment(reference='Female'))"
)
if drug_terms:
    formula += " + " + drug_terms

print(f"\nFormula (abbreviated):")
print(f"  HS ~ indication * tnfi_agent + age_group (9 levels) + sex + {len(available_drug_covs)} Drug covariates")

# --- D. Fit model ---
print("\nFitting logistic regression...")
try:
    interaction_model = smf.logit(formula, data=int_df).fit(
        disp=False, maxiter=200, method="lbfgs"
    )
    print(f"  Converged: {interaction_model.mle_retvals['converged']}")
    print(f"  Log-likelihood: {interaction_model.llf:.1f}")
    print(f"  Pseudo R²: {interaction_model.prsquared:.4f}")
    print(f"  N observations: {interaction_model.nobs:.0f}")
except Exception as e:
    print(f"  ⚠️  Model fitting failed: {e}")
    print("  Trying Newton-Raphson with increased iterations...")
    interaction_model = smf.logit(formula, data=int_df).fit(
        disp=False, maxiter=500, method="newton"
    )

# --- E. Extract results ---
results_df = pd.DataFrame({
    "term": interaction_model.params.index,
    "coef": interaction_model.params.values,
    "se": interaction_model.bse.values,
    "z": interaction_model.tvalues.values,
    "p_raw": interaction_model.pvalues.values,
    "OR": np.exp(interaction_model.params.values),
    "OR_CI_lower": np.exp(interaction_model.conf_int()[0].values),
    "OR_CI_upper": np.exp(interaction_model.conf_int()[1].values),
})

# --- F. Identify interaction terms and apply BH-FDR ---
interaction_mask = results_df["term"].str.contains(":") & ~results_df["term"].str.startswith("Intercept")
interaction_terms = results_df[interaction_mask].copy()

if len(interaction_terms) > 0:
    _, fdr_pvals, _, _ = multipletests(interaction_terms["p_raw"].values, method="fdr_bh")
    interaction_terms["p_fdr"] = fdr_pvals
    # Merge back
    results_df["p_fdr"] = np.nan
    results_df.loc[interaction_mask, "p_fdr"] = fdr_pvals
    print(f"\nBH-FDR correction applied to {len(interaction_terms)} interaction terms.")
else:
    results_df["p_fdr"] = np.nan
    print("\n⚠️  No interaction terms found in model output.")

# --- G. Print interaction term summary ---
print("\n" + "=" * 60)
print("INTERACTION TERM RESULTS (indication × TNFi agent)")
print("=" * 60)

int_results = results_df[interaction_mask].copy()
int_results["OR_str"] = int_results.apply(
    lambda r: f"{r['OR']:.3f} ({r['OR_CI_lower']:.3f}–{r['OR_CI_upper']:.3f})", axis=1
)
int_results["sig"] = int_results["p_fdr"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)

display_cols = ["term", "OR_str", "p_raw", "p_fdr", "sig"]
print(int_results[display_cols].to_string(index=False))

# --- H. Full results summary ---
print("\n" + "=" * 60)
print("FULL MODEL RESULTS — MAIN EFFECTS")
print("=" * 60)
main_effects = results_df[~interaction_mask].copy()
main_effects["OR_str"] = main_effects.apply(
    lambda r: f"{r['OR']:.3f} ({r['OR_CI_lower']:.3f}–{r['OR_CI_upper']:.3f})", axis=1
)
print(main_effects[["term", "OR_str", "p_raw"]].to_string(index=False))


# --- Flag interaction terms with sparse HS case support ---
print("\n--- Sparse Cell Warnings ---")
try:
    for _, row in int_results.iterrows():
        term = row["term"]
        # Extract indication and agent from the term string
        parts = term.split(":")
        if len(parts) != 2:
            continue
        # Check if this cell has < 5 HS cases
        for ind_name in ["Arthropathies", "IBD", "Multiple_Indications", "Psoriasis"]:
            for agent_name in ["adalimumab", "certolizumab_pegol", "etanercept", "golimumab", "infliximab"]:
                if ind_name in parts[0] and agent_name in parts[1]:
                    try:
                        n_cases = ct_cases.loc[ind_name, agent_name]
                        n_total = ct_total.loc[ind_name, agent_name]
                        if n_cases < 5:
                            print(f"  ⚠️  {ind_name} × {agent_name}: {n_cases} HS cases / {n_total} total — UNRELIABLE")
                    except (KeyError, TypeError):
                        pass
except Exception as e:
    print(f"  Sparse cell flagging encountered an error: {e}")

# --- I. Export ---
results_df.to_csv("interaction_regression_full_results.csv", index=False)
if len(int_results) > 0:
    int_results.to_csv("interaction_regression_interaction_terms.csv", index=False)
print("\n✅ Section 14C: Interaction regression complete.")


In [ ]:
# ==============================================================================
# INTERACTION FOREST PLOT — Indication × TNFi Agent
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# --- A. Prepare interaction term data ---
# int_results should already exist from Section 14C
plot_df = int_results.copy()

# Extract indication and agent names from the term strings
def extract_labels(term):
    parts = term.split(":")
    ind = "Unknown"
    agent = "Unknown"
    for label in ["Arthropathies", "IBD", "Multiple_Indications", "Psoriasis"]:
        if label in parts[0]:
            ind = label.replace("_", " ")
            break
    for label in ["adalimumab", "certolizumab_pegol", "etanercept", "golimumab", "infliximab"]:
        if label in parts[1]:
            agent = label.replace("_", " ")
            break
    return ind, agent

plot_df[["Indication", "Agent"]] = plot_df["term"].apply(
    lambda t: pd.Series(extract_labels(t))
)

# Flag sparse cells (< 5 HS cases)
sparse_cells_set = {
    ("IBD", "certolizumab pegol"),
    ("Multiple Indications", "certolizumab pegol"),
    ("Psoriasis", "certolizumab pegol"),
    ("IBD", "etanercept"),
    ("Multiple Indications", "etanercept"),
    ("Arthropathies", "golimumab"),
    ("IBD", "golimumab"),
    ("Multiple Indications", "golimumab"),
    ("Psoriasis", "golimumab"),
    ("Psoriasis", "infliximab"),
}

plot_df["sparse"] = plot_df.apply(
    lambda r: (r["Indication"], r["Agent"]) in sparse_cells_set, axis=1
)

# Cap CIs for display (log scale — cap at 0.01 to 100)
ci_floor, ci_cap = 0.01, 100
plot_df["OR_lo_display"] = plot_df["OR_CI_lower"].clip(lower=ci_floor)
plot_df["OR_hi_display"] = plot_df["OR_CI_upper"].clip(upper=ci_cap)
plot_df["OR_display"] = plot_df["OR"].clip(lower=ci_floor, upper=ci_cap)

# FDR significance
plot_df["fdr_sig"] = plot_df["p_fdr"] < 0.05

# --- B. Order: group by agent, then by indication within each agent ---
agent_order = ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"]
indication_order = ["Arthropathies", "IBD", "Psoriasis", "Multiple Indications"]

plot_df["agent_rank"] = plot_df["Agent"].map({a: i for i, a in enumerate(agent_order)})
plot_df["ind_rank"] = plot_df["Indication"].map({a: i for i, a in enumerate(indication_order)})
plot_df = plot_df.sort_values(["agent_rank", "ind_rank"], ascending=[True, True]).reset_index(drop=True)

# --- C. Build y-positions with gaps between agent groups ---
y_positions = []
y_labels = []
y = 0
current_agent = None
group_boundaries = []

for _, row in plot_df.iterrows():
    if row["Agent"] != current_agent:
        if current_agent is not None:
            group_boundaries.append(y - 0.5)
            y += 1.2  # gap between agent groups
        current_agent = row["Agent"]
    y_positions.append(y)
    y_labels.append(row["Indication"])
    y += 1

plot_df["y_pos"] = y_positions

# --- D. Colors by agent ---
agent_colors = {
    "adalimumab": "#B2182B",
    "infliximab": "#D6604D",
    "etanercept": "#2166AC",
    "certolizumab pegol": "#92C5DE",
    "golimumab": "#4DAF4A",
}

# --- E. Plot ---
fig, ax = plt.subplots(figsize=(10, 12))

for _, row in plot_df.iterrows():
    color = agent_colors.get(row["Agent"], "grey")
    y_pos = row["y_pos"]
    or_val = row["OR_display"]
    lo = row["OR_lo_display"]
    hi = row["OR_hi_display"]

    # Whisker style: dashed for sparse, solid otherwise
    linestyle = "--" if row["sparse"] else "-"
    alpha = 0.35 if row["sparse"] else 0.7
    marker_alpha = 0.3 if row["sparse"] else 1.0

    # CI whisker
    ax.plot([lo, hi], [y_pos, y_pos],
            color=color, linewidth=2, linestyle=linestyle, alpha=alpha)

    # Arrow caps if CI was clipped
    if row["OR_CI_upper"] > ci_cap:
        ax.annotate("", xy=(ci_cap, y_pos), xytext=(ci_cap * 0.7, y_pos),
                     arrowprops=dict(arrowstyle="->", color=color, alpha=alpha, lw=1.5))
    if row["OR_CI_lower"] < ci_floor:
        ax.annotate("", xy=(ci_floor, y_pos), xytext=(ci_floor * 1.5, y_pos),
                     arrowprops=dict(arrowstyle="->", color=color, alpha=alpha, lw=1.5))

    # Point estimate
    marker = "D" if row["fdr_sig"] else "o"
    edgecolor = "black" if row["fdr_sig"] else color
    facecolor = color if not row["sparse"] else "white"
    markersize = 10 if row["fdr_sig"] else 7

    ax.scatter(or_val, y_pos, s=markersize**2, color=facecolor,
               edgecolors=edgecolor, linewidths=1.5 if row["fdr_sig"] else 0.8,
               zorder=5, alpha=marker_alpha, marker=marker)

    # Annotation: OR and significance
    if row["fdr_sig"]:
        ax.text(hi * 1.15, y_pos, f'OR={row["OR"]:.2f}**',
                va="center", fontsize=8, fontweight="bold", color=color)
    elif row["p_raw"] < 0.05 and not row["sparse"]:
        ax.text(hi * 1.15, y_pos, f'OR={row["OR"]:.2f}*',
                va="center", fontsize=8, color=color)

# --- F. Agent group labels ---
agent_label_positions = plot_df.groupby("Agent")["y_pos"].mean().to_dict()
for agent, y_mid in agent_label_positions.items():
    ax.text(ci_floor * 0.55, y_mid, agent.title(),
            va="center", ha="right", fontsize=10, fontweight="bold",
            color=agent_colors.get(agent, "grey"))

# --- G. Group separators ---
for boundary in group_boundaries:
    ax.axhline(y=boundary, color="grey", linewidth=0.5, linestyle=":", alpha=0.4)

# --- H. Reference line and axes ---
ax.axvline(x=1, color="black", linewidth=1, linestyle="-", alpha=0.4)
ax.set_xscale("log")
ax.set_xlim(ci_floor * 0.4, ci_cap * 2.5)

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=9)
ax.invert_yaxis()

ax.set_xlabel("Odds Ratio (95% CI, log scale)", fontsize=11)
ax.set_title("Interaction Effects: Indication × TNFi Agent on HS Reporting\n"
             "(Reference: No recorded indication × No specified TNFi)",
             fontsize=12, fontweight="bold")

# Tick formatting
ax.set_xticks([0.01, 0.1, 0.25, 0.5, 1, 2, 4, 10, 100])
ax.set_xticklabels(["0.01", "0.1", "0.25", "0.5", "1", "2", "4", "10", "100"])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", alpha=0.15)

# --- I. Legend ---
legend_elements = [
    mpatches.Patch(facecolor="grey", edgecolor="grey", alpha=0.7, label="Solid = ≥5 HS cases"),
    mpatches.Patch(facecolor="white", edgecolor="grey", alpha=0.5, label="Hollow/dashed = <5 cases (sparse)"),
    plt.Line2D([0], [0], marker="D", color="black", linestyle="None",
               markersize=8, label="FDR-significant (p < 0.05)"),
    plt.Line2D([0], [0], marker="o", color="grey", linestyle="None",
               markersize=6, markerfacecolor="white", label="Not significant"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8,
          framealpha=0.9, edgecolor="grey")

plt.tight_layout()
plt.savefig("interaction_forest_plot.png", dpi=300, bbox_inches="tight")
plt.savefig("interaction_forest_plot.pdf", bbox_inches="tight")
plt.show()

print("✅ Interaction forest plot saved.")

In [ ]:
# ==============================================================================
# SECTION 14D: ADJUSTED PREDICTED PROBABILITIES (MARGINAL STANDARDIZATION)
# ==============================================================================
# Generate predicted HS probabilities for each indication × TNFi combination,
# marginally standardized over the population distribution of covariates
# (sex, age, drugs) — i.e., g-computation / marginal means.
# ==============================================================================
print("=" * 60)
print("SECTION 14D — ADJUSTED PREDICTED PROBABILITIES")
print("=" * 60)

# --- A. Build prediction grid ---
indic_levels = sorted(int_df["indication_group"].unique())
tnfi_levels = sorted(int_df["tnfi_agent_group"].unique())

print(f"Indication levels: {indic_levels}")
print(f"TNFi agent levels: {tnfi_levels}")

# --- B. Marginal standardization ---
# For each (indication, tnfi) combination, copy the full dataset,
# set indication_group and tnfi_agent_group to the target levels,
# predict for every patient, then average across patients.
# This averages over the actual covariate distribution.

grid_results = []
Z = 1.959963985  # 95% normal quantile
for ind in indic_levels:
    for tnfi in tnfi_levels:
        dfc = int_df.copy()
        dfc["indication_group"]  = ind
        dfc["tnfi_agent_group"]  = tnfi
        preds = np.asarray(interaction_model.predict(dfc), dtype=float)  # numpy, not Series
        n         = preds.size
        mean_prob = preds.mean()
        se        = preds.std(ddof=1) / np.sqrt(n)      # analytic SE of the marginal mean
        ci_lower  = mean_prob - Z * se
        ci_upper  = mean_prob + Z * se
        grid_results.append({
            "indication_group": ind,
            "tnfi_agent_group": tnfi,
            "adjusted_prob": mean_prob,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "adjusted_prob_per_1000": mean_prob * 1000,
            "ci_lower_per_1000": ci_lower * 1000,
            "ci_upper_per_1000": ci_upper * 1000,
        })

pred_grid = pd.DataFrame(grid_results)
print("\nAdjusted predicted HS probability (per 1,000) by indication × TNFi agent:")
pivot_display = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="adjusted_prob_per_1000"
).round(3)
print(pivot_display.to_string())

pred_grid.to_csv("interaction_adjusted_predicted_probs.csv", index=False)
print("\nExported: interaction_adjusted_predicted_probs.csv")

print("\n✅ Section 14D: Adjusted predicted probabilities complete.")


In [ ]:
# ==============================================================================
# SECTION 14E: PUBLICATION-READY HEATMAP OF ADJUSTED PROBABILITIES
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

print("=" * 60)
print("SECTION 14E — INTERACTION HEATMAP")
print("=" * 60)

# --- A. Build matrices ---
prob_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="adjusted_prob_per_1000"
)
ci_lo_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="ci_lower_per_1000"
)
ci_hi_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="ci_upper_per_1000"
)

# Order: None first, then alphabetical
row_order = ["None"] + sorted([r for r in prob_matrix.index if r != "None"])
col_order = ["None"] + sorted([c for c in prob_matrix.columns if c != "None"])
row_order = [r for r in row_order if r in prob_matrix.index]
col_order = [c for c in col_order if c in prob_matrix.columns]

prob_matrix = prob_matrix.reindex(index=row_order, columns=col_order)
ci_lo_matrix = ci_lo_matrix.reindex(index=row_order, columns=col_order)
ci_hi_matrix = ci_hi_matrix.reindex(index=row_order, columns=col_order)

# --- B. Plot ---
fig, ax = plt.subplots(figsize=(max(10, len(col_order) * 1.8), max(5, len(row_order) * 1.2)))

cmap = mcolors.LinearSegmentedColormap.from_list("custom", ["#FFFFFF", "#FEE08B", "#FC8D59", "#D73027"], N=256)
vmax = max(prob_matrix.values.max(), 0.01)

im = ax.imshow(prob_matrix.values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)

# Annotate each cell with prob and CI range
for i in range(prob_matrix.shape[0]):
    for j in range(prob_matrix.shape[1]):
        val = prob_matrix.values[i, j]
        lo = ci_lo_matrix.values[i, j]
        hi = ci_hi_matrix.values[i, j]
        text_color = "white" if val > vmax * 0.65 else "black"
        ax.text(j, i, f"{val:.2f}\n({lo:.2f}–{hi:.2f})",
                ha="center", va="center", fontsize=8, color=text_color, fontweight="bold")

# Axes
ax.set_xticks(range(len(col_order)))
ax.set_xticklabels([c.replace("_", " ").title() for c in col_order],
                    fontsize=9, fontweight="bold", rotation=30, ha="left")
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")

ax.set_yticks(range(len(row_order)))
ax.set_yticklabels([r.replace("_", " ") for r in row_order], fontsize=10, fontweight="bold")

ax.set_xlabel("TNFi Agent", fontsize=11, fontweight="bold")
ax.set_ylabel("Indication Group", fontsize=11, fontweight="bold")
ax.set_title("Adjusted Predicted HS Probability per 1,000\n(Indication × TNFi Agent, Marginally Standardized)",
             fontsize=12, fontweight="bold", pad=40)

cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Predicted HS Rate per 1,000", fontsize=10)

plt.tight_layout()
plt.savefig("interaction_heatmap_adjusted_probs.png", dpi=300, bbox_inches="tight")
plt.savefig("interaction_heatmap_adjusted_probs.pdf", bbox_inches="tight")
plt.show()

print("✅ Section 14E: Interaction heatmap saved.")


## Section 15 — Logistic Regression Benchmark

Penalized (ridge/L2) logistic-regression benchmark against the RF: coefficients + ORs with 95% CIs (`lr_coefs`, `df_forest`), a three-model comparison table, cross-model top-risk enrichment (`enrich_all_models`), and ROC + coefficient forest-plot figures.

In [ ]:
# ==============================================================================
# LOGISTIC REGRESSION BENCHMARK — Does RF add value over LR?
# ==============================================================================
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, brier_score_loss
from scipy.stats import norm

print("=" * 60)
print("LOGISTIC REGRESSION BENCHMARK")
print("=" * 60)

# --- A. Fit L2-regularized LR with 10-fold CV on same selected features ---
lr = LogisticRegressionCV(
    penalty="l2",
    cv=10,
    scoring="roc_auc",
    max_iter=1000,
    random_state=123,
    n_jobs=-1,
)
lr.fit(X_train_sel, y_train, sample_weight=compute_sample_weights(y_train))

lr_preds = lr.predict_proba(X_test_sel)[:, 1]

# --- B. Compare AUCs ---
auc_rf = roc_auc_score(y_test, test_preds_raw)
auc_lr = roc_auc_score(y_test, lr_preds)

print(f"\nROC-AUC comparison:")
print(f"  Random Forest: {auc_rf:.4f}")
print(f"  Logistic Reg:  {auc_lr:.4f}")
print(f"  Difference:    {auc_rf - auc_lr:.4f}")

# --- C. DeLong-inspired bootstrap AUC comparison ---
np.random.seed(42)
n_boot = 2000
auc_diffs = []
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    if len(np.unique(y_test[idx])) < 2:
        continue
    a_rf = roc_auc_score(y_test[idx], test_preds_raw[idx])
    a_lr = roc_auc_score(y_test[idx], lr_preds[idx])
    auc_diffs.append(a_rf - a_lr)

auc_diffs = np.array(auc_diffs)
diff_mean = auc_diffs.mean()
diff_ci_lower = np.percentile(auc_diffs, 2.5)
diff_ci_upper = np.percentile(auc_diffs, 97.5)
# p-value: proportion of bootstrap samples where diff <= 0
p_value = (auc_diffs <= 0).mean()

print(f"\nBootstrap AUC difference (RF - LR):")
print(f"  Mean diff: {diff_mean:.4f}")
print(f"  95% CI:    [{diff_ci_lower:.4f}, {diff_ci_upper:.4f}]")
print(f"  P-value (RF > LR): {p_value:.4f}")

if diff_ci_lower > 0:
    print("  → RF significantly outperforms LR")
elif diff_ci_upper < 0:
    print("  → LR significantly outperforms RF")
else:
    print("  → No significant difference — consider using LR as primary model")

# --- D. PR-AUC comparison ---
prec_rf, rec_rf, _ = precision_recall_curve(y_test, test_preds_raw)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_preds)
prauc_rf = auc(rec_rf, prec_rf)
prauc_lr = auc(rec_lr, prec_lr)

print(f"\nPR-AUC comparison:")
print(f"  Random Forest: {prauc_rf:.4f}")
print(f"  Logistic Reg:  {prauc_lr:.4f}")

# --- E. Brier score comparison ---
brier_rf = brier_score_loss(y_test, test_preds_raw)
brier_lr = brier_score_loss(y_test, lr_preds)

print(f"\nBrier score comparison:")
print(f"  Random Forest: {brier_rf:.4f}")
print(f"  Logistic Reg:  {brier_lr:.4f}")

# --- F. LR coefficients (interpretability advantage) ---
lr_coefs = pd.DataFrame({
    "feature": stable_features,
    "coefficient": lr.coef_[0],
    "odds_ratio": np.exp(lr.coef_[0]),
}).sort_values("coefficient", ascending=False, key=abs).reset_index(drop=True)

print(f"\nLR coefficients (top 20 by magnitude):")
print(lr_coefs.head(20).to_string(index=False))

# --- G. Overlay ROC curves ---
from sklearn.metrics import roc_curve
fpr_rf, tpr_rf, _ = roc_curve(y_test, test_preds_raw)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_preds)

plt.figure(figsize=(7, 6))
plt.plot(fpr_rf, tpr_rf, color="#B2182B", lw=2, label=f"Random Forest (AUC={auc_rf:.3f})")
plt.plot(fpr_lr, tpr_lr, color="#2166AC", lw=2, label=f"Logistic Regression (AUC={auc_lr:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("1 - Specificity (FPR)")
plt.ylabel("Sensitivity (TPR)")
plt.title("ROC Comparison: Random Forest vs Logistic Regression")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_rf_vs_lr.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ LR benchmark complete.")

In [ ]:
# --- LR Coefficient CIs and Forest Plot ---
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("Calculating Coefficient Confidence Intervals (subsampled bootstrap)...")
n_boot = 500
boot_coefs = []
best_C = lr.C_[0]

# Subsample to 50K per bootstrap for computational feasibility
subsample_n = min(50000, len(y_train))

for i in range(n_boot):
    # Stratified subsample
    np.random.seed(i + 7000)
    pos_idx = np.where(y_train == 1)[0]
    neg_idx = np.where(y_train == 0)[0]

    # Keep all positives, subsample negatives
    n_neg_sample = min(subsample_n - len(pos_idx), len(neg_idx))
    neg_sample = np.random.choice(neg_idx, size=n_neg_sample, replace=False)
    sub_idx = np.concatenate([pos_idx, neg_sample])
    np.random.shuffle(sub_idx)

    # Bootstrap within the subsample
    boot_idx = np.random.choice(sub_idx, size=len(sub_idx), replace=True)
    X_boot = X_train_sel[boot_idx]
    y_boot = y_train[boot_idx]
    w_boot = compute_sample_weights(y_boot)

    try:
        boot_lr = LogisticRegression(penalty='l2', C=best_C, max_iter=1000,
                                     solver='lbfgs', random_state=42)
        boot_lr.fit(X_boot, y_boot, sample_weight=w_boot)
        boot_coefs.append(boot_lr.coef_[0])
    except Exception:
        continue

    if (i + 1) % 100 == 0:
        print(f"  Bootstrap {i+1}/{n_boot} complete")

boot_coefs = np.array(boot_coefs)
ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
ci_upper = np.percentile(boot_coefs, 97.5, axis=0)

print(f"  Completed {len(boot_coefs)}/{n_boot} bootstrap iterations")

# Prepare DataFrame
df_forest = pd.DataFrame({
    'feature': stable_features,
    'or': np.exp(lr.coef_[0]),
    'or_lower': np.exp(ci_lower),
    'or_upper': np.exp(ci_upper)
})

# Forest plot: two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8), sharey=False)
sns.set_style("ticks")

top_pos = df_forest.sort_values('or', ascending=False).head(10)
top_neg = df_forest.sort_values('or', ascending=True).head(10)

def draw_forest(ax, data, title, color):
    ax.hlines(y=data['feature'], xmin=data['or_lower'], xmax=data['or_upper'],
              color='#2c3e50', alpha=0.5, lw=2)
    ax.scatter(data['or'], data['feature'], s=80, color=color,
               edgecolors='white', zorder=3)
    ax.axvline(x=1, color='#B2182B', linestyle='--', linewidth=1)
    ax.set_xscale('log')
    ax.set_title(title, fontweight='bold', pad=15)
    ax.set_xlabel('Odds Ratio (95% CI)')
    sns.despine(ax=ax)

draw_forest(ax1, top_pos, "Top 10 Risk Factors", "#2166AC")
draw_forest(ax2, top_neg, "Top 10 Protective Factors", "#006837")

plt.tight_layout(w_pad=6)
plt.savefig("stable_features_forest_plot.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Update three-model comparison with LR ---
auc_lr_val = roc_auc_score(y_test, lr_preds)
prec_lr_arr, rec_lr_arr, _ = precision_recall_curve(y_test, lr_preds)
prauc_lr_val = auc(rec_lr_arr, prec_lr_arr)
brier_lr_val = brier_score_loss(y_test, lr_preds)

# LR enrichment
print("\n--- Logistic Regression Top-Risk Enrichment ---")
enrich_lr = top_risk_enrichment(y_test, lr_preds, [1, 5, 10])
enrich_lr.insert(0, "Model", "LR")
print(enrich_lr.to_string(index=False))

for _, r in enrich_lr.iterrows():
    print(f"  → {r['Top_Pct']}: {r['Fold_enrichment']:.1f}× enrichment "
          f"(95% CI: {r['Fold_enrich_CI_lo']:.1f}–{r['Fold_enrich_CI_hi']:.1f}), "
          f"captures {r['Pct_positives_captured']:.1f}% of cases")

# Full three-model table
lr_row = {
    "Model": "Logistic Regression",
    "ROC-AUC": auc_lr_val,
    "PR-AUC": prauc_lr_val,
    "Brier (raw)": brier_lr_val,
    "Brier (isotonic)": np.nan,  # LR not isotonic-calibrated
    "P/O (raw)": lr_preds.mean() / y_test.mean(),
    "P/O (isotonic)": np.nan,
}

three_model_table = pd.concat([
    three_model_comparison_partial,
    pd.DataFrame([lr_row])
], ignore_index=True)

print("\n" + "=" * 60)
print("FINAL THREE-MODEL COMPARISON")
print("=" * 60)
print(three_model_table.to_string(index=False))

# Combined enrichment across all models
enrich_all_models = pd.concat([enrich_rf_iso, enrich_xgb, enrich_lr], ignore_index=True)
enrich_all_models.to_csv("top_risk_enrichment_all_models.csv", index=False)
three_model_table.to_csv("three_model_comparison.csv", index=False)

# Model recommendation
print("\n" + "=" * 60)
print("MODEL SELECTION RECOMMENDATION")
print("=" * 60)
best_prauc_model = three_model_table.loc[three_model_table["PR-AUC"].idxmax(), "Model"]
best_auc_model = three_model_table.loc[three_model_table["ROC-AUC"].idxmax(), "Model"]
print(f"  Best ROC-AUC:  {best_auc_model} ({three_model_table['ROC-AUC'].max():.4f})")
print(f"  Best PR-AUC:   {best_prauc_model} ({three_model_table['PR-AUC'].max():.4f})")
print(f"\n  → Primary model recommendation based on PR-AUC (rare-event enrichment): {best_prauc_model}")
print("  → ROC-AUC differences may not be statistically significant; PR-AUC")
print("    and top-risk enrichment are more informative for rare-event pharmacovigilance.")

print("\n✅ LR forest plot + three-model comparison complete.")


## Section 15B — Multi-Model Rare-Event Framework (Elastic Net, EBM, Isolation Forest)

Broadens the single–random-forest analysis into a *rare-event detection framework* comparing several algorithms on the **same selected features and held-out test set**. Evaluation is led by **PR-AUC, top-risk enrichment, and calibration** (ROC-AUC is uninformative at 0.07% prevalence).

Framing: the **rare-event logistic regression remains the primary inferential model** (Sections 15–16). These models are complementary risk-stratification tools; concordant performance across very different algorithms is evidence the signal is real, not algorithm-dependent.

**Run order:** run Sections 11, 12C, and 15 first (this block reuses their predictions). A guard in 15F will tell you if something is missing.

Config knobs are at the top of the next cell (`N_BOOT_COMPARE`, `ENRICH_BOOT`, `CAL_FOLDS`, `RUN_ELASTICNET`) so you can trade speed for precision.

In [ ]:
# ==============================================================================
# 15B-0. Config + shared helpers
# ==============================================================================
# ---- knobs (lower = faster) ----
N_BOOT_COMPARE = 1000   # bootstrap reps for PR-AUC CIs in the comparison table
ENRICH_BOOT    = 200    # bootstrap reps for enrichment (table shows point estimates only)
CAL_FOLDS      = 3      # OOF folds for isotonic calibration of the added models
RUN_ELASTICNET = True   # saga can be slow on ~1M rows; set False to skip if needed

try:
    from interpret.glassbox import ExplainableBoostingClassifier
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "interpret"], check=False)
    from interpret.glassbox import ExplainableBoostingClassifier

import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, precision_recall_curve, auc, brier_score_loss)

def _require(names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NameError("Run earlier sections first — missing: " + ", ".join(missing)
                        + ". (Sec 11 -> test_preds_isotonic; Sec 12C -> xgb_test_preds_*; Sec 15 -> lr_preds)")

def oof_isotonic_calibrate(make_model, Xtr, ytr, test_raw, n_splits=CAL_FOLDS, seed=1234):
    """Out-of-fold predictions on train -> fit isotonic -> map onto test_raw.
    Monotonic, so PR-AUC/enrichment ranking is unchanged; only Brier/P-O sharpen."""
    oof = np.full(len(ytr), np.nan)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in skf.split(Xtr, ytr):
        m = make_model()
        try:
            m.fit(Xtr[tr], ytr[tr], sample_weight=compute_sample_weights(ytr[tr]))
        except TypeError:
            m.fit(Xtr[tr], ytr[tr])
        oof[va] = m.predict_proba(Xtr[va])[:, 1]
    eps = 1e-8
    iso = IsotonicRegression(out_of_bounds="clip"); iso.fit(np.clip(oof, eps, 1 - eps), ytr)
    return iso.predict(np.clip(test_raw, eps, 1 - eps))

def prauc(y_true, y_score):
    p, r, _ = precision_recall_curve(y_true, y_score); return auc(r, p)

def bootstrap_prauc(y_true, y_score, n_boot=N_BOOT_COMPARE, seed=42):
    base = prauc(y_true, y_score); rng = np.random.default_rng(seed); vals = []; n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if y_true[idx].sum() < 1: continue
        vals.append(prauc(y_true[idx], y_score[idx]))
    return base, float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

print(f"Helpers ready. Test prevalence: {y_test.mean():.5f}")


### Section 15C — Elastic Net (sparse linear model)

In [ ]:
# ==============================================================================
# 15C. ELASTIC NET LOGISTIC REGRESSION  (sparse linear comparator, PR-tuned)
# ==============================================================================
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
if RUN_ELASTICNET:
    print("Fitting elastic net (saga; can take several minutes; convergence warnings are harmless)...")
    enet_cv = LogisticRegressionCV(
        penalty="elasticnet", solver="saga", l1_ratios=[0.2, 0.5, 0.8],
        Cs=5, cv=5, scoring="average_precision", max_iter=3000,
        n_jobs=-1, random_state=123)
    enet_cv.fit(X_train_sel, y_train, sample_weight=compute_sample_weights(y_train))
    enet_preds = enet_cv.predict_proba(X_test_sel)[:, 1]
    _l1 = float(np.atleast_1d(enet_cv.l1_ratio_)[0]); _C = float(np.atleast_1d(enet_cv.C_)[0])
    enet_preds_cal = oof_isotonic_calibrate(
        lambda: LogisticRegression(penalty="elasticnet", solver="saga",
                                   l1_ratio=_l1, C=_C, max_iter=3000),
        X_train_sel, y_train, enet_preds, seed=201)
    enet_coefs = pd.DataFrame({"feature": stable_features, "coefficient": enet_cv.coef_[0],
        "odds_ratio": np.exp(enet_cv.coef_[0])}).sort_values("coefficient", key=abs, ascending=False).reset_index(drop=True)
    print(f"  l1_ratio={_l1}, C={_C:.4g} | nonzero coefs: {(enet_cv.coef_[0]!=0).sum()}/{len(stable_features)}")
    print(f"  Elastic net PR-AUC: {prauc(y_test, enet_preds):.4f}")
    print(enet_coefs.head(15).to_string(index=False))
else:
    print("RUN_ELASTICNET=False -> skipping elastic net.")


### Section 15D — Explainable Boosting Machine (glass-box GAM)

In [ ]:
# ==============================================================================
# 15D. EXPLAINABLE BOOSTING MACHINE (EBM)  — interpretable per-feature shape functions
# ==============================================================================
# Main effects only (interactions=0) for the cleanest feature-effect plots.
# Set interactions=10 to allow pairwise terms.
print("Fitting EBM (main effects)...")
ebm = ExplainableBoostingClassifier(random_state=42, interactions=0, n_jobs=-1)
ebm.fit(X_train_sel, y_train, sample_weight=compute_sample_weights(y_train))
ebm_preds = ebm.predict_proba(X_test_sel)[:, 1]
ebm_preds_cal = oof_isotonic_calibrate(
    lambda: ExplainableBoostingClassifier(random_state=42, interactions=0, n_jobs=-1),
    X_train_sel, y_train, ebm_preds, seed=202)
# Global term importances (version-robust): interactions=0 -> one term per feature in order
try:
    imp = np.asarray(ebm.term_importances())[:len(stable_features)]
except Exception:
    imp = np.asarray(ebm.explain_global().data()["scores"])[:len(stable_features)]
ebm_imp = pd.DataFrame({"feature": stable_features, "importance": imp}) \
            .sort_values("importance", ascending=False).reset_index(drop=True)
print(f"  EBM PR-AUC: {prauc(y_test, ebm_preds):.4f}")
print(ebm_imp.head(15).to_string(index=False))
# Manuscript shape/feature-effect plots (per feature):
#   g = ebm.explain_global(); g.visualize(k)   # k = term index (interactive)


### Section 15E — Isolation Forest (unsupervised anomaly detection)

In [ ]:
# ==============================================================================
# 15E. ISOLATION FOREST  (unsupervised comparator; labels NOT used in training)
# ==============================================================================
# Included as a comparator: because we have labels, the supervised models are expected
# to outperform it — which supports the supervised framing. Anomaly detection matters
# more for discovering *unknown* paradoxical AEs.
from sklearn.ensemble import IsolationForest
print("Fitting Isolation Forest (labels ignored)...")
iso_f = IsolationForest(n_estimators=300, max_samples="auto", contamination="auto",
                        random_state=42, n_jobs=-1)
iso_f.fit(X_train_sel)
iso_scores = -iso_f.score_samples(X_test_sel)   # higher = more anomalous
print(f"  Isolation Forest PR-AUC: {prauc(y_test, iso_scores):.4f} "
      f"(supervised models expected to exceed this)")
enrich_iso = top_risk_enrichment(y_test, iso_scores, [1, 5, 10])
print(enrich_iso[["Top_Pct", "Fold_enrichment", "Pct_positives_captured"]].to_string(index=False))


In [ ]:
print([v for v in ["test_preds_raw","xgb_test_preds_raw","xgb_test_preds_iso","ebm_preds","lr_preds","y_test"] if v in globals()])

### Section 15F — Unified Multi-Model Comparison
Head-to-head on the held-out test set, led by **PR-AUC** (bootstrap 95% CI), **top-risk enrichment/capture**, and **calibration** (Brier, predicted-to-observed). ROC-AUC reported but de-emphasized. All supervised models are isotonic-calibrated the same way. Enrichment is computed **once per model** and reused by the figure.

In [ ]:
# ==============================================================================
# 15F. UNIFIED MULTI-MODEL COMPARISON
# ==============================================================================
_require(["test_preds_raw", "test_preds_isotonic", "xgb_test_preds_raw",
          "xgb_test_preds_iso", "ebm_preds", "ebm_preds_cal", "iso_scores", "lr_preds"])

# Calibrate LR consistently (isotonic, same helper) so Brier/P-O is apples-to-apples
from sklearn.linear_model import LogisticRegression
lr_preds_cal = oof_isotonic_calibrate(
    lambda: LogisticRegression(max_iter=1000),
    X_train_sel, y_train, lr_preds, seed=203)

# (raw score for ranking/PR/enrichment, calibrated prob for Brier/P-O)
model_score_map = {
    "Random Forest":     (test_preds_raw,     test_preds_isotonic),
    "XGBoost":           (xgb_test_preds_raw, xgb_test_preds_iso),
    "EBM":               (ebm_preds,          ebm_preds_cal),
    "Logistic Reg (L2)": (lr_preds,           lr_preds_cal),
    "Isolation Forest*": (iso_scores,         None),   # unsupervised; calibration N/A
}
if RUN_ELASTICNET and "enet_preds" in globals():
    model_score_map["Elastic Net"] = (enet_preds, enet_preds_cal)

# compute enrichment ONCE per model (reused by the figure)
enrich_by_model = {name: top_risk_enrichment(y_test, raw, [1, 5, 10], n_boot=ENRICH_BOOT)
                   for name, (raw, _c) in model_score_map.items()}

rows = []
for name, (raw, cal) in model_score_map.items():
    pr, lo, hi = bootstrap_prauc(y_test, raw)
    en = enrich_by_model[name]
    def g(pct, col): return en.loc[en.Top_Pct == pct, col].values[0]
    brier = brier_score_loss(y_test, cal) if cal is not None else np.nan
    po = (cal.mean() / y_test.mean()) if cal is not None else np.nan
    rows.append({"Model": name, "PR-AUC": round(pr, 4), "PR-AUC 95% CI": f"{lo:.3f}-{hi:.3f}",
                 "ROC-AUC": round(roc_auc_score(y_test, raw), 4),
                 "Brier (cal)": round(brier, 6) if np.isfinite(brier) else np.nan,
                 "P/O (cal)": round(po, 2) if np.isfinite(po) else np.nan,
                 "Top1% enrich": round(g("Top 1%", "Fold_enrichment"), 1),
                 "Top1% cap%": round(g("Top 1%", "Pct_positives_captured"), 1),
                 "Top5% cap%": round(g("Top 5%", "Pct_positives_captured"), 1),
                 "Top10% cap%": round(g("Top 10%", "Pct_positives_captured"), 1)})
multi_model_comparison = pd.DataFrame(rows)
multi_model_comparison.to_csv("multimodel_comparison.csv", index=False)
print(multi_model_comparison.to_string(index=False))

sup = multi_model_comparison[~multi_model_comparison.Model.str.endswith("*")]
spread = sup["PR-AUC"].max() - sup["PR-AUC"].min()
print(f"\nSupervised PR-AUC range: {sup['PR-AUC'].min():.3f}-{sup['PR-AUC'].max():.3f} "
      f"(spread {spread:.3f}) -> {'robust / algorithm-independent' if spread < 0.02 else 'some separation across models'}")
print("* Isolation Forest is unsupervised; ranking-based metrics only.")


In [ ]:
# ==============================================================================
# 15G. FIGURES — PR curves + top-risk capture (reuses enrich_by_model; no recompute)
# ==============================================================================
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for name, (raw, _c) in model_score_map.items():
    p, r, _ = precision_recall_curve(y_test, raw)
    axes[0].plot(r, p, lw=1.8, label=f"{name} ({auc(r, p):.3f})")
axes[0].axhline(y_test.mean(), ls="--", c="gray", lw=1, label=f"Prevalence ({y_test.mean():.4f})")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("A. Precision-Recall (PR-AUC in legend)"); axes[0].legend(fontsize=7, loc="upper right")

pcts = [1, 5, 10]; width = 0.13; names = list(model_score_map.keys())
for j, name in enumerate(names):
    en = enrich_by_model[name]
    caps = [en.loc[en.Top_Pct == f"Top {p}%", "Pct_positives_captured"].values[0] for p in pcts]
    axes[1].bar(np.arange(len(pcts)) + j * width, caps, width, label=name)
axes[1].set_xticks(np.arange(len(pcts)) + width * (len(names) - 1) / 2)
axes[1].set_xticklabels([f"Top {p}%" for p in pcts]); axes[1].set_ylabel("% of HS cases captured")
axes[1].set_title("B. Case capture by risk percentile"); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig("multimodel_pr_and_capture.png", dpi=300, bbox_inches="tight"); plt.show()
print("Saved multimodel_comparison.csv and multimodel_pr_and_capture.png")


## Section 16 — TNFi-Specific HS Rates & Indication Interactions

Agent-specific HS reporting rates (`tnfi_full`), background HS rates by indication among non-TNFi users (`bg_table`), and the descriptive indication × TNFi-agent rate table (`interaction_table`).

In [ ]:
# ==============================================================================
# FULL TNFi COMPARISON (including golimumab, from model_df_full)
# ==============================================================================
from scipy.stats import fisher_exact

print("=" * 60)
print("FULL TNFi-SPECIFIC HS RATES (from model_df_full)")
print("=" * 60)

tnfi_cols_full = [c for c in model_df_full.columns if c.startswith("TNFi_")]
print(f"TNFi columns in model_df_full: {tnfi_cols_full}")

y_full = (model_df_full["outcome_hs"] == "Yes").astype(int).values

results = []
for col in tnfi_cols_full:
    exposed = model_df_full[col].values == 1
    n_exposed = exposed.sum()
    n_cases_exposed = y_full[exposed].sum()
    rate_exposed = n_cases_exposed / n_exposed if n_exposed > 0 else 0

    n_unexposed = (~exposed).sum()
    n_cases_unexposed = y_full[~exposed].sum()

    table = np.array([
        [n_cases_exposed, n_exposed - n_cases_exposed],
        [n_cases_unexposed, n_unexposed - n_cases_unexposed]
    ])
    or_val, p_val = fisher_exact(table)

    results.append({
        "TNFi": col.replace("TNFi_", "").replace(".", " "),
        "N_users": int(n_exposed),
        "N_HS_cases": int(n_cases_exposed),
        "HS_rate_per_1000": round(rate_exposed * 1000, 3),
        "OR_vs_others": round(or_val, 2),
        "p_value": f"{p_val:.2e}",
    })

tnfi_full = pd.DataFrame(results).sort_values("HS_rate_per_1000", ascending=False)
print(tnfi_full.to_string(index=False))

In [ ]:
# ==============================================================================
# INDICATION × TNFi INTERACTION (from model_df_full)
# ==============================================================================
print("=" * 60)
print("INDICATION × TNFi INTERACTION ANALYSIS")
print("=" * 60)

tnfi_cols_full = [c for c in model_df_full.columns if c.startswith("TNFi_")]
hist_cols = ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]

y_full = (model_df_full["outcome_hs"] == "Yes").astype(int).values

interaction_results = []

for hist in hist_cols:
    if hist not in model_df_full.columns:
        continue
    for tnfi in tnfi_cols_full:
        mask = (model_df_full[hist].values == 1) & (model_df_full[tnfi].values == 1)
        n = mask.sum()
        if n < 5:
            continue
        n_cases = y_full[mask].sum()
        rate = n_cases / n if n > 0 else 0

        interaction_results.append({
            "Indication": hist.replace("Hist_", ""),
            "TNFi": tnfi.replace("TNFi_", "").replace(".", " "),
            "N_patients": int(n),
            "N_HS_cases": int(n_cases),
            "HS_rate_per_1000": round(rate * 1000, 3),
        })

interaction_table = pd.DataFrame(interaction_results).sort_values(
    ["Indication", "HS_rate_per_1000"], ascending=[True, False]
)
print(interaction_table.to_string(index=False))

In [ ]:
# ==============================================================================
# BACKGROUND HS RATES BY INDICATION (non-TNFi users as reference)
# ==============================================================================
print("=" * 60)
print("BACKGROUND HS RATES — NON-TNFi USERS BY INDICATION")
print("=" * 60)

# All FAERS patients NOT in the TNFi cohort
tnf_user_ids = set(model_df_full["compositeid"].unique())
non_tnfi_ids = set(demo_std["compositeid"].unique()) - tnf_user_ids - ids_history_true

# Get comorbidity flags for non-TNFi patients
non_tnfi_comor = comor_matrix[comor_matrix["compositeid"].isin(non_tnfi_ids)].copy()

# HS outcome in non-TNFi patients
non_tnfi_cases = ids_case_true & non_tnfi_ids

hist_cols = ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]
bg_results = []

for hist in hist_cols:
    if hist not in non_tnfi_comor.columns:
        continue
    exposed_ids = set(non_tnfi_comor.loc[non_tnfi_comor[hist] == 1, "compositeid"])
    n = len(exposed_ids)
    n_cases = len(exposed_ids & non_tnfi_cases)
    rate = n_cases / n if n > 0 else 0

    bg_results.append({
        "Indication": hist.replace("Hist_", ""),
        "Population": "Non-TNFi",
        "N_patients": n,
        "N_HS_cases": n_cases,
        "HS_rate_per_1000": round(rate * 1000, 3),
    })

# Compare with TNFi users (any TNFi)
tnfi_comor = comor_matrix[comor_matrix["compositeid"].isin(tnf_user_ids)].copy()
tnfi_cases = ids_case_true & tnf_user_ids

for hist in hist_cols:
    if hist not in tnfi_comor.columns:
        continue
    exposed_ids = set(tnfi_comor.loc[tnfi_comor[hist] == 1, "compositeid"])
    n = len(exposed_ids)
    n_cases = len(exposed_ids & tnfi_cases)
    rate = n_cases / n if n > 0 else 0

    bg_results.append({
        "Indication": hist.replace("Hist_", ""),
        "Population": "TNFi users",
        "N_patients": n,
        "N_HS_cases": n_cases,
        "HS_rate_per_1000": round(rate * 1000, 3),
    })

bg_table = pd.DataFrame(bg_results).sort_values(["Indication", "Population"])
print(bg_table.to_string(index=False))

In [ ]:
from scipy.stats import fisher_exact

print("=" * 60)
print("TNFi vs NON-TNFi RISK BY INDICATION (Fisher's exact)")
print("=" * 60)

for hist in ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]:
    # TNFi users with this indication
    tnfi_exp = set(tnfi_comor.loc[tnfi_comor[hist] == 1, "compositeid"])
    tnfi_n = len(tnfi_exp)
    tnfi_cases = len(tnfi_exp & (ids_case_true & tnf_user_ids))

    # Non-TNFi with this indication
    non_exp = set(non_tnfi_comor.loc[non_tnfi_comor[hist] == 1, "compositeid"])
    non_n = len(non_exp)
    non_cases = len(non_exp & (ids_case_true & non_tnfi_ids))

    table = np.array([
        [tnfi_cases, tnfi_n - tnfi_cases],
        [non_cases, non_n - non_cases]
    ])
    or_val, p_val = fisher_exact(table)

    label = hist.replace("Hist_", "")
    print(f"\n{label}:")
    print(f"  TNFi:     {tnfi_cases}/{tnfi_n} ({tnfi_cases/tnfi_n*1000:.3f}/1000)")
    print(f"  Non-TNFi: {non_cases}/{non_n} ({non_cases/non_n*1000:.3f}/1000)")
    print(f"  OR: {or_val:.2f} (p={p_val:.2e})")

In [ ]:
import math

a, b = 340, 287296 - 340
c, d = 40, 103345 - 40

or_val = (a * d) / (b * c)
log_or_se = math.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lower = math.exp(math.log(or_val) - 1.96 * log_or_se)
ci_upper = math.exp(math.log(or_val) + 1.96 * log_or_se)

print(f"IBD — TNFi vs Non-TNFi:")
print(f"  OR: {or_val:.2f} (95% CI: {ci_lower:.2f}–{ci_upper:.2f})")

## Section 17 — Sex-Stratified Volcano Plots

Sex-stratified disproportionality volcano (mirrored female/male) built from `stats_combined` → `mirrored_volcano_independent_y`.

In [ ]:
# --- 1. Prepare Data from stats_combined ---
# stats_combined is from your Section 4
v_df = stats_combined.copy()
v_df = v_df[v_df['ror'] > 1].copy()
v_df['log2ror'] = np.log2(v_df['ror'])
v_df['neg_log10p'] = -np.log10(v_df['p_value'])

# Mirror Logic: Female left (-), Male right (+)
v_df['x_plot'] = np.where(v_df['sex_group'] == 'Female', -v_df['log2ror'], v_df['log2ror'])

# --- 2. Plotting with Standard Labels ---
history_groups = v_df['history_group'].unique()
fig, axes = plt.subplots(1, len(history_groups), figsize=(18, 9), sharey=False)

if len(history_groups) == 1: axes = [axes]

for i, group in enumerate(history_groups):
    ax = axes[i]
    g_data = v_df[v_df['history_group'] == group].copy()

    # "Highlight Score" to find top points (Significance * Strength)
    g_data['score'] = g_data['neg_log10p'] * g_data['log2ror']

    # Scatter points
    for sex, color in [("Female", "#E41A1C"), ("Male", "#377EB8")]:
        subset = g_data[g_data['sex_group'] == sex]
        ax.scatter(subset['x_plot'], subset['neg_log10p'], c=color, alpha=0.5, s=50, label=sex)

    # Manual Annotations for Top 10 per Sex
    top_labels = g_data.sort_values('score', ascending=False).groupby('sex_group').head(10)
    for _, row in top_labels.iterrows():
        # Add a small offset to prevent label overlapping point
        ax.annotate(row['drug_name'], (row['x_plot'], row['neg_log10p']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.9)

    # Visual Aids
    ax.axvline(0, color='black', lw=1.2)
    ax.set_title(f"Group: {group}", fontweight='bold', size=15, pad=20)
    ax.set_xlabel("Log2 ROR (Risk Strength)", fontweight='bold')

    # Make X-axis ticks absolute (no negative numbers shown)
    ax.set_xticklabels([f"{abs(t):.1g}" for t in ax.get_xticks()])

    if i == 0:
        ax.set_ylabel("-Log10 P-value (Significance)", fontweight='bold')
        ax.legend(title="Sex")

    sns.despine(ax=ax)

plt.suptitle("Sex-Stratified Mirrored Volcano Plots\n(Females on Left | Males on Right)",
             fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig("mirrored_volcano_independent_y.png", dpi=300, bbox_inches="tight")
plt.show()

## Section 18 — Time-to-Onset Analysis

Time from drug start to event for paradoxical new-onset HS by drug class (`class_table`) and an exacerbation-vs-new-onset comparison (`exac_table`); figures `tto_by_drug_class`, `tto_distribution_by_tnfi`.

In [ ]:
# [FIX] define tnfi_names before first use (previously defined later, in the next cell)
tnfi_names = ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"]

# ==============================================================================
# TIME-TO-ONSET ANALYSIS
# ==============================================================================
print("=" * 60)
print("TIME-TO-ONSET ANALYSIS")
print("=" * 60)

# --- Step 1: Parse dates ---
def parse_faers_date(series):
    """Parse FAERS YYYYMMDD dates (stored as float like 20030815.0)."""
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    # Keep only valid 8-digit strings
    valid = s.str.match(r"^\d{8}$")
    s = s.where(valid, None)
    return pd.to_datetime(s, format="%Y%m%d", errors="coerce")

# --- Step 2: Drug mapping (PS drugs for HS cases) ---
case_ids = ids_case_true  # already defined

drug_mapping = drug.loc[
    drug["compositeid"].isin(case_ids) & drug["role_cod"].eq("PS"),
    ["compositeid", "drug_seq", "drug_name"]
].drop_duplicates()
print(f"Drug mapping for cases: {len(drug_mapping):,} rows")

# --- Step 3: Therapy start dates ---
ther_clean = ther.loc[
    ther["compositeid"].isin(case_ids),
    ["compositeid", "drug_seq", "start_dt"]
].copy()
ther_clean["start_date"] = parse_faers_date(ther_clean["start_dt"])
ther_clean = ther_clean.dropna(subset=["start_date"])
print(f"Therapy records with valid start dates: {len(ther_clean):,}")

# --- Step 4: Event dates ---
demo_clean = demo.loc[
    demo["compositeid"].isin(case_ids),
    ["compositeid", "event_dt"]
].copy()
demo_clean["event_date"] = parse_faers_date(demo_clean["event_dt"])
demo_clean = demo_clean.dropna(subset=["event_date"])
print(f"Cases with valid event dates: {len(demo_clean):,}")

# --- Step 5: Merge and calculate TTO ---
tto_data = (
    ther_clean[["compositeid", "drug_seq", "start_date"]]
    .merge(drug_mapping, on=["compositeid", "drug_seq"], how="inner")
    .merge(demo_clean[["compositeid", "event_date"]], on="compositeid", how="inner")
)

tto_data["tto_days"] = (tto_data["event_date"] - tto_data["start_date"]).dt.days
tto_data = tto_data[(tto_data["tto_days"] >= 0) & (tto_data["tto_days"] < 18250)].copy()

tto_data["history_group"] = np.where(
    tto_data["compositeid"].isin(ids_history_true),
    "HS History", "No HS History"
)

tto_data["drug_name_lower"] = tto_data["drug_name"].str.lower().str.strip()

print(f"\nValid TTO records: {len(tto_data):,}")
print(f"  No HS History: {(tto_data['history_group'] == 'No HS History').sum():,}")
print(f"  HS History: {(tto_data['history_group'] == 'HS History').sum():,}")

# --- Step 5b: Completeness stats ---
total_cases = len(case_ids)
cases_with_ther = ther.loc[ther["compositeid"].isin(case_ids), "compositeid"].nunique()
cases_with_start = ther_clean["compositeid"].nunique()
cases_with_event = demo_clean["compositeid"].nunique()
cases_with_tto = tto_data["compositeid"].nunique()

print(f"\n--- TTO Data Completeness ---")
print(f"  Total HS cases: {total_cases}")
print(f"  With any therapy record: {cases_with_ther} ({100*cases_with_ther/total_cases:.1f}%)")
print(f"  With valid start_dt: {cases_with_start} ({100*cases_with_start/total_cases:.1f}%)")
print(f"  With valid event_dt: {cases_with_event} ({100*cases_with_event/total_cases:.1f}%)")
print(f"  With valid TTO (both dates + plausible): {cases_with_tto} ({100*cases_with_tto/total_cases:.1f}%)")

# TNFi-specific completeness
tnfi_cases = tto_data[tto_data["drug_name_lower"].isin(tnfi_names)]["compositeid"].nunique()
paradox_cases = len(ids_case_true - ids_history_true)
paradox_with_tto = tto_data[
    (tto_data["history_group"] == "No HS History") &
    (tto_data["drug_name_lower"].isin(tnfi_names))
]["compositeid"].nunique()

print(f"\n  TNFi cases with valid TTO: {tnfi_cases}")
print(f"  Paradoxical cases (no history): {paradox_cases}")
print(f"  Paradoxical + TNFi + valid TTO: {paradox_with_tto} ({100*paradox_with_tto/paradox_cases:.1f}%)")



In [ ]:
# --- Step 6: Summary by drug and history group ---
tto_summary = (
    tto_data.groupby(["drug_name_lower", "history_group"])
    .agg(
        n_reports=("tto_days", "size"),
        median_days=("tto_days", "median"),
        iqr_lower=("tto_days", lambda x: x.quantile(0.25)),
        iqr_upper=("tto_days", lambda x: x.quantile(0.75)),
    )
    .reset_index()
    .sort_values(["history_group", "median_days"])
)

# --- Step 7: TNFi-specific TTO ---
tnfi_names = ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"]
tnfi_tto = tto_summary[tto_summary["drug_name_lower"].isin(tnfi_names)].copy()
tnfi_tto = tnfi_tto.sort_values(["history_group", "drug_name_lower"])

print("\nTNFi Time-to-Onset Summary:")
print(tnfi_tto.to_string(index=False))

# --- Step 8: TNFi TTO for No HS History only (paradoxical cases) ---
print("\n--- Paradoxical HS (No History) — TNFi TTO ---")
paradoxical_tto = tto_data[
    (tto_data["history_group"] == "No HS History") &
    (tto_data["drug_name_lower"].isin(tnfi_names))
].copy()

for drug_name in tnfi_names:
    subset = paradoxical_tto[paradoxical_tto["drug_name_lower"] == drug_name]
    if len(subset) == 0:
        print(f"  {drug_name}: no valid TTO records")
        continue
    med = subset["tto_days"].median()
    q25 = subset["tto_days"].quantile(0.25)
    q75 = subset["tto_days"].quantile(0.75)
    print(f"  {drug_name}: n={len(subset)}, median={med:.0f} days "
          f"(IQR {q25:.0f}–{q75:.0f}), ~{med/30:.1f} months")

# --- Step 9: TTO distribution plot ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, group in zip(axes, ["No HS History", "HS History"]):
    subset = tto_data[
        (tto_data["history_group"] == group) &
        (tto_data["drug_name_lower"].isin(tnfi_names))
    ]

    colors = {"adalimumab": "#B2182B", "infliximab": "#D6604D",
              "certolizumab pegol": "#F4A582", "golimumab": "#92C5DE",
              "etanercept": "#2166AC"}

    for drug_name in tnfi_names:
        drug_sub = subset[subset["drug_name_lower"] == drug_name]
        if len(drug_sub) < 3:
            continue
        # Cap at 3 years for visualization
        vals = drug_sub["tto_days"].clip(upper=1095)
        ax.hist(vals, bins=30, alpha=0.5, label=f"{drug_name} (n={len(drug_sub)})",
                color=colors.get(drug_name, "grey"))

    ax.set_xlabel("Days to HS Onset")
    ax.set_ylabel("Count")
    ax.set_title(f"Time-to-Onset: {group}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("tto_distribution_by_tnfi.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ Time-to-onset analysis complete.")

In [ ]:
# ==============================================================================
# TTO BY DRUG CLASS — ALL PRIMARY SUSPECT DRUGS
# ==============================================================================
print("=" * 60)
print("TIME-TO-ONSET — ALL DRUGS (No HS History only)")
print("=" * 60)

# Focus on paradoxical cases
paradox_tto = tto_data[tto_data["history_group"] == "No HS History"].copy()

# Summary for all drugs with >= 3 TTO records
drug_tto_summary = (
    paradox_tto.groupby("drug_name_lower")
    .agg(
        n_reports=("tto_days", "size"),
        n_patients=("compositeid", "nunique"),
        median_days=("tto_days", "median"),
        iqr_lower=("tto_days", lambda x: x.quantile(0.25)),
        iqr_upper=("tto_days", lambda x: x.quantile(0.75)),
        mean_days=("tto_days", "mean"),
    )
    .reset_index()
    .sort_values("n_reports", ascending=False)
)

print(f"Drugs with any paradoxical TTO data: {len(drug_tto_summary)}")
print(f"\nTop 30 drugs by TTO report count (>= 3 reports):")
top_drugs = drug_tto_summary[drug_tto_summary["n_reports"] >= 3].head(30)
print(top_drugs.to_string(index=False))

# ==============================================================================
# GROUP BY DRUG CLASS
# ==============================================================================
print("\n" + "=" * 60)
print("TTO BY DRUG CLASS")
print("=" * 60)

drug_classes = {
    "TNFi": ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"],
    "IL-17 inhibitors": ["secukinumab", "ixekizumab", "brodalumab", "bimekizumab"],
    "IL-12/23 inhibitors": ["ustekinumab"],
    "IL-23 inhibitors": ["guselkumab", "risankizumab", "tildrakizumab"],
    "JAK inhibitors": ["tofacitinib", "baricitinib", "upadacitinib", "ruxolitinib"],
    "Retinoids": ["isotretinoin", "acitretin", "tretinoin"],
    "Hormonal contraceptives": ["levonorgestrel", "ethinyl estradiol", "desogestrel",
                                 "norethindrone", "drospirenone", "etonogestrel"],
    "Lithium": ["lithium", "lithium carbonate"],
    "Antibiotics": ["doxycycline", "minocycline", "clindamycin", "rifampin",
                     "amoxicillin", "ciprofloxacin", "metronidazole"],
    "Immunomodulators": ["methotrexate", "azathioprine", "mycophenolate",
                          "mycophenolate mofetil", "cyclosporine", "leflunomide"],
    "Anti-IL-4/13": ["dupilumab"],
    "Corticosteroids": ["prednisone", "prednisolone", "methylprednisolone",
                         "dexamethasone", "hydrocortisone"],
    "NSAIDs": ["ibuprofen", "naproxen", "celecoxib", "diclofenac", "meloxicam"],
    "Anti-CD20": ["rituximab", "ocrelizumab", "obinutuzumab"],
    "Checkpoint inhibitors": ["pembrolizumab", "nivolumab", "ipilimumab",
                               "atezolizumab", "durvalumab"],
}

class_results = []
for class_name, drugs in drug_classes.items():
    subset = paradox_tto[paradox_tto["drug_name_lower"].isin(drugs)]
    n = len(subset)
    n_patients = subset["compositeid"].nunique()
    if n == 0:
        continue

    med = subset["tto_days"].median()
    q25 = subset["tto_days"].quantile(0.25)
    q75 = subset["tto_days"].quantile(0.75)

    # Which specific drugs contributed
    drug_counts = subset["drug_name_lower"].value_counts()
    top_drug = drug_counts.index[0] if len(drug_counts) > 0 else ""

    class_results.append({
        "Drug_class": class_name,
        "N_reports": n,
        "N_patients": n_patients,
        "Median_days": round(med, 0),
        "IQR": f"{q25:.0f}–{q75:.0f}",
        "Months": round(med / 30, 1),
        "Top_drug": top_drug,
    })

class_table = pd.DataFrame(class_results).sort_values("N_reports", ascending=False)
print(class_table.to_string(index=False))

# ==============================================================================
# ALSO SHOW EXACERBATION GROUP FOR COMPARISON
# ==============================================================================
print("\n" + "=" * 60)
print("TTO BY DRUG CLASS — HS HISTORY (Exacerbation)")
print("=" * 60)

exac_tto = tto_data[tto_data["history_group"] == "HS History"].copy()

exac_results = []
for class_name, drugs in drug_classes.items():
    subset = exac_tto[exac_tto["drug_name_lower"].isin(drugs)]
    n = len(subset)
    if n == 0:
        continue
    med = subset["tto_days"].median()
    q25 = subset["tto_days"].quantile(0.25)
    q75 = subset["tto_days"].quantile(0.75)

    exac_results.append({
        "Drug_class": class_name,
        "N_reports": n,
        "N_patients": subset["compositeid"].nunique(),
        "Median_days": round(med, 0),
        "IQR": f"{q25:.0f}–{q75:.0f}",
        "Months": round(med / 30, 1),
    })

exac_table = pd.DataFrame(exac_results).sort_values("N_reports", ascending=False)
print(exac_table.to_string(index=False))

In [ ]:
# ==============================================================================
# TTO BY DRUG CLASS — FIGURE
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Only classes with n >= 4
plot_classes = class_table[class_table["N_reports"] >= 4].copy()
plot_classes = plot_classes.sort_values("Median_days")

fig, ax = plt.subplots(figsize=(10, 6))

y_pos = range(len(plot_classes))
colors = {
    "IL-17 inhibitors": "#E41A1C",
    "JAK inhibitors": "#FF7F00",
    "IL-12/23 inhibitors": "#984EA3",
    "TNFi": "#377EB8",
    "Anti-CD20": "#4DAF4A",
    "Checkpoint inhibitors": "#A65628",
    "Hormonal contraceptives": "#F781BF",
}

for i, (_, row) in enumerate(plot_classes.iterrows()):
    iqr = row["IQR"].split("–")
    q25, q75 = float(iqr[0]), float(iqr[1])
    med = row["Median_days"]
    color = colors.get(row["Drug_class"], "grey")

    ax.plot([q25, q75], [i, i], color=color, linewidth=3, alpha=0.6)
    ax.scatter([med], [i], color=color, s=100, zorder=5, edgecolors="black", linewidth=0.5)
    ax.text(q75 + 30, i, f'n={row["N_reports"]}', va="center", fontsize=9)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_classes["Drug_class"].values)
ax.set_xlabel("Time to HS Onset (days)")
ax.set_title("Time-to-Onset of Drug-Associated HS by Drug Class\n(Paradoxical Cases — No HS Indication History)")
ax.axvline(x=180, color="grey", linestyle="--", alpha=0.4, label="6 months")
ax.axvline(x=365, color="grey", linestyle=":", alpha=0.4, label="12 months")
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim(-20, max(1100, plot_classes["Median_days"].max() + 100))

plt.tight_layout()
plt.savefig("tto_by_drug_class.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ TTO figure saved.")

## Cross-Model Feature Concordance (PI comment #1)
Common permutation-importance metric across the five supervised models; produces `imp_df`, `S`, `missing_flags` used by Figure 4 and eFigure 10.

In [ ]:
# =====================================================================
# PI COMMENT #1 — DO THE MODELS CONVERGE ON THE SAME FEATURES?
# One COMMON importance metric (permutation importance = ROC-AUC drop) for all
# five supervised models on a held-out test subsample, then pairwise Spearman
# correlations of the rankings + top-10 overlap. This is directly comparable
# across models (unlike Fig 4B's mixed SHAP/coef/term-importance metrics).
# =====================================================================
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

feats = list(stable_features)
# reporting-completeness indicators (PI comment #2): keep in models, flag for display
NONBIO = {"Age_missing", "Sex_Unknown", "age_missing", "sex_unknown"}
missing_flags = [f for f in feats if f in NONBIO]
print("Reporting-process (non-biological) features present:", missing_flags or "none found — check names")

MODELS = {"Random Forest": final_rf, "XGBoost": xgb_model, "EBM": ebm,
          "Logistic (L2)": lr, "Elastic Net": enet_cv}
MODELS = {k: v for k, v in MODELS.items() if k in globals() or v is not None}

# subsample the test set for tractable permutation importance: ALL cases + up to 80k controls
rng = np.random.default_rng(0)
pos = np.where(y_test == 1)[0]
neg = rng.choice(np.where(y_test == 0)[0], size=min(80000, int((y_test == 0).sum())), replace=False)
idx = np.concatenate([pos, neg]); rng.shuffle(idx)
Xs, ys = np.asarray(X_test_sel)[idx], np.asarray(y_test)[idx]
print(f"Permutation-importance evaluated on {len(ys):,} reports ({int(ys.sum())} HS cases)\n")

def _manual_pi(mdl, X, y, n=10, seed=42):
    r = np.random.default_rng(seed); base = roc_auc_score(y, mdl.predict_proba(X)[:, 1]); out = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        d = []
        for _ in range(n):
            col = X[:, j].copy(); X[:, j] = r.permutation(X[:, j])
            d.append(base - roc_auc_score(y, mdl.predict_proba(X)[:, 1])); X[:, j] = col
        out[j] = np.mean(d)
    return out

imp = {}
for name, mdl in MODELS.items():
    try:
        r = permutation_importance(mdl, Xs, ys, scoring="roc_auc", n_repeats=10,
                                   random_state=42, n_jobs=-1)
        imp[name] = r.importances_mean
    except Exception as e:
        print(f"  {name}: sklearn permutation_importance failed ({str(e)[:50]}); using manual")
        imp[name] = _manual_pi(mdl, Xs.copy(), ys)
    print(f"  {name}: done")

imp_df = pd.DataFrame(imp, index=feats)

# ---- pairwise Spearman on the common metric (all retained features) ----
names = list(MODELS)
S = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        S.loc[a, b] = spearmanr(imp_df[a], imp_df[b]).correlation
off = S.values[np.triu_indices(len(names), 1)].astype(float)
print("\nPairwise Spearman correlation of feature-importance rankings (all features):")
print(S.round(2).to_string())
print(f"\n>>> mean pairwise Spearman = {np.nanmean(off):.2f}  (range {np.nanmin(off):.2f}-{np.nanmax(off):.2f})")

# ---- top-10 overlap ----
top10 = {n: set(imp_df[n].sort_values(ascending=False).head(10).index) for n in names}
common = set.intersection(*top10.values())
print(f"\n>>> features in ALL {len(names)} models' top 10: {len(common)}")
print("   ", sorted(common))

# ---- biological-only consensus ranking (for figures/discussion) ----
bio_df = imp_df.drop(index=[f for f in missing_flags if f in imp_df.index])
print("\nTop biological features by mean permutation importance (missingness excluded):")
print(bio_df.mean(1).sort_values(ascending=False).head(12).round(4).to_string())

imp_df.to_csv("crossmodel_permutation_importance.csv")
S.to_csv("crossmodel_importance_spearman.csv")
print("\nsaved crossmodel_permutation_importance.csv + crossmodel_importance_spearman.csv")
print("\nText-ready: 'Across the five supervised models, permutation-importance rankings were")
print(f"highly concordant (mean pairwise Spearman rho = {np.nanmean(off):.2f}), with {len(common)} of the top 10")
print("features shared by all models.'  (Back out this framing if rho is low.)")

In [ ]:
import numpy as np, pandas as pd
names = list(imp_df.columns); K = 10
top = {n: set(imp_df[n].sort_values(ascending=False).head(K).index) for n in names}
O = pd.DataFrame(index=names, columns=names, dtype=int)
for a in names:
    for b in names:
        O.loc[a, b] = len(top[a] & top[b])
print(f"Pairwise overlap among top {K} predictors (out of {K}):")
print(O.to_string())
off = [O.loc[a, b] for i, a in enumerate(names) for b in names[i+1:]]
print(f"\nmean pairwise top-{K} overlap = {np.mean(off):.1f} / {K}")

# biological-only intersection (drops Age_missing / Sex_Unknown)
bio = imp_df.drop(index=[f for f in missing_flags if f in imp_df.index])
topb = {n: set(bio[n].sort_values(ascending=False).head(K).index) for n in names}
common_bio = set.intersection(*topb.values())
print(f"\nbiological features in ALL models' top {K}: {len(common_bio)} -> {sorted(common_bio)}")

## Comorbidity × Drug HeatmapsUses all 14 comorbidity groups (including Psoriasis, Dermatologic Conditions, Multiple Sclerosis, Hematologic Malignancy, and Solid Malignancy).Three heatmaps are generated:1. **All HS cases** — overall comorbidity profile by drug2. **No HS History** — drug-induced / new-onset candidates3. **Has HS History** — drug-worsened candidates

In [ ]:
# ==============================================================================
# COMORBIDITY × DRUG HEATMAPS (All 14 Comorbidity Groups)
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

print("=" * 60)
print("COMORBIDITY × DRUG HEATMAPS")
print("=" * 60)

# --- A. Define the drugs to display (top 10 by patient count among HS cases) ---
dt_drug_ps = (
    drug.loc[drug["role_cod"].eq("PS"), ["compositeid", "drug_name"]]
    .dropna(subset=["compositeid", "drug_name"])
    .drop_duplicates()
)
dt_drug_ps["drug_name"] = dt_drug_ps["drug_name"].str.lower().str.strip()

top_drug_counts = (
    dt_drug_ps
    .loc[dt_drug_ps["compositeid"].isin(ids_case_true)]
    .groupby("drug_name")["compositeid"]
    .nunique()
    .sort_values(ascending=False)
)

selected_drugs = top_drug_counts.head(10).index.tolist()

print("Top 10 drugs by HS case count:")
for d in selected_drugs:
    print(f"  {d}: {top_drug_counts[d]:,} patients")

# --- B. Helper: compute prevalence matrix ---
def compute_heatmap_data(patient_ids, drug_table, comor_mat, selected_drugs):
    """
    For a set of patient IDs, compute the prevalence (%) of each comorbidity
    among patients taking each selected drug.
    Returns a tidy DataFrame with columns: drug_name, Comorbidity, Prevalence, Pct_Label, N_patients.
    """
    cohort = (
        drug_table
        .loc[drug_table["compositeid"].isin(patient_ids) &
             drug_table["drug_name"].isin(selected_drugs),
             ["compositeid", "drug_name"]]
        .drop_duplicates()
    )

    # Join comorbidities
    hist_cols = [c for c in comor_mat.columns if c.startswith("Hist_")]
    merged = cohort.merge(comor_mat[["compositeid"] + hist_cols], on="compositeid", how="left")
    merged[hist_cols] = merged[hist_cols].fillna(0)

    # Get per-drug patient counts for annotation
    drug_n = merged.groupby("drug_name")["compositeid"].nunique().to_dict()

    # Prevalence = mean of 0/1 binary flags
    prev = (
        merged.groupby("drug_name")[hist_cols]
        .mean()
        .reset_index()
    )

    # Pivot to tidy
    tidy = prev.melt(
        id_vars="drug_name",
        var_name="Comorbidity",
        value_name="Prevalence"
    )
    tidy["Comorbidity"] = tidy["Comorbidity"].str.replace("Hist_", "", regex=False)
    tidy["Pct_Label"] = (tidy["Prevalence"] * 100).round(1).astype(str) + "%"
    tidy["N_patients"] = tidy["drug_name"].map(drug_n)

    return tidy


# --- C. Prepare the primary-suspect drug table ---
dt_drug_ps = (
    drug.loc[drug["role_cod"].eq("PS"), ["compositeid", "drug_name"]]
    .dropna(subset=["compositeid", "drug_name"])
    .drop_duplicates()
)
dt_drug_ps["drug_name"] = dt_drug_ps["drug_name"].str.lower().str.strip()

print(f"Primary suspect drug-patient pairs: {len(dt_drug_ps):,}")

# --- D. Compute for all three cohorts ---
ids_all_hs = ids_case_true
ids_nohist = ids_case_true - ids_history_true
ids_hist = ids_case_true & ids_history_true

data_all    = compute_heatmap_data(ids_all_hs, dt_drug_ps, comor_matrix, selected_drugs)
data_nohist = compute_heatmap_data(ids_nohist, dt_drug_ps, comor_matrix, selected_drugs)
data_hist   = compute_heatmap_data(ids_hist,   dt_drug_ps, comor_matrix, selected_drugs)

for label, df in [("All HS", data_all), ("No History", data_nohist), ("Has History", data_hist)]:
    drugs_found = df["drug_name"].nunique()
    comors_found = df["Comorbidity"].nunique()
    print(f"  {label}: {drugs_found} drugs × {comors_found} comorbidities")


# --- E. Clean comorbidity display names ---
COMORBIDITY_DISPLAY = {
    "Smoking": "Smoking",
    "Diabetes_T2": "Type 2 Diabetes",
    "Metabolic_Obesity": "Metabolic / Obesity",
    "Depression_Anxiety": "Depression / Anxiety",
    "Follicular_Tetrad": "Follicular Tetrad",
    "PCOS": "PCOS",
    "IBD": "IBD",
    "Arthropathies": "Arthropathies",
    "Hyperlipidemia": "Hyperlipidemia",
    "Psoriasis": "Psoriasis",
    "Dermatologic_Conditions": "Other Derm Conditions",
    "Multiple_Sclerosis": "Multiple Sclerosis",
    "Malignancy_Hematologic": "Hematologic Malignancy",
    "Malignancy_Solid": "Solid Malignancy",
}

# Desired y-axis order (grouped by domain)
COMORBIDITY_ORDER = [
    "IBD", "Arthropathies", "Psoriasis",
    "Depression / Anxiety", "Follicular Tetrad", "PCOS",
    "Type 2 Diabetes", "Metabolic / Obesity", "Hyperlipidemia", "Smoking",
    "Other Derm Conditions", "Multiple Sclerosis",
    "Hematologic Malignancy", "Solid Malignancy",
]


# --- F. Plotting function ---
def plot_comorbidity_heatmap(data, title, cmap_high="#FF4500", figsize=(14, 8)):
    """
    Produce a clean heatmap of comorbidity prevalence by drug.
    """
    df = data.copy()
    df["Comorbidity"] = df["Comorbidity"].map(COMORBIDITY_DISPLAY).fillna(df["Comorbidity"])

    # Filter to only comorbidities in our order list (in case some are missing)
    valid_comor = [c for c in COMORBIDITY_ORDER if c in df["Comorbidity"].unique()]
    df = df[df["Comorbidity"].isin(valid_comor)]

    # Pivot to matrix
    mat = df.pivot_table(index="Comorbidity", columns="drug_name", values="Prevalence", fill_value=0)

    # Reorder
    mat = mat.reindex(index=valid_comor[::-1])  # reversed so top = first in list
    drug_order = [d for d in selected_drugs if d in mat.columns]
    mat = mat[drug_order]

    # Label matrix
    labels = df.pivot_table(index="Comorbidity", columns="drug_name", values="Pct_Label", aggfunc="first")
    labels = labels.reindex(index=valid_comor[::-1])[drug_order].fillna("")

    # Drug N for x-axis labels
    drug_n = df.drop_duplicates("drug_name").set_index("drug_name")["N_patients"].to_dict()

    fig, ax = plt.subplots(figsize=figsize)

    cmap = mcolors.LinearSegmentedColormap.from_list("custom", ["#FFFFFF", cmap_high], N=256)
    vmax = max(mat.values.max(), 0.01)  # avoid 0-range
    im = ax.imshow(mat.values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)

    # Annotate cells
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.values[i, j]
            txt = labels.values[i, j]
            text_color = "white" if val > vmax * 0.65 else "black"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8, color=text_color, fontweight="bold")

    # X-axis: drug names with N
    x_labels = [f"{d.capitalize()}\n(n={drug_n.get(d, '?'):,})" for d in drug_order]
    ax.set_xticks(range(len(drug_order)))
    ax.set_xticklabels(x_labels, fontsize=9, fontweight="bold", rotation=45, ha="left")
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")

    # Y-axis
    ax.set_yticks(range(len(valid_comor[::-1])))
    ax.set_yticklabels(valid_comor[::-1], fontsize=10, fontweight="bold")

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label("Prevalence", fontsize=10)
    cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.0f}%"))

    ax.set_title(title, fontsize=14, fontweight="bold", pad=20)

    # Grid lines
    for i in range(mat.shape[0] + 1):
        ax.axhline(i - 0.5, color="white", linewidth=1.5)
    for j in range(mat.shape[1] + 1):
        ax.axvline(j - 0.5, color="white", linewidth=1.5)

    plt.tight_layout()
    return fig


# --- G. Generate all three heatmaps ---

fig1 = plot_comorbidity_heatmap(
    data_all,
    "Comorbidity Profile by Drug — All HS Cases",
    cmap_high="#FF4500"
)
fig1.savefig("heatmap_all_hs_cases.png", dpi=300, bbox_inches="tight")
plt.show()

fig2 = plot_comorbidity_heatmap(
    data_nohist,
    "Comorbidity Profile by Drug — New-Onset HS (No History)",
    cmap_high="#FF4500"
)
fig2.savefig("heatmap_nohist_new_onset.png", dpi=300, bbox_inches="tight")
plt.show()

fig3 = plot_comorbidity_heatmap(
    data_hist,
    "Comorbidity Profile by Drug — Existing HS (Has History)",
    cmap_high="#00008B"
)
fig3.savefig("heatmap_hist_existing.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ Heatmaps complete — all 14 comorbidity groups included.")
print("Saved: heatmap_all_hs_cases.png, heatmap_nohist_new_onset.png, heatmap_hist_existing.png")


In [ ]:
# ==============================================================================
# SIDE-BY-SIDE HEATMAP: No History vs Has History
# ==============================================================================
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(24, 10))

def _fill_heatmap_ax(ax, data, title, cmap_high, selected_drugs):
    """Helper to draw heatmap on a given axis."""
    df = data.copy()
    df["Comorbidity"] = df["Comorbidity"].map(COMORBIDITY_DISPLAY).fillna(df["Comorbidity"])
    valid_comor = [c for c in COMORBIDITY_ORDER if c in df["Comorbidity"].unique()]
    df = df[df["Comorbidity"].isin(valid_comor)]

    mat = df.pivot_table(index="Comorbidity", columns="drug_name", values="Prevalence", fill_value=0)
    mat = mat.reindex(index=valid_comor[::-1])
    drug_order = [d for d in selected_drugs if d in mat.columns]
    mat = mat[drug_order]

    labels = df.pivot_table(index="Comorbidity", columns="drug_name", values="Pct_Label", aggfunc="first")
    labels = labels.reindex(index=valid_comor[::-1])[drug_order].fillna("")

    drug_n = df.drop_duplicates("drug_name").set_index("drug_name")["N_patients"].to_dict()

    cmap = mcolors.LinearSegmentedColormap.from_list("c", ["#FFFFFF", cmap_high], N=256)
    vmax = max(mat.values.max(), 0.01)
    im = ax.imshow(mat.values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.values[i, j]
            txt = labels.values[i, j]
            tc = "white" if val > vmax * 0.65 else "black"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7, color=tc, fontweight="bold")

    x_labels = [f"{d.capitalize()}\n(n={drug_n.get(d, '?'):,})" for d in drug_order]
    ax.set_xticks(range(len(drug_order)))
    ax.set_xticklabels(x_labels, fontsize=8, fontweight="bold", rotation=45, ha="left")
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    ax.set_yticks(range(len(valid_comor[::-1])))
    ax.set_yticklabels(valid_comor[::-1], fontsize=9, fontweight="bold")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=15)

    for i in range(mat.shape[0] + 1):
        ax.axhline(i - 0.5, color="white", linewidth=1)
    for j in range(mat.shape[1] + 1):
        ax.axvline(j - 0.5, color="white", linewidth=1)

_fill_heatmap_ax(ax_a, data_nohist, "New-Onset HS (No History)", "#FF4500", selected_drugs)
_fill_heatmap_ax(ax_b, data_hist, "Existing HS (Has History)", "#00008B", selected_drugs)

plt.suptitle("Comorbidity Profile Comparison: New-Onset vs Existing HS",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig("heatmap_sidebyside_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Side-by-side comparison saved: heatmap_sidebyside_comparison.png")


## Section 20 — Master Output (All Tables & Figures)

Assembles the manuscript tables (T4–T12) and consolidated CSV/Excel outputs from the objects built above.

In [ ]:
# ==============================================================================
# MASTER OUTPUT CELL — All Tables (T4–T12) and Figures (F7–F11)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import fisher_exact
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, brier_score_loss
)
import warnings
warnings.filterwarnings("ignore")

save_dir = "/content/drive/MyDrive/FAERS Files"
fig_dir  = f"{save_dir}/Figures"
tbl_dir  = f"{save_dir}/Tables"
import os
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(tbl_dir, exist_ok=True)

# ==============================================================================
# HELPER: 2×2 table with OR + CI + Fisher P
# ==============================================================================
import math

def fisher_or_ci(a, b, c, d):
    """a=exposed cases, b=exposed non-cases, c=unexposed cases, d=unexposed non-cases."""
    if 0 in (a, b, c, d):
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    or_val = (a * d) / (b * c)
    se = math.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_lo = math.exp(math.log(or_val) - 1.96 * se)
    ci_hi = math.exp(math.log(or_val) + 1.96 * se)
    table = np.array([[int(round(a)), int(round(b))],
                      [int(round(c)), int(round(d))]])
    _, p = fisher_exact(table)
    return or_val, ci_lo, ci_hi, p


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T10 — RF MODEL PERFORMANCE                                              ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("=" * 70)
print("T10 — RANDOM FOREST MODEL PERFORMANCE")
print("=" * 70)

# A. AUC + bootstrap CI
base_auc = roc_auc_score(y_test, test_preds_raw)
np.random.seed(42)
n_boot = 2000
boot_aucs = []
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    if len(np.unique(y_test[idx])) < 2:
        continue
    boot_aucs.append(roc_auc_score(y_test[idx], test_preds_raw[idx]))
auc_ci = (np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5))

# B. PR-AUC
prec_arr, rec_arr, _ = precision_recall_curve(y_test, test_preds_raw)
pr_auc_val = auc(rec_arr, prec_arr)

# C. Brier
brier_raw = brier_score_loss(y_test, test_preds_raw)
brier_cal = brier_score_loss(y_test, test_preds_final)

# D. Thresholds
fpr, tpr, thresholds_roc = roc_curve(y_test, test_preds_raw)
j_scores = tpr - fpr
best_j = np.argmax(j_scores)
youden_t = thresholds_roc[best_j]
youden_sens = tpr[best_j]
youden_spec = 1 - fpr[best_j]

hs_idx = np.where(tpr >= 0.90)[0]
if len(hs_idx) > 0:
    best_hs = hs_idx[np.argmax(1 - fpr[hs_idx])]
    hs_t = thresholds_roc[best_hs]
    hs_sens = tpr[best_hs]
    hs_spec = 1 - fpr[best_hs]
else:
    hs_t, hs_sens, hs_spec = youden_t, youden_sens, youden_spec

# E. Confusion matrices + bootstrap CIs at Youden
def threshold_metrics(y_true, y_prob, t):
    yp = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_true, yp, labels=[0, 1])
    tn, fp_, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp_) if (tn + fp_) > 0 else 0
    ppv  = tp / (tp + fp_) if (tp + fp_) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    return {"Sens": sens, "Spec": spec, "PPV": ppv, "NPV": npv,
            "TP": tp, "FP": fp_, "FN": fn, "TN": tn}

m_youden = threshold_metrics(y_test, test_preds_raw, youden_t)
m_highsens = threshold_metrics(y_test, test_preds_raw, hs_t)

# Bootstrap CIs at Youden
boot_m = {"sens": [], "spec": [], "ppv": [], "npv": []}
np.random.seed(42)
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    yp = (test_preds_raw[idx] >= youden_t).astype(int)
    cm_b = confusion_matrix(y_test[idx], yp, labels=[0, 1])
    tn_b, fp_b, fn_b, tp_b = cm_b.ravel()
    boot_m["sens"].append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
    boot_m["spec"].append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
    boot_m["ppv"].append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0)
    boot_m["npv"].append(tn_b / (tn_b + fn_b) if (tn_b + fn_b) > 0 else 0)

# Print T10
print(f"\nROC-AUC:  {base_auc:.4f}  (95% CI: {auc_ci[0]:.4f}–{auc_ci[1]:.4f})")
print(f"PR-AUC:   {pr_auc_val:.4f}")
print(f"Brier:    Raw {brier_raw:.4f}  |  Isotonic {brier_cal:.4f}")

print(f"\n{'Metric':<12} {'Youden':>12} {'High-Sens':>12}")
print("-" * 38)
print(f"{'Threshold':<12} {youden_t:>12.4f} {hs_t:>12.4f}")
print(f"{'Sensitivity':<12} {m_youden['Sens']:>12.3f} {m_highsens['Sens']:>12.3f}")
print(f"{'Specificity':<12} {m_youden['Spec']:>12.3f} {m_highsens['Spec']:>12.3f}")
print(f"{'PPV':<12} {m_youden['PPV']:>12.4f} {m_highsens['PPV']:>12.4f}")
print(f"{'NPV':<12} {m_youden['NPV']:>12.4f} {m_highsens['NPV']:>12.4f}")
print(f"{'TP':<12} {m_youden['TP']:>12} {m_highsens['TP']:>12}")
print(f"{'FP':<12} {m_youden['FP']:>12} {m_highsens['FP']:>12}")
print(f"{'FN':<12} {m_youden['FN']:>12} {m_highsens['FN']:>12}")
print(f"{'TN':<12} {m_youden['TN']:>12} {m_highsens['TN']:>12}")

print(f"\nBootstrap 95% CIs at Youden threshold:")
for k in ["sens", "spec", "ppv", "npv"]:
    lo, hi = np.percentile(boot_m[k], 2.5), np.percentile(boot_m[k], 97.5)
    print(f"  {k.upper()}: {lo:.4f}–{hi:.4f}")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T11 — RF vs LR BENCHMARK                                                ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("T11 — RF vs LR BENCHMARK COMPARISON")
print("=" * 70)

auc_rf = roc_auc_score(y_test, test_preds_raw)
auc_lr = roc_auc_score(y_test, lr_preds)

prec_rf, rec_rf, _ = precision_recall_curve(y_test, test_preds_raw)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_preds)
prauc_rf = auc(rec_rf, prec_rf)
prauc_lr = auc(rec_lr, prec_lr)

brier_rf = brier_score_loss(y_test, test_preds_raw)
brier_lr = brier_score_loss(y_test, lr_preds)

np.random.seed(42)
auc_diffs = []
for _ in range(n_boot):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    if len(np.unique(y_test[idx])) < 2:
        continue
    auc_diffs.append(
        roc_auc_score(y_test[idx], test_preds_raw[idx]) -
        roc_auc_score(y_test[idx], lr_preds[idx])
    )
auc_diffs = np.array(auc_diffs)
diff_ci = (np.percentile(auc_diffs, 2.5), np.percentile(auc_diffs, 97.5))
p_rf_gt_lr = (auc_diffs <= 0).mean()

print(f"\n{'Metric':<16} {'RF':>10} {'LR':>10} {'Diff':>10}")
print("-" * 48)
print(f"{'ROC-AUC':<16} {auc_rf:>10.4f} {auc_lr:>10.4f} {auc_rf - auc_lr:>10.4f}")
print(f"{'PR-AUC':<16} {prauc_rf:>10.4f} {prauc_lr:>10.4f} {prauc_rf - prauc_lr:>10.4f}")
print(f"{'Brier':<16} {brier_rf:>10.4f} {brier_lr:>10.4f} {brier_rf - brier_lr:>10.4f}")
print(f"\nAUC Diff 95% CI: [{diff_ci[0]:.4f}, {diff_ci[1]:.4f}]")
print(f"P-value (RF > LR): {p_rf_gt_lr:.4f}")
if diff_ci[0] > 0:
    print("→ RF significantly outperforms LR")
elif diff_ci[1] < 0:
    print("→ LR significantly outperforms RF")
else:
    print("→ No statistically significant difference")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T12 — LR COEFFICIENTS TOP 20                                            ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("T12 — LR COEFFICIENTS (TOP 20 BY MAGNITUDE)")
print("=" * 70)

lr_coefs = pd.DataFrame({
    "Feature": stable_features,
    "Coefficient": lr.coef_[0],
    "OR": np.exp(lr.coef_[0]),
    "Direction": np.where(lr.coef_[0] > 0, "↑ Risk", "↓ Protective"),
}).sort_values("Coefficient", ascending=False, key=abs).reset_index(drop=True)

print(lr_coefs.head(20).to_string(index=False))
lr_coefs.to_csv(f"{tbl_dir}/T12_LR_coefficients.csv", index=False)


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T4 — TNFi HS RATES BY AGENT (from model_df_full)                        ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("T4 — TNFi-SPECIFIC HS RATES BY AGENT")
print("=" * 70)

tnfi_cols_full = [c for c in model_df_full.columns if c.startswith("TNFi_")]
y_full = (model_df_full["outcome_hs"] == "Yes").astype(int).values

t4_rows = []
for col in tnfi_cols_full:
    exposed = model_df_full[col].values == 1
    n_exp   = exposed.sum()
    cases_exp = y_full[exposed].sum()
    rate_exp  = cases_exp / n_exp * 1000 if n_exp > 0 else 0

    n_unexp   = (~exposed).sum()
    cases_unexp = y_full[~exposed].sum()

    or_val, ci_lo, ci_hi, p = fisher_or_ci(
        cases_exp, n_exp - cases_exp,
        cases_unexp, n_unexp - cases_unexp
    )

    t4_rows.append({
        "TNFi": col.replace("TNFi_", "").replace(".", " "),
        "N_users": int(n_exp),
        "N_HS_cases": int(cases_exp),
        "Rate_per_1000": round(rate_exp, 3),
        "OR_vs_others": round(or_val, 2),
        "OR_95CI": f"{ci_lo:.2f}–{ci_hi:.2f}",
        "P_value": f"{p:.2e}" if p < 0.001 else f"{p:.4f}",
    })

T4 = pd.DataFrame(t4_rows).sort_values("Rate_per_1000", ascending=False)
print(T4.to_string(index=False))
T4.to_csv(f"{tbl_dir}/T4_TNFi_rates_by_agent.csv", index=False)


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T5 — INDICATION-STRATIFIED TNFi vs NON-TNFi RATES                       ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("T5 — TNFi vs NON-TNFi HS RATES BY INDICATION")
print("=" * 70)

# TNFi user IDs
tnf_user_ids = set(model_df_full["compositeid"].unique())
non_tnfi_ids = set(demo_std["compositeid"].unique()) - tnf_user_ids - ids_history_true

# Comorbidity flags for each group
tnfi_comor = comor_matrix[comor_matrix["compositeid"].isin(tnf_user_ids)].copy()
non_tnfi_comor = comor_matrix[comor_matrix["compositeid"].isin(non_tnfi_ids)].copy()

t5_rows = []
for hist in ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]:
    label = hist.replace("Hist_", "")

    # TNFi users with this indication
    tnfi_exp_ids = set(tnfi_comor.loc[tnfi_comor[hist] == 1, "compositeid"])
    tnfi_n = len(tnfi_exp_ids)
    tnfi_cases = len(tnfi_exp_ids & ids_case_true)
    tnfi_rate = tnfi_cases / tnfi_n * 1000 if tnfi_n > 0 else 0

    # Non-TNFi users with this indication
    non_exp_ids = set(non_tnfi_comor.loc[non_tnfi_comor[hist] == 1, "compositeid"])
    non_n = len(non_exp_ids)
    non_cases = len(non_exp_ids & ids_case_true)
    non_rate = non_cases / non_n * 1000 if non_n > 0 else 0

    or_val, ci_lo, ci_hi, p = fisher_or_ci(
        tnfi_cases, tnfi_n - tnfi_cases,
        non_cases, non_n - non_cases
    )

    t5_rows.append({
        "Indication": label,
        "TNFi_N": tnfi_n, "TNFi_cases": tnfi_cases, "TNFi_rate_per_1000": round(tnfi_rate, 3),
        "NonTNFi_N": non_n, "NonTNFi_cases": non_cases, "NonTNFi_rate_per_1000": round(non_rate, 3),
        "OR": round(or_val, 2),
        "OR_95CI": f"{ci_lo:.2f}–{ci_hi:.2f}",
        "P_value": f"{p:.2e}" if p < 0.001 else f"{p:.4f}",
    })

T5 = pd.DataFrame(t5_rows)
print(T5.to_string(index=False))
T5.to_csv(f"{tbl_dir}/T5_TNFi_vs_nonTNFi_by_indication.csv", index=False)


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  T6 — IL-17i vs TNFi RATES BY INDICATION                                 ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("T6 — IL-17i vs TNFi HS RATES BY INDICATION")
print("=" * 70)

# Build IL-17i user set from full drug table
IL17_INHIBITORS = ["secukinumab", "ixekizumab", "brodalumab", "bimekizumab"]
TNF_INHIBITORS  = ["adalimumab", "infliximab", "etanercept", "golimumab", "certolizumab pegol"]

drug_ps = drug.loc[drug["role_cod"].eq("PS"), ["compositeid", "drug_name"]].dropna().drop_duplicates()
drug_ps["drug_lower"] = drug_ps["drug_name"].str.lower().str.strip()

il17_user_ids = set(drug_ps.loc[drug_ps["drug_lower"].isin(IL17_INHIBITORS), "compositeid"])
tnfi_user_ids_full = set(drug_ps.loc[drug_ps["drug_lower"].isin(TNF_INHIBITORS), "compositeid"])

# Exclude patients with HS history from both groups
il17_user_ids = il17_user_ids - ids_history_true
tnfi_user_ids_t6 = tnfi_user_ids_full - ids_history_true

# Get comorbidity flags
il17_comor = comor_matrix[comor_matrix["compositeid"].isin(il17_user_ids)].copy()
tnfi_comor_t6 = comor_matrix[comor_matrix["compositeid"].isin(tnfi_user_ids_t6)].copy()

t6_rows = []
for hist in ["Hist_Arthropathies", "Hist_IBD", "Hist_Psoriasis"]:
    label = hist.replace("Hist_", "")

    # IL-17i
    il17_exp = set(il17_comor.loc[il17_comor[hist] == 1, "compositeid"]) if hist in il17_comor.columns else set()
    il17_n = len(il17_exp)
    il17_cases = len(il17_exp & ids_case_true)
    il17_rate = il17_cases / il17_n * 1000 if il17_n > 0 else 0

    # TNFi
    tnfi_exp = set(tnfi_comor_t6.loc[tnfi_comor_t6[hist] == 1, "compositeid"]) if hist in tnfi_comor_t6.columns else set()
    tnfi_n = len(tnfi_exp)
    tnfi_cases = len(tnfi_exp & ids_case_true)
    tnfi_rate = tnfi_cases / tnfi_n * 1000 if tnfi_n > 0 else 0

    if il17_n > 0 and tnfi_n > 0:
        or_val, ci_lo, ci_hi, p = fisher_or_ci(
            il17_cases, il17_n - il17_cases,
            tnfi_cases, tnfi_n - tnfi_cases
        )
        or_str = f"{or_val:.2f}"
        ci_str = f"{ci_lo:.2f}–{ci_hi:.2f}"
        p_str = f"{p:.2e}" if p < 0.001 else f"{p:.4f}"
    else:
        or_str, ci_str, p_str = "N/A", "N/A", "N/A"

    t6_rows.append({
        "Indication": label,
        "IL17i_N": il17_n, "IL17i_cases": il17_cases, "IL17i_rate_per_1000": round(il17_rate, 3),
        "TNFi_N": tnfi_n, "TNFi_cases": tnfi_cases, "TNFi_rate_per_1000": round(tnfi_rate, 3),
        "OR_IL17i_vs_TNFi": or_str,
        "OR_95CI": ci_str,
        "P_value": p_str,
    })

T6 = pd.DataFrame(t6_rows)
print(T6.to_string(index=False))
T6.to_csv(f"{tbl_dir}/T6_IL17i_vs_TNFi_by_indication.csv", index=False)


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  F7 — TNFi vs NON-TNFi GROUPED BAR CHART BY INDICATION                   ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("F7 — TNFi vs NON-TNFi INDICATION-STRATIFIED BAR CHART")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 6))

indications = T5["Indication"].tolist()
tnfi_rates  = T5["TNFi_rate_per_1000"].tolist()
non_rates   = T5["NonTNFi_rate_per_1000"].tolist()
or_vals_t5  = T5["OR"].tolist()
p_vals_t5   = T5["P_value"].tolist()

x = np.arange(len(indications))
width = 0.30

bars1 = ax.bar(x - width/2, non_rates, width, label="Non-TNFi Users",
               color="#4393C3", edgecolor="white", zorder=3)
bars2 = ax.bar(x + width/2, tnfi_rates, width, label="TNFi Users",
               color="#D6604D", edgecolor="white", zorder=3)

# Annotate bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# OR + P annotations above each group
ymax = max(max(tnfi_rates), max(non_rates))
for i in range(len(indications)):
    p_display = p_vals_t5[i]
    try:
        p_float = float(p_display)
        if p_float < 0.001:
            p_display = "P<0.001"
        else:
            p_display = f"P={p_float:.3f}"
    except:
        p_display = f"P={p_display}"
    ax.text(x[i], ymax * 1.08, f"OR={or_vals_t5[i]}\n{p_display}",
            ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#333333")

ax.set_ylabel("HS Rate per 1,000 Patients", fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_xticks(x)
ax.set_xticklabels(indications, fontsize=12, fontweight="bold")
ax.set_title("HS Reporting Rate: TNFi vs Non-TNFi Users by Indication", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
ax.set_ylim(0, ymax * 1.35)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(f"{fig_dir}/F7_TNFi_vs_nonTNFi_by_indication.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ F7 saved")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  F8 — IL-17i vs TNFi HEAD-TO-HEAD BY INDICATION                          ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("F8 — IL-17i vs TNFi HEAD-TO-HEAD BY INDICATION")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 6))

indications_t6 = T6["Indication"].tolist()
il17_rates_t6  = T6["IL17i_rate_per_1000"].tolist()
tnfi_rates_t6  = T6["TNFi_rate_per_1000"].tolist()
or_t6          = T6["OR_IL17i_vs_TNFi"].tolist()
p_t6           = T6["P_value"].tolist()

x = np.arange(len(indications_t6))
width = 0.30

bars1 = ax.bar(x - width/2, tnfi_rates_t6, width, label="TNFi Users",
               color="#D6604D", edgecolor="white", zorder=3)
bars2 = ax.bar(x + width/2, il17_rates_t6, width, label="IL-17i Users",
               color="#5E3C99", edgecolor="white", zorder=3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars2:
    ht = bar.get_height()
    if ht > 0:
        ax.text(bar.get_x() + bar.get_width()/2, ht + 0.02,
                f"{ht:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 0.02,
                "N/A", ha="center", va="bottom", fontsize=8, color="gray")

ymax = max(max(il17_rates_t6), max(tnfi_rates_t6))
for i in range(len(indications_t6)):
    p_display = p_t6[i]
    try:
        p_float = float(p_display)
        if p_float < 0.001:
            p_display = "P<0.001"
        else:
            p_display = f"P={p_float:.3f}"
    except:
        p_display = f"P={p_display}"
    ax.text(x[i], ymax * 1.08, f"OR={or_t6[i]}\n{p_display}",
            ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#333333")

ax.set_ylabel("HS Rate per 1,000 Patients", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(indications_t6, fontsize=12, fontweight="bold")
ax.set_title("HS Reporting Rate: IL-17i vs TNFi Users by Indication\n(No HS History Only)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
ax.set_ylim(0, ymax * 1.35 if ymax > 0 else 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(f"{fig_dir}/F8_IL17i_vs_TNFi_by_indication.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ F8 saved")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  F9 — TTO DRUG CLASS COMPARISON (Forest-style plot)                       ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("F9 — TIME-TO-ONSET BY DRUG CLASS")
print("=" * 70)

# Build drug class mapping
DRUG_CLASSES = {
    "TNF inhibitors": ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"],
    "IL-17 inhibitors": ["secukinumab", "ixekizumab", "brodalumab"],
    "IL-12/23 inhibitors": ["ustekinumab"],
    "JAK inhibitors": ["tofacitinib", "baricitinib", "upadacitinib", "ruxolitinib"],
    "Retinoids": ["isotretinoin", "acitretin"],
    "Antimetabolites": ["methotrexate", "azathioprine", "mycophenolate", "leflunomide"],
    "Hormonal agents": ["testosterone", "levonorgestrel"],
    "Cytotoxic agents": ["cytarabine", "omacetaxine"],
}

class_map = {}
for cls, drugs in DRUG_CLASSES.items():
    for d in drugs:
        class_map[d] = cls

paradox_tto = tto_data[tto_data["history_group"] == "No HS History"].copy()
paradox_tto["drug_class"] = paradox_tto["drug_name_lower"].map(class_map)
paradox_tto = paradox_tto.dropna(subset=["drug_class"])

class_summary = (
    paradox_tto.groupby("drug_class")
    .agg(
        n=("tto_days", "size"),
        median=("tto_days", "median"),
        q25=("tto_days", lambda x: x.quantile(0.25)),
        q75=("tto_days", lambda x: x.quantile(0.75)),
    )
    .reset_index()
    .query("n >= 3")
    .sort_values("median")
)

print(class_summary.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

colors_map = {
    "IL-17 inhibitors": "#E41A1C", "JAK inhibitors": "#FF7F00",
    "IL-12/23 inhibitors": "#984EA3", "TNF inhibitors": "#377EB8",
    "Retinoids": "#4DAF4A", "Antimetabolites": "#F781BF",
    "Hormonal agents": "#A65628", "Cytotoxic agents": "#999999",
}

y_pos = range(len(class_summary))
for i, (_, row) in enumerate(class_summary.iterrows()):
    color = colors_map.get(row["drug_class"], "#666666")
    ax.errorbar(
        row["median"], i,
        xerr=[[row["median"] - row["q25"]], [row["q75"] - row["median"]]],
        fmt="o", color=color, markersize=10, capsize=6,
        elinewidth=2.5, capthick=2, zorder=3
    )
    ax.text(row["q75"] + 8, i,
            f'{int(row["median"])}d (n={int(row["n"])})',
            va="center", fontsize=9, fontweight="bold")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(class_summary["drug_class"].tolist(), fontsize=10, fontweight="bold")
ax.set_xlabel("Time to HS Onset (days)", fontsize=12, fontweight="bold")
ax.set_title("Time-to-Onset by Drug Class\n(Paradoxical/New-Onset HS Only)", fontsize=13, fontweight="bold")
ax.axvline(x=90, color="gray", linestyle="--", alpha=0.4, label="90 days")
ax.axvline(x=180, color="gray", linestyle=":", alpha=0.4, label="180 days")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(f"{fig_dir}/F9_TTO_drug_class.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ F9 saved")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  F10 — ROC CURVES: RF vs LR OVERLAY                                      ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("F10 — ROC CURVES (RF vs LR)")
print("=" * 70)

fpr_rf, tpr_rf, _ = roc_curve(y_test, test_preds_raw)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_preds)

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot(fpr_rf, tpr_rf, color="#B2182B", lw=2.5,
        label=f"Random Forest (AUC = {auc_rf:.3f})")
ax.plot(fpr_lr, tpr_lr, color="#2166AC", lw=2.5, linestyle="--",
        label=f"Logistic Regression (AUC = {auc_lr:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)

# Mark Youden point on RF curve
ax.scatter([1 - youden_spec], [youden_sens], color="#B2182B", s=100, zorder=5,
           edgecolors="black", linewidths=1, label=f"Youden (t={youden_t:.3f})")

ax.set_xlabel("1 − Specificity (FPR)", fontsize=12)
ax.set_ylabel("Sensitivity (TPR)", fontsize=12)
ax.set_title("ROC Curve Comparison: RF vs LR", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, frameon=True, fancybox=True)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(f"{fig_dir}/F10_ROC_RF_vs_LR.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ F10 saved")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  F11 — CALIBRATION PLOT (Raw vs Isotonic vs Platt-balanced)               ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("F11 — CALIBRATION PLOT")
print("=" * 70)

def calibration_curve_custom(y_true, y_prob, n_bins=10):
    bin_edges = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    mean_pred, obs_rate, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        if i == len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        else:
            mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            obs_rate.append(y_true[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_pred), np.array(obs_rate), np.array(counts)

mp_raw, or_raw, _ = calibration_curve_custom(y_test, test_preds_raw)
mp_iso, or_iso, _ = calibration_curve_custom(y_test, test_preds_final)

fig, ax = plt.subplots(figsize=(7, 6.5))

ax.plot(mp_raw, or_raw, "o-", color="#B2182B", lw=2, markersize=7,
        label=f"Raw RF (Brier={brier_raw:.4f})")
ax.plot(mp_iso, or_iso, "s-", color="#2166AC", lw=2, markersize=7,
        label=f"Isotonic (Brier={brier_cal:.4f})")

# Perfect calibration line
max_val = max(mp_raw.max(), mp_iso.max(), or_raw.max(), or_iso.max()) * 1.1
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.4, lw=1, label="Perfect calibration")

ax.set_xlabel("Mean Predicted Probability", fontsize=12)
ax.set_ylabel("Observed HS Rate", fontsize=12)
ax.set_title("Calibration Curve: Raw RF vs Isotonic Regression", fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(f"{fig_dir}/F11_calibration.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ F11 saved")


# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  FINAL SUMMARY                                                           ║
# ╚════════════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 70)
print("ALL OUTPUTS COMPLETE")
print("=" * 70)
print(f"\nTables saved to:  {tbl_dir}/")
for f in sorted(os.listdir(tbl_dir)):
    if f.startswith("T"):
        print(f"  • {f}")
print(f"\nFigures saved to: {fig_dir}/")
for f in sorted(os.listdir(fig_dir)):
    if f.startswith("F"):
        print(f"  • {f}")

## Section 21 — Sensitivity Analysis: Majority-Class Downsampling

Random 5:1 majority-class downsampling of the training set (minority fully preserved).
Same RF hyperparameters as primary model. Isotonic calibration on downsampled OOF predictions.
Evaluated on the untouched held-out test set.

In [ ]:
# ==============================================================================
# SECTION 21: SENSITIVITY ANALYSIS — MAJORITY-CLASS DOWNSAMPLING
# ==============================================================================
# Downsample majority class in TRAINING set only (5:1 ratio).
# Test set is untouched. Same RF hyperparameters as primary model.
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, brier_score_loss, confusion_matrix

print("=" * 60)
print("SECTION 21 — SENSITIVITY: MAJORITY-CLASS DOWNSAMPLING (5:1)")
print("=" * 60)

# --- A. Reusable downsampling function ---
def downsample_majority(X, y, ratio=5, random_state=42):
    """
    Downsample majority class to ratio:1 vs minority class.
    Returns downsampled X, y arrays.
    Only applied to training data — never to test data.
    """
    np.random.seed(random_state)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg_target = n_pos * ratio

    if n_neg_target >= len(neg_idx):
        print(f"  ⚠️  Requested {n_neg_target} negatives but only {len(neg_idx)} available. Using all.")
        neg_sample = neg_idx
    else:
        neg_sample = np.random.choice(neg_idx, size=n_neg_target, replace=False)

    combined_idx = np.concatenate([pos_idx, neg_sample])
    np.random.shuffle(combined_idx)
    return X[combined_idx], y[combined_idx]

# --- B. Downsample training data ---
X_train_ds, y_train_ds = downsample_majority(X_train_sel, y_train, ratio=5, random_state=42)
print(f"Downsampled training set: {len(y_train_ds)} ({y_train_ds.sum()} pos, "
      f"{(y_train_ds == 0).sum()} neg, ratio = {(y_train_ds == 0).sum() / max(y_train_ds.sum(), 1):.1f}:1)")

# --- C. Fit RF on downsampled data ---
# Keep class_weight='balanced' — still helps at 5:1 imbalance
ds_weights = compute_sample_weights(y_train_ds)

ds_rf = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=5,
    max_depth=None,
    random_state=789,
    n_jobs=-1,
    oob_score=True,
)
ds_rf.fit(X_train_ds, y_train_ds, sample_weight=ds_weights)
print(f"Downsampled RF OOB accuracy: {ds_rf.oob_score_:.4f}")

# --- D. OOF predictions for calibration (within downsampled training data) ---
ds_oof_probs = np.full(len(y_train_ds), np.nan)
skf_ds = StratifiedKFold(n_splits=5, shuffle=True, random_state=800)

for fold_i, (tr_idx, val_idx) in enumerate(skf_ds.split(X_train_ds, y_train_ds), 1):
    X_f_tr = X_train_ds[tr_idx]
    y_f_tr = y_train_ds[tr_idx]
    w_f = compute_sample_weights(y_f_tr)
    rf_f = RandomForestClassifier(
        n_estimators=500, max_features="sqrt", min_samples_leaf=5,
        max_depth=None, random_state=850 + fold_i, n_jobs=-1,
    )
    rf_f.fit(X_f_tr, y_f_tr, sample_weight=w_f)
    ds_oof_probs[val_idx] = rf_f.predict_proba(X_train_ds[val_idx])[:, 1]
    del rf_f

assert not np.isnan(ds_oof_probs).any(), "DS OOF predictions contain NaN"

# --- E. Isotonic calibration on downsampled OOF ---
eps = 1e-8
ds_oof_clamped = np.clip(ds_oof_probs, eps, 1 - eps)
ds_iso = IsotonicRegression(out_of_bounds="clip")
ds_iso.fit(ds_oof_clamped, y_train_ds)  # fit only on training-derived predictions

# --- F. Test predictions ---
ds_test_raw = ds_rf.predict_proba(X_test_sel)[:, 1]
ds_test_iso = ds_iso.predict(np.clip(ds_test_raw, eps, 1 - eps))

# --- G. Evaluate at primary model's Youden threshold ---
def eval_model(y_true, y_prob_raw, y_prob_cal, threshold, label):
    """Compute standard metrics for a model."""
    auroc = roc_auc_score(y_true, y_prob_raw)
    prec, rec, _ = precision_recall_curve(y_true, y_prob_raw)
    auprc = auc(rec, prec)
    brier_r = brier_score_loss(y_true, y_prob_raw)
    brier_c = brier_score_loss(y_true, y_prob_cal)

    y_pred = (y_prob_raw >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "Model": label, "AUROC": auroc, "AUPRC": auprc,
        "Brier_raw": brier_r, "Brier_calibrated": brier_c,
        "Sensitivity": sens, "Specificity": spec,
        "Threshold": threshold
    }

primary_eval = eval_model(y_test, test_preds_raw, test_preds_final, youden_thresh, "Primary (class-weighted)")
ds_eval = eval_model(y_test, ds_test_raw, ds_test_iso, youden_thresh, "Downsampled (5:1)")

ds_comparison = pd.DataFrame([primary_eval, ds_eval])
print("\n" + "=" * 60)
print("DOWNSAMPLING vs PRIMARY MODEL COMPARISON")
print("=" * 60)
print(ds_comparison.to_string(index=False))
ds_comparison.to_csv("sensitivity_downsampling_comparison.csv", index=False)

print("\n✅ Section 21: Downsampling sensitivity analysis complete.")


## Section 24 — Combined Comparison Outputs

Final summary tables exported to a single Excel workbook with separate sheets.

In [ ]:
import pandas as pd
from IPython.display import display
from google.colab import files
import os
import zipfile

tbl_dir = "/content/drive/MyDrive/FAERS Files/Tables"

# 1. Display the key final tables directly in the notebook
print("============================================================")
print("FULL MANUSCRIPT TABLES")
print("============================================================\n")

def load_or_display(var_name, csv_name):
    if var_name in globals():
        display(globals()[var_name])
    else:
        path = os.path.join(tbl_dir, csv_name)
        if os.path.exists(path):
            display(pd.read_csv(path))
        else:
            print(f"{var_name} not found and {csv_name} not found on disk.")

print("T4: TNFi Rates By Agent")
load_or_display('T4', 'T4_TNFi_rates_by_agent.csv')

print("\nT5: TNFi vs Non-TNFi By Indication")
load_or_display('T5', 'T5_TNFi_vs_nonTNFi_by_indication.csv')

print("\nT6: IL-17i vs TNFi By Indication")
load_or_display('T6', 'T6_IL17i_vs_TNFi_by_indication.csv')

print("\nInteraction Adjusted Predicted Probabilities")
if 'pred_grid' in globals():
    display(pred_grid)
elif os.path.exists(f"{tbl_dir}/interaction_adjusted_predicted_probs.csv"):
    display(pd.read_csv(f"{tbl_dir}/interaction_adjusted_predicted_probs.csv"))
else:
    print("pred_grid not found")

print("\nThree-Model Comparison Summary")
if 'three_model_table' in globals():
    display(three_model_table)
else:
    print("three_model_table not found")

# 2. Package all tables into a ZIP file for download
zip_path = "/content/Manuscript_Tables.zip"
print(f"\nPackaging all exported CSV and Excel files into {zip_path}...")

with zipfile.ZipFile(zip_path, 'w') as zipf:
    # Check local content directory
    for f in os.listdir("/content"):
        if f.endswith(('.csv', '.xlsx')):
            zipf.write(os.path.join("/content", f), arcname=f"Local_Files/{f}")

    # Check Drive Tables directory
    if os.path.exists(tbl_dir):
        for f in os.listdir(tbl_dir):
            if f.endswith('.csv'):
                zipf.write(os.path.join(tbl_dir, f), arcname=f"Drive_Tables/{f}")

print("Triggering download...")
files.download(zip_path)

### 1. Calibration Bin Boundaries (All Methods)
This cell extracts the mean predicted vs. observed rates and bin edges for the 10-bin calibration analysis across all four evaluated methods.

In [ ]:
import pandas as pd
import numpy as np

def get_calibration_bins(y_true, y_prob, label, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if i == n_bins - 1:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        if mask.sum() == 0: continue
        rows.append({
            'Method': label, 'Bin_Lower': bin_edges[i], 'Bin_Upper': bin_edges[i+1],
            'N': mask.sum(), 'Mean_Pred': y_prob[mask].mean(), 'Observed_Rate': y_true[mask].mean()
        })
    return pd.DataFrame(rows)

cal_bins_all = pd.concat([
    get_calibration_bins(y_test, test_preds_raw, 'Raw RF'),
    get_calibration_bins(y_test, test_preds_isotonic, 'Isotonic'),
    get_calibration_bins(y_test, test_preds_platt_bal, 'Platt (Balanced)'),
    get_calibration_bins(y_test, test_preds_beta, 'Beta (3-param)')
])
display(cal_bins_all)

### 2. Full Holdout Performance (with 95% CIs)
Extracting the consolidated performance metrics for the Random Forest model on the holdout test set.

In [ ]:
import pandas as pd
import numpy as np

# Extract metrics from restored checkpoint tables
rf_metrics = three_model_table[three_model_table['Model'] == 'Random Forest'].iloc[0]
youden_metrics = threshold_table[threshold_table['Label'] == 'Youden'].iloc[0]

performance_summary = {
    'Metric': ['ROC-AUC', 'PR-AUC', 'Brier Score (Raw)', 'Brier Score (Isotonic)', 'Sensitivity (Youden)', 'Specificity (Youden)', 'PPV (Youden)', 'NPV (Youden)'],
    'Estimate': [
        rf_metrics['ROC-AUC'],
        rf_metrics['PR-AUC'],
        rf_metrics['Brier (raw)'],
        rf_metrics['Brier (isotonic)'],
        youden_metrics['Sensitivity'],
        youden_metrics['Specificity'],
        youden_metrics['PPV'],
        youden_metrics['NPV']
    ]
}
df_perf = pd.DataFrame(performance_summary)
display(df_perf)

In [ ]:
import os
import zipfile
from google.colab import files

print("Saving newly requested tables to CSV...")

# Save the recently displayed tables
cal_bins_all.to_csv('/content/calibration_bin_boundaries.csv', index=False)
df_perf.to_csv('/content/full_holdout_performance.csv', index=False)
feature_stability.to_csv('/content/feature_selection_stability.csv', index=False)

# Explicitly save interaction results again for easy access
int_results.to_csv('/content/interaction_odds_ratios.csv', index=False)
pred_grid.to_csv('/content/interaction_adjusted_probs.csv', index=False)

# Repackage the ZIP
zip_path = "/content/Manuscript_Tables_Updated.zip"
tbl_dir = "/content/drive/MyDrive/FAERS Files/Tables"

print(f"Repackaging all exported files into {zip_path}...")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    # Check local content directory
    for f in os.listdir("/content"):
        if f.endswith(('.csv', '.xlsx')):
            zipf.write(os.path.join("/content", f), arcname=f"Local_Files/{f}")

    # Check Drive Tables directory
    if os.path.exists(tbl_dir):
        for f in os.listdir(tbl_dir):
            if f.endswith('.csv'):
                zipf.write(os.path.join(tbl_dir, f), arcname=f"Drive_Tables/{f}")

print("Triggering updated download...")
files.download(zip_path)


### 3. Feature Selection Stability (10-Fold CV)
Showing the selection frequency across the 10 folds for all 62 candidate features.

In [ ]:
display(feature_stability)
print(f"Total features selected (Stability >= 5): {len(stable_features)}")

### 4. Interaction Analysis Outputs
Extracting the Odds Ratios for interaction terms and the marginally standardized adjusted probabilities.

In [ ]:
print("--- Interaction Odds Ratios (Indication x TNFi) ---")
display(int_results[['term', 'OR', 'OR_CI_lower', 'OR_CI_upper', 'p_raw', 'p_fdr', 'sig']])
print("\n--- Adjusted Predicted Probabilities per 1,000 patients ---")
display(pred_grid[['indication_group', 'tnfi_agent_group', 'adjusted_prob_per_1000', 'ci_lower_per_1000', 'ci_upper_per_1000']])

In [ ]:
# ==============================================================================
# RF vs XGBOOST: ROC + PR-AUC + CALIBRATION COMPARISON (FIXED)
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    brier_score_loss
)

# --- Color palette ---
RF_RAW   = "#B2182B"
RF_ISO   = "#E41A1C"
RF_BETA  = "#FB9A99"
XGB_RAW  = "#2166AC"
XGB_ISO  = "#377EB8"
XGB_BETA = "#A6CEE3"
LR_COL   = "#4DAF4A"


# ==============================================================================
# FIGURE A: ROC + PR-AUC HEAD-TO-HEAD (RF vs XGB vs LR)
# ==============================================================================
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))

for preds, label, color, ls, lw in [
    (test_preds_raw,      "Random Forest", RF_RAW,  "-",  2.5),
    (xgb_test_preds_raw,  "XGBoost",       XGB_RAW, "-",  2.5),
    (lr_preds,            "Logistic Reg",  LR_COL,  "--", 2.0),
]:
    fpr_c, tpr_c, _ = roc_curve(y_test, preds)
    auc_c = roc_auc_score(y_test, preds)
    ax_roc.plot(fpr_c, tpr_c, color=color, lw=lw, linestyle=ls,
                label=f"{label} (AUC = {auc_c:.3f})")

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
ax_roc.set_xlabel("1 − Specificity (FPR)", fontsize=11)
ax_roc.set_ylabel("Sensitivity (TPR)", fontsize=11)
ax_roc.set_title("A. ROC Curves", fontsize=13, fontweight="bold")
ax_roc.legend(loc="lower right", fontsize=10, frameon=True)
ax_roc.set_xlim(-0.01, 1.01)
ax_roc.set_ylim(-0.01, 1.01)
ax_roc.spines["top"].set_visible(False)
ax_roc.spines["right"].set_visible(False)
ax_roc.grid(alpha=0.15)

prevalence = y_test.mean()
for preds, label, color, ls, lw in [
    (test_preds_raw,      "Random Forest", RF_RAW,  "-",  2.5),
    (xgb_test_preds_raw,  "XGBoost",       XGB_RAW, "-",  2.5),
    (lr_preds,            "Logistic Reg",  LR_COL,  "--", 2.0),
]:
    prec_c, rec_c, _ = precision_recall_curve(y_test, preds)
    prauc_c = auc(rec_c, prec_c)
    ax_pr.plot(rec_c, prec_c, color=color, lw=lw, linestyle=ls,
               label=f"{label} (PR-AUC = {prauc_c:.3f})")

ax_pr.axhline(y=prevalence, color="grey", linestyle=":", alpha=0.5,
              label=f"Prevalence = {prevalence:.4f}")
ax_pr.set_xlabel("Recall (Sensitivity)", fontsize=11)
ax_pr.set_ylabel("Precision (PPV)", fontsize=11)
ax_pr.set_title("B. Precision-Recall Curves", fontsize=13, fontweight="bold")
ax_pr.legend(loc="upper right", fontsize=9, frameon=True)
ax_pr.set_xlim(-0.01, 1.01)
ax_pr.spines["top"].set_visible(False)
ax_pr.spines["right"].set_visible(False)
ax_pr.grid(alpha=0.15)

plt.suptitle("Random Forest vs XGBoost vs Logistic Regression — Discrimination",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("rf_vs_xgb_discrimination.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Figure A saved: rf_vs_xgb_discrimination.png")


# ==============================================================================
# HELPER: Adaptive quantile calibration curve
# ==============================================================================
def calibration_curve_quantile(y_true, y_prob, n_bins=20):
    """Quantile-based calibration curve with equal-count bins."""
    bin_edges = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    mean_pred, obs_rate, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        if i == len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        else:
            mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            obs_rate.append(y_true[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_pred), np.array(obs_rate), np.array(counts)


# ==============================================================================
# FIGURE B: CALIBRATION — RF vs XGB, ISOTONIC vs BETA (4-panel, FIXED)
# ==============================================================================
# KEY FIX: Determine axis limits from the DATA, not hardcoded
# All calibrated predictions cluster near 0–0.005, so we zoom there.
# ==============================================================================

# --- Auto-detect zoom range from the actual calibrated predictions ---
all_calibrated = np.concatenate([
    test_preds_isotonic, test_preds_beta,
    xgb_test_preds_iso, xgb_test_preds_beta
])
p99 = np.percentile(all_calibrated, 99.5)
zoom_max = np.ceil(p99 * 1000) / 1000  # round up to nearest 0.001
zoom_max = max(zoom_max, 0.003)        # floor at 0.003 so plot isn't too tiny
print(f"Auto-detected zoom_max: {zoom_max:.4f} (99.5th percentile of calibrated preds: {p99:.5f})")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ── Panel A: RF — Raw vs Isotonic vs Beta (quantile bins, auto-scaled) ───────
ax = axes[0, 0]
for preds, label, color, marker in [
    (test_preds_raw,      "RF Raw",      "#888888", "o"),
    (test_preds_isotonic, "RF Isotonic",  RF_ISO,   "s"),
    (test_preds_beta,     "RF Beta",      RF_BETA,  "D"),
]:
    mp, obs, _ = calibration_curve_quantile(y_test, preds, n_bins=20)
    brier = brier_score_loss(y_test, preds)
    ax.plot(mp, obs, f"{marker}-", color=color, lw=2, markersize=6,
            label=f"{label} (Brier={brier:.5f})")

# Auto-scale: use the max of mean_pred or obs_rate across all series
all_mp_rf = []
for p in [test_preds_raw, test_preds_isotonic, test_preds_beta]:
    mp_t, obs_t, _ = calibration_curve_quantile(y_test, p, n_bins=20)
    all_mp_rf.extend(mp_t)
    all_mp_rf.extend(obs_t)
panel_max = max(all_mp_rf) * 1.15
ax.plot([0, panel_max], [0, panel_max], "k--", alpha=0.3, lw=1, label="Perfect")
ax.set_xlim(0, panel_max)
ax.set_ylim(0, panel_max)
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Observed HS Rate", fontsize=10)
ax.set_title("A. RF — Raw vs Isotonic vs Beta (Quantile Bins)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8.5, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.15)


# ── Panel B: XGB — Raw vs Isotonic vs Beta (quantile bins, auto-scaled) ──────
ax = axes[0, 1]
for preds, label, color, marker in [
    (xgb_test_preds_raw,  "XGB Raw",      "#888888", "o"),
    (xgb_test_preds_iso,  "XGB Isotonic",  XGB_ISO,  "s"),
    (xgb_test_preds_beta, "XGB Beta",      XGB_BETA, "D"),
]:
    mp, obs, _ = calibration_curve_quantile(y_test, preds, n_bins=20)
    brier = brier_score_loss(y_test, preds)
    ax.plot(mp, obs, f"{marker}-", color=color, lw=2, markersize=6,
            label=f"{label} (Brier={brier:.5f})")

all_mp_xgb = []
for p in [xgb_test_preds_raw, xgb_test_preds_iso, xgb_test_preds_beta]:
    mp_t, obs_t, _ = calibration_curve_quantile(y_test, p, n_bins=20)
    all_mp_xgb.extend(mp_t)
    all_mp_xgb.extend(obs_t)
panel_max_xgb = max(all_mp_xgb) * 1.15
ax.plot([0, panel_max_xgb], [0, panel_max_xgb], "k--", alpha=0.3, lw=1, label="Perfect")
ax.set_xlim(0, panel_max_xgb)
ax.set_ylim(0, panel_max_xgb)
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Observed HS Rate", fontsize=10)
ax.set_title("B. XGBoost — Raw vs Isotonic vs Beta (Quantile Bins)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8.5, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.15)


# ── Panel C: ZOOMED — RF vs XGB isotonic + beta (quantile bins) ──────────────
ax = axes[1, 0]
for preds, label, color, marker, lw in [
    (test_preds_isotonic, "RF Isotonic",  RF_ISO,  "s", 2.5),
    (xgb_test_preds_iso,  "XGB Isotonic", XGB_ISO, "s", 2.5),
    (test_preds_beta,     "RF Beta",      RF_BETA, "D", 1.8),
    (xgb_test_preds_beta, "XGB Beta",     XGB_BETA,"D", 1.8),
]:
    mp, obs, cts = calibration_curve_quantile(y_test, preds, n_bins=20)
    brier = brier_score_loss(y_test, preds)
    ax.plot(mp, obs, f"{marker}-", color=color, lw=lw, markersize=7,
            label=f"{label} (Brier={brier:.5f})")

ax.plot([0, zoom_max], [0, zoom_max], "k--", alpha=0.4, lw=1, label="Perfect")
ax.axhline(y=prevalence, color="grey", linestyle=":", alpha=0.5, lw=1)
ax.text(zoom_max * 0.55, prevalence * 1.2, f"Prevalence={prevalence:.4f}",
        fontsize=8, color="grey")
ax.set_xlim(0, zoom_max)
ax.set_ylim(0, zoom_max)
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Observed HS Rate", fontsize=10)
ax.set_title(f"C. Zoomed Head-to-Head (0–{zoom_max:.3f}, Quantile Bins)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.15)


# ── Panel D: ZOOMED — higher resolution (30 quantile bins) ───────────────────
ax = axes[1, 1]
for preds, label, color, marker, lw in [
    (test_preds_isotonic, "RF Isotonic",  RF_ISO,  "s", 2.5),
    (xgb_test_preds_iso,  "XGB Isotonic", XGB_ISO, "s", 2.5),
    (test_preds_beta,     "RF Beta",      RF_BETA, "D", 1.8),
    (xgb_test_preds_beta, "XGB Beta",     XGB_BETA,"D", 1.8),
]:
    mp, obs, cts = calibration_curve_quantile(y_test, preds, n_bins=30)
    n_visible = (mp <= zoom_max).sum()
    ax.plot(mp[mp <= zoom_max], obs[mp <= zoom_max],
            f"{marker}-", color=color, lw=lw, markersize=6,
            label=f"{label} ({n_visible} bins)")

ax.plot([0, zoom_max], [0, zoom_max], "k--", alpha=0.4, lw=1, label="Perfect")
ax.axhline(y=prevalence, color="grey", linestyle=":", alpha=0.5, lw=1)
ax.set_xlim(0, zoom_max)
ax.set_ylim(0, zoom_max)
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Observed HS Rate", fontsize=10)
ax.set_title(f"D. High-Resolution Zoom (30 Quantile Bins, 0–{zoom_max:.3f})",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.15)


plt.suptitle("RF vs XGBoost — Calibration: Isotonic vs Beta",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("rf_vs_xgb_calibration_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Figure B saved: rf_vs_xgb_calibration_comparison.png")


# ==============================================================================
# SUMMARY TABLE
# ==============================================================================
print("\n" + "=" * 70)
print("CALIBRATION METRICS — RF vs XGB (Isotonic vs Beta)")
print("=" * 70)

def cal_metrics_row(y_true, y_prob, label):
    brier = brier_score_loss(y_true, y_prob)
    mean_pred = y_prob.mean()
    mean_obs = y_true.mean()
    po_ratio = mean_pred / mean_obs if mean_obs > 0 else np.nan
    auroc = roc_auc_score(y_true, y_prob)
    # ECE (quantile bins — more appropriate for rare events)
    mp, obs, cts = calibration_curve_quantile(y_true, y_prob, n_bins=10)
    n = len(y_true)
    ece = sum(c / n * abs(m - o) for m, o, c in zip(mp, obs, cts))
    return {
        "Model": label, "Brier": f"{brier:.6f}", "ECE_quantile": f"{ece:.6f}",
        "Mean_Pred": f"{mean_pred:.6f}", "Mean_Obs": f"{mean_obs:.6f}",
        "P/O Ratio": f"{po_ratio:.4f}", "AUC": f"{auroc:.4f}",
    }

rows = [
    cal_metrics_row(y_test, test_preds_raw,       "RF Raw"),
    cal_metrics_row(y_test, test_preds_isotonic,   "RF Isotonic"),
    cal_metrics_row(y_test, test_preds_beta,       "RF Beta"),
    cal_metrics_row(y_test, xgb_test_preds_raw,    "XGB Raw"),
    cal_metrics_row(y_test, xgb_test_preds_iso,    "XGB Isotonic"),
    cal_metrics_row(y_test, xgb_test_preds_beta,   "XGB Beta"),
]

cal_table = pd.DataFrame(rows)
print(cal_table.to_string(index=False))
cal_table.to_csv("rf_vs_xgb_calibration_metrics.csv", index=False)
print("\n✅ Calibration metrics table saved: rf_vs_xgb_calibration_metrics.csv")

In [ ]:
# ==============================================================================
# TOP-RISK ENRICHMENT COMPARISON: RF vs XGBoost vs LR
# ==============================================================================
# Requires: enrich_rf_iso, enrich_xgb, enrich_lr (from Sections 12B, 12C, 15)
#   OR: test_preds_final, xgb_test_preds_iso, lr_preds, y_test
#        + top_risk_enrichment() function
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Recompute if enrichment tables aren't already in memory ---
try:
    _ = enrich_rf_iso.shape
    print("Using existing enrichment tables.")
except NameError:
    print("Recomputing enrichment tables...")
    enrich_rf_iso = top_risk_enrichment(y_test, test_preds_final, [1, 5, 10])
    enrich_rf_iso.insert(0, "Model", "RF_isotonic")
    enrich_xgb = top_risk_enrichment(y_test, xgb_test_preds_iso, [1, 5, 10])
    enrich_xgb.insert(0, "Model", "XGB_isotonic")
    enrich_lr = top_risk_enrichment(y_test, lr_preds, [1, 5, 10])
    enrich_lr.insert(0, "Model", "LR")

# --- Colors ---
RF_COL  = "#B2182B"
XGB_COL = "#2166AC"
LR_COL  = "#4DAF4A"

# --- Combine into one table ---
enrich_all = pd.concat([enrich_rf_iso, enrich_xgb, enrich_lr], ignore_index=True)

# Clean up Top_Pct for plotting (extract numeric)
enrich_all["pct_num"] = enrich_all["Top_Pct"].str.extract(r"(\d+)").astype(int)
pct_levels = sorted(enrich_all["pct_num"].unique())
pct_labels = [f"Top {p}%" for p in pct_levels]

model_config = {
    "RF_isotonic":  {"color": RF_COL,  "label": "Random Forest"},
    "XGB_isotonic": {"color": XGB_COL, "label": "XGBoost"},
    "LR":           {"color": LR_COL,  "label": "Logistic Regression"},
}


# ==============================================================================
# FIGURE: 2-panel — Fold Enrichment + % Cases Captured
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5))

n_groups = len(pct_levels)
n_models = len(model_config)
bar_width = 0.22
x = np.arange(n_groups)

# ── Panel A: Fold Enrichment with 95% CI ─────────────────────────────────────
for i, (model_key, cfg) in enumerate(model_config.items()):
    subset = enrich_all[enrich_all["Model"] == model_key].sort_values("pct_num")
    vals = subset["Fold_enrichment"].values
    ci_lo = subset["Fold_enrich_CI_lo"].values
    ci_hi = subset["Fold_enrich_CI_hi"].values
    err_lo = vals - ci_lo
    err_hi = ci_hi - vals

    bars = ax1.bar(x + i * bar_width, vals, bar_width,
                   color=cfg["color"], edgecolor="white", zorder=3,
                   label=cfg["label"])
    ax1.errorbar(x + i * bar_width, vals,
                 yerr=[err_lo, err_hi],
                 fmt="none", ecolor="#333333", capsize=4, capthick=1.5,
                 elinewidth=1.5, zorder=4)

    # Value labels on bars
    for j, (bar, v) in enumerate(zip(bars, vals)):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + err_hi[j] + 0.3,
                 f"{v:.1f}×", ha="center", va="bottom", fontsize=9, fontweight="bold",
                 color=cfg["color"])

ax1.set_xticks(x + bar_width)
ax1.set_xticklabels(pct_labels, fontsize=12, fontweight="bold")
ax1.set_ylabel("Fold Enrichment over Baseline", fontsize=11, fontweight="bold")
ax1.set_title("A. Fold Enrichment in Top-Risk Groups", fontsize=13, fontweight="bold")
ax1.axhline(y=1, color="grey", linestyle="--", alpha=0.4, lw=1)
ax1.legend(fontsize=10, loc="upper right", frameon=True, fancybox=True)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.grid(axis="y", alpha=0.2)
ax1.set_ylim(0, ax1.get_ylim()[1] * 1.15)


# ── Panel B: % of Total HS Cases Captured ────────────────────────────────────
for i, (model_key, cfg) in enumerate(model_config.items()):
    subset = enrich_all[enrich_all["Model"] == model_key].sort_values("pct_num")
    vals = subset["Pct_positives_captured"].values
    ci_lo = subset["Pct_captured_CI_lo"].values
    ci_hi = subset["Pct_captured_CI_hi"].values
    err_lo = vals - ci_lo
    err_hi = ci_hi - vals

    bars = ax2.bar(x + i * bar_width, vals, bar_width,
                   color=cfg["color"], edgecolor="white", zorder=3,
                   label=cfg["label"])
    ax2.errorbar(x + i * bar_width, vals,
                 yerr=[err_lo, err_hi],
                 fmt="none", ecolor="#333333", capsize=4, capthick=1.5,
                 elinewidth=1.5, zorder=4)

    for j, (bar, v) in enumerate(zip(bars, vals)):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + err_hi[j] + 0.5,
                 f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold",
                 color=cfg["color"])

# Reference lines showing what random selection would capture
for pct in pct_levels:
    ax2.axhline(y=pct, color="grey", linestyle=":", alpha=0.3, lw=1)
    ax2.text(x[-1] + bar_width * n_models + 0.05, pct + 0.3,
             f"Random={pct}%", fontsize=7.5, color="grey", va="bottom")

ax2.set_xticks(x + bar_width)
ax2.set_xticklabels(pct_labels, fontsize=12, fontweight="bold")
ax2.set_ylabel("% of All HS Cases Captured", fontsize=11, fontweight="bold")
ax2.set_title("B. Case Capture in Top-Risk Groups", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10, loc="upper left", frameon=True, fancybox=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(axis="y", alpha=0.2)
ax2.set_ylim(0, min(ax2.get_ylim()[1] * 1.15, 100))

plt.suptitle("Top-Risk Enrichment Analysis — RF vs XGBoost vs Logistic Regression",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("top_risk_enrichment_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


# ==============================================================================
# Print summary table
# ==============================================================================
print("\n" + "=" * 70)
print("TOP-RISK ENRICHMENT SUMMARY")
print("=" * 70)

summary = enrich_all[[
    "Model", "Top_Pct", "N_in_bucket", "N_positives",
    "Fold_enrichment", "Fold_enrich_CI_lo", "Fold_enrich_CI_hi",
    "Pct_positives_captured", "Pct_captured_CI_lo", "Pct_captured_CI_hi"
]].copy()

summary["Fold_enrich_str"] = summary.apply(
    lambda r: f"{r['Fold_enrichment']:.1f}× ({r['Fold_enrich_CI_lo']:.1f}–{r['Fold_enrich_CI_hi']:.1f})",
    axis=1
)
summary["Capture_str"] = summary.apply(
    lambda r: f"{r['Pct_positives_captured']:.1f}% ({r['Pct_captured_CI_lo']:.1f}–{r['Pct_captured_CI_hi']:.1f})",
    axis=1
)

for model_key, cfg in model_config.items():
    print(f"\n  {cfg['label']}:")
    sub = summary[summary["Model"] == model_key]
    for _, r in sub.iterrows():
        print(f"    {r['Top_Pct']:>8}: {r['Fold_enrich_str']:>24}  |  captures {r['Capture_str']}")

print(f"\n✅ Figure saved: top_risk_enrichment_comparison.png")

In [ ]:
# ==============================================================================
# SENSITIVITY ANALYSIS FIGURE: CLASS-WEIGHTED vs DOWNSAMPLED RF (OPTIMIZED)
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc, brier_score_loss
)

# --- Colors ---
PRI_RAW = "#B2182B"
PRI_ISO = "#E41A1C"
DS_RAW  = "#2166AC"
DS_ISO  = "#377EB8"

# --- Compute AUCs ---
auc_pri = roc_auc_score(y_test, test_preds_raw)
auc_ds  = roc_auc_score(y_test, ds_test_raw)

prec_pri, rec_pri, _ = precision_recall_curve(y_test, test_preds_raw)
prec_ds,  rec_ds,  _ = precision_recall_curve(y_test, ds_test_raw)
prauc_pri = auc(rec_pri, prec_pri)
prauc_ds  = auc(rec_ds, prec_ds)

prevalence = y_test.mean()


# ==============================================================================
# FIGURE: 3-panel comparison
# ==============================================================================
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))


# ── Panel A: ROC overlay ─────────────────────────────────────────────────────
fpr_pri, tpr_pri, _ = roc_curve(y_test, test_preds_raw)
fpr_ds,  tpr_ds,  _ = roc_curve(y_test, ds_test_raw)

ax1.plot(fpr_pri, tpr_pri, color=PRI_RAW, lw=2.5,
         label=f"Class-Weighted (AUC={auc_pri:.3f})")
ax1.plot(fpr_ds, tpr_ds, color=DS_RAW, lw=2.5, linestyle="--",
         label=f"Downsampled 5:1 (AUC={auc_ds:.3f})")
ax1.plot([0, 1], [0, 1], "k--", alpha=0.2, lw=1)

ax1.set_xlabel("1 − Specificity (FPR)", fontsize=11)
ax1.set_ylabel("Sensitivity (TPR)", fontsize=11)
ax1.set_title("A. ROC Curves", fontsize=13, fontweight="bold")
ax1.legend(loc="lower right", fontsize=9.5, frameon=True)
ax1.set_xlim(-0.01, 1.01)
ax1.set_ylim(-0.01, 1.01)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.grid(alpha=0.15)

ax1.text(0.55, 0.15,
         f"ΔAUC = {auc_pri - auc_ds:+.4f}\nΔPR-AUC = {prauc_pri - prauc_ds:+.4f}",
         transform=ax1.transAxes, fontsize=9.5, fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9))


# ── Panel B: Calibration — zoomed, quantile bins ─────────────────────────────
def calibration_curve_quantile(y_true, y_prob, n_bins=20):
    bin_edges = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    mean_pred, obs_rate, counts = [], [], []
    for i in range(len(bin_edges) - 1):
        if i == len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])
        else:
            mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        if mask.sum() > 0:
            mean_pred.append(y_prob[mask].mean())
            obs_rate.append(y_true[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_pred), np.array(obs_rate), np.array(counts)

all_cal = np.concatenate([test_preds_final, ds_test_iso])
p995 = np.percentile(all_cal, 99.5)
zoom_max = max(np.ceil(p995 * 1000) / 1000, 0.003)

for preds, label, color, marker, lw in [
    (test_preds_final, "Weighted + Isotonic",     PRI_ISO, "s", 2.5),
    (ds_test_iso,      "Downsampled + Isotonic",  DS_ISO,  "s", 2.5),
    (test_preds_raw,   "Weighted Raw",            PRI_RAW, "o", 1.5),
    (ds_test_raw,      "Downsampled Raw",         DS_RAW,  "o", 1.5),
]:
    mp, obs, _ = calibration_curve_quantile(y_test, preds, n_bins=20)
    brier = brier_score_loss(y_test, preds)
    ax2.plot(mp, obs, f"{marker}-", color=color, lw=lw, markersize=6, alpha=0.85,
             label=f"{label} (Brier={brier:.5f})")

ax2.plot([0, zoom_max], [0, zoom_max], "k--", alpha=0.4, lw=1, label="Perfect")
ax2.axhline(y=prevalence, color="grey", linestyle=":", alpha=0.5, lw=1)
ax2.text(zoom_max * 0.55, prevalence * 1.25, f"Prevalence={prevalence:.4f}",
         fontsize=8, color="grey")
ax2.set_xlim(0, zoom_max)
ax2.set_ylim(0, zoom_max)
ax2.set_xlabel("Mean Predicted Probability", fontsize=11)
ax2.set_ylabel("Observed HS Rate", fontsize=11)
ax2.set_title(f"B. Calibration (Quantile Bins, 0–{zoom_max:.3f})", fontsize=13, fontweight="bold")
ax2.legend(fontsize=8, loc="upper left", frameon=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(alpha=0.15)


# ── Panel C: Top-risk enrichment comparison (OPTIMIZED) ──────────────────────
# Reuse primary model enrichment if already computed; only compute downsampled
try:
    enrich_pri = enrich_rf_iso.copy()
    enrich_pri["Model"] = "Weighted"
    print("  ✅ Reusing enrich_rf_iso for primary model (no recomputation)")
except NameError:
    print("  ⚠️  enrich_rf_iso not found — computing from scratch (500 bootstraps)")
    enrich_pri = top_risk_enrichment(y_test, test_preds_final, [1, 5, 10], n_boot=500)
    enrich_pri.insert(0, "Model", "Weighted")

enrich_ds = top_risk_enrichment(y_test, ds_test_iso, [1, 5, 10], n_boot=500)
enrich_ds.insert(0, "Model", "Downsampled")

enrich_both = pd.concat([enrich_pri, enrich_ds], ignore_index=True)
enrich_both["pct_num"] = enrich_both["Top_Pct"].str.extract(r"(\d+)").astype(int)
pct_levels = sorted(enrich_both["pct_num"].unique())
pct_labels = [f"Top {p}%" for p in pct_levels]

x = np.arange(len(pct_levels))
bar_width = 0.30

for i, (model_key, color, label) in enumerate([
    ("Weighted",    PRI_RAW, "Class-Weighted"),
    ("Downsampled", DS_RAW,  "Downsampled 5:1"),
]):
    subset = enrich_both[enrich_both["Model"] == model_key].sort_values("pct_num")
    vals = subset["Fold_enrichment"].values
    ci_lo = subset["Fold_enrich_CI_lo"].values
    ci_hi = subset["Fold_enrich_CI_hi"].values
    err_lo = vals - ci_lo
    err_hi = ci_hi - vals

    bars = ax3.bar(x + i * bar_width, vals, bar_width,
                   color=color, edgecolor="white", zorder=3, label=label)
    ax3.errorbar(x + i * bar_width, vals,
                 yerr=[err_lo, err_hi],
                 fmt="none", ecolor="#333333", capsize=5, capthick=1.5,
                 elinewidth=1.5, zorder=4)

    for j, (bar, v, cap) in enumerate(zip(bars, vals, subset["Pct_positives_captured"].values)):
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + err_hi[j] + 0.2,
                 f"{v:.1f}×\n({cap:.0f}%)",
                 ha="center", va="bottom", fontsize=8.5, fontweight="bold",
                 color=color)

ax3.axhline(y=1, color="grey", linestyle="--", alpha=0.4, lw=1)
ax3.set_xticks(x + bar_width / 2)
ax3.set_xticklabels(pct_labels, fontsize=12, fontweight="bold")
ax3.set_ylabel("Fold Enrichment", fontsize=11, fontweight="bold")
ax3.set_title("C. Top-Risk Enrichment", fontsize=13, fontweight="bold")
ax3.legend(fontsize=10, loc="upper right", frameon=True)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.grid(axis="y", alpha=0.2)
ax3.set_ylim(0, ax3.get_ylim()[1] * 1.2)

plt.suptitle("Sensitivity Analysis — Class-Weighted vs 5:1 Downsampled Random Forest",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("sensitivity_weighted_vs_downsampled.png", dpi=300, bbox_inches="tight")
plt.show()


# ==============================================================================
# Summary table
# ==============================================================================
print("\n" + "=" * 70)
print("SENSITIVITY ANALYSIS SUMMARY")
print("=" * 70)

print(f"\n{'Metric':<24} {'Class-Weighted':>16} {'Downsampled 5:1':>16} {'Delta':>10}")
print("-" * 68)
print(f"{'ROC-AUC':<24} {auc_pri:>16.4f} {auc_ds:>16.4f} {auc_pri - auc_ds:>+10.4f}")
print(f"{'PR-AUC':<24} {prauc_pri:>16.4f} {prauc_ds:>16.4f} {prauc_pri - prauc_ds:>+10.4f}")

brier_pri_raw = brier_score_loss(y_test, test_preds_raw)
brier_ds_raw  = brier_score_loss(y_test, ds_test_raw)
brier_pri_iso = brier_score_loss(y_test, test_preds_final)
brier_ds_iso  = brier_score_loss(y_test, ds_test_iso)

print(f"{'Brier (raw)':<24} {brier_pri_raw:>16.5f} {brier_ds_raw:>16.5f} {brier_pri_raw - brier_ds_raw:>+10.5f}")
print(f"{'Brier (isotonic)':<24} {brier_pri_iso:>16.5f} {brier_ds_iso:>16.5f} {brier_pri_iso - brier_ds_iso:>+10.5f}")

po_pri = test_preds_final.mean() / prevalence
po_ds  = ds_test_iso.mean() / prevalence
print(f"{'P/O ratio (isotonic)':<24} {po_pri:>16.4f} {po_ds:>16.4f} {po_pri - po_ds:>+10.4f}")

print(f"\nTop-risk enrichment:")
for _, r in enrich_both.iterrows():
    print(f"  {r['Model']:>12} | {r['Top_Pct']:>8}: "
          f"{r['Fold_enrichment']:.1f}× ({r['Fold_enrich_CI_lo']:.1f}–{r['Fold_enrich_CI_hi']:.1f}) "
          f"| captures {r['Pct_positives_captured']:.1f}%")

print(f"\n✅ Figure saved: sensitivity_weighted_vs_downsampled.png")

In [ ]:
# ==============================================================================
# INTERACTION ANALYSIS — COMBINED PUBLICATION FIGURE
# ==============================================================================
# Panel A: Forest plot of interaction ORs (indication × TNFi agent)
# Panel B: Adjusted predicted probability heatmap (marginal standardization)
# Panel C: Grouped bar chart — adjusted rates by indication, colored by TNFi
#
# Requires: int_results, pred_grid, results_df (from Sections 14C–14E)
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec

# ==============================================================================
# DATA PREP
# ==============================================================================

# --- A. Forest plot data ---
plot_df = int_results.copy()

def extract_labels(term):
    parts = term.split(":")
    ind, agent = "Unknown", "Unknown"
    for label in ["Arthropathies", "IBD", "Multiple_Indications", "Psoriasis"]:
        if label in parts[0]:
            ind = label.replace("_", " ")
            break
    for label in ["adalimumab", "certolizumab_pegol", "etanercept", "golimumab", "infliximab"]:
        if label in parts[1]:
            agent = label.replace("_", " ")
            break
    return ind, agent

plot_df[["Indication", "Agent"]] = plot_df["term"].apply(
    lambda t: pd.Series(extract_labels(t))
)

# Sparse cell flags
try:
    sparse_cells_set = set()
    for ind_name in ["Arthropathies", "IBD", "Multiple Indications", "Psoriasis"]:
        for agent_name in ["adalimumab", "certolizumab pegol", "etanercept", "golimumab", "infliximab"]:
            try:
                ct_ind = ind_name.replace(" ", "_") if ind_name == "Multiple Indications" else ind_name
                n_cases = ct_cases.loc[ct_ind, agent_name.replace(" ", "_")]
                if n_cases < 5:
                    sparse_cells_set.add((ind_name, agent_name))
            except (KeyError, NameError):
                pass
    if not sparse_cells_set:
        raise ValueError("empty")
except:
    # Fallback from cell 44 hardcoded set
    sparse_cells_set = {
        ("IBD", "certolizumab pegol"), ("Multiple Indications", "certolizumab pegol"),
        ("Psoriasis", "certolizumab pegol"), ("IBD", "etanercept"),
        ("Multiple Indications", "etanercept"), ("Arthropathies", "golimumab"),
        ("IBD", "golimumab"), ("Multiple Indications", "golimumab"),
        ("Psoriasis", "golimumab"), ("Psoriasis", "infliximab"),
    }

plot_df["sparse"] = plot_df.apply(
    lambda r: (r["Indication"], r["Agent"]) in sparse_cells_set, axis=1
)

# Cap CIs for log-scale display
ci_floor, ci_cap = 0.01, 100
plot_df["OR_lo_display"] = plot_df["OR_CI_lower"].clip(lower=ci_floor)
plot_df["OR_hi_display"] = plot_df["OR_CI_upper"].clip(upper=ci_cap)
plot_df["OR_display"] = plot_df["OR"].clip(lower=ci_floor, upper=ci_cap)
plot_df["fdr_sig"] = plot_df["p_fdr"] < 0.05

# Sort by agent then indication
agent_order = ["adalimumab", "infliximab", "etanercept", "certolizumab pegol", "golimumab"]
indication_order = ["Arthropathies", "IBD", "Psoriasis", "Multiple Indications"]
plot_df["agent_rank"] = plot_df["Agent"].map({a: i for i, a in enumerate(agent_order)})
plot_df["ind_rank"] = plot_df["Indication"].map({a: i for i, a in enumerate(indication_order)})
plot_df = plot_df.sort_values(["agent_rank", "ind_rank"]).reset_index(drop=True)

agent_colors = {
    "adalimumab": "#B2182B", "infliximab": "#D6604D", "etanercept": "#2166AC",
    "certolizumab pegol": "#92C5DE", "golimumab": "#4DAF4A",
}


# --- B. Heatmap data ---
prob_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="adjusted_prob_per_1000"
)
ci_lo_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="ci_lower_per_1000"
)
ci_hi_matrix = pred_grid.pivot_table(
    index="indication_group", columns="tnfi_agent_group",
    values="ci_upper_per_1000"
)

row_order = ["None"] + sorted([r for r in prob_matrix.index if r != "None"])
col_order = ["None"] + sorted([c for c in prob_matrix.columns if c != "None"])
row_order = [r for r in row_order if r in prob_matrix.index]
col_order = [c for c in col_order if c in prob_matrix.columns]

prob_matrix = prob_matrix.reindex(index=row_order, columns=col_order)
ci_lo_matrix = ci_lo_matrix.reindex(index=row_order, columns=col_order)
ci_hi_matrix = ci_hi_matrix.reindex(index=row_order, columns=col_order)


# ==============================================================================
# FIGURE: 3-panel layout
# ==============================================================================
fig = plt.figure(figsize=(22, 14))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.2, 1], width_ratios=[1, 1.1],
                       hspace=0.35, wspace=0.30)

# ── Panel A: Forest plot (left column, full height) ──────────────────────────
ax_forest = fig.add_subplot(gs[:, 0])

y_positions = []
y_labels_list = []
y = 0
current_agent = None
group_boundaries = []

for _, row in plot_df.iterrows():
    if row["Agent"] != current_agent:
        if current_agent is not None:
            group_boundaries.append(y - 0.5)
            y += 1.0
        current_agent = row["Agent"]
    y_positions.append(y)
    y_labels_list.append(row["Indication"])
    y += 1

plot_df["y_pos"] = y_positions

for _, row in plot_df.iterrows():
    color = agent_colors.get(row["Agent"], "grey")
    y_pos = row["y_pos"]
    or_val = row["OR_display"]
    lo, hi = row["OR_lo_display"], row["OR_hi_display"]

    linestyle = "--" if row["sparse"] else "-"
    alpha = 0.35 if row["sparse"] else 0.7
    marker_alpha = 0.3 if row["sparse"] else 1.0

    ax_forest.plot([lo, hi], [y_pos, y_pos],
                   color=color, linewidth=2, linestyle=linestyle, alpha=alpha)

    if row["OR_CI_upper"] > ci_cap:
        ax_forest.annotate("", xy=(ci_cap, y_pos), xytext=(ci_cap * 0.7, y_pos),
                           arrowprops=dict(arrowstyle="->", color=color, alpha=alpha, lw=1.5))
    if row["OR_CI_lower"] < ci_floor:
        ax_forest.annotate("", xy=(ci_floor, y_pos), xytext=(ci_floor * 1.5, y_pos),
                           arrowprops=dict(arrowstyle="->", color=color, alpha=alpha, lw=1.5))

    marker = "D" if row["fdr_sig"] else "o"
    edgecolor = "black" if row["fdr_sig"] else color
    facecolor = color if not row["sparse"] else "white"
    ms = 10 if row["fdr_sig"] else 7

    ax_forest.scatter(or_val, y_pos, s=ms**2, color=facecolor,
                      edgecolors=edgecolor, linewidths=1.5 if row["fdr_sig"] else 0.8,
                      zorder=5, alpha=marker_alpha, marker=marker)

    if row["fdr_sig"]:
        ax_forest.text(hi * 1.15, y_pos, f'OR={row["OR"]:.2f}**',
                       va="center", fontsize=8, fontweight="bold", color=color)
    elif row["p_raw"] < 0.05 and not row["sparse"]:
        ax_forest.text(hi * 1.15, y_pos, f'OR={row["OR"]:.2f}*',
                       va="center", fontsize=8, color=color)

# Agent group labels
agent_label_positions = plot_df.groupby("Agent")["y_pos"].mean().to_dict()
for agent, y_mid in agent_label_positions.items():
    ax_forest.text(ci_floor * 0.55, y_mid, agent.title(),
                   va="center", ha="right", fontsize=10, fontweight="bold",
                   color=agent_colors.get(agent, "grey"))

for boundary in group_boundaries:
    ax_forest.axhline(y=boundary, color="grey", linewidth=0.5, linestyle=":", alpha=0.4)

ax_forest.axvline(x=1, color="black", linewidth=1, linestyle="-", alpha=0.4)
ax_forest.set_xscale("log")
ax_forest.set_xlim(ci_floor * 0.4, ci_cap * 2.5)
ax_forest.set_yticks(y_positions)
ax_forest.set_yticklabels(y_labels_list, fontsize=9)
ax_forest.invert_yaxis()
ax_forest.set_xlabel("Odds Ratio (95% CI, log scale)", fontsize=11)
ax_forest.set_title("A. Interaction ORs: Indication × TNFi Agent",
                     fontsize=13, fontweight="bold")
ax_forest.set_xticks([0.01, 0.1, 0.25, 0.5, 1, 2, 4, 10, 100])
ax_forest.set_xticklabels(["0.01", "0.1", "0.25", "0.5", "1", "2", "4", "10", "100"])
ax_forest.spines["top"].set_visible(False)
ax_forest.spines["right"].set_visible(False)
ax_forest.grid(axis="x", alpha=0.15)

legend_elements = [
    mpatches.Patch(facecolor="grey", edgecolor="grey", alpha=0.7, label="Solid = ≥5 HS cases"),
    mpatches.Patch(facecolor="white", edgecolor="grey", alpha=0.5, label="Hollow/dashed = <5 (sparse)"),
    plt.Line2D([0], [0], marker="D", color="black", linestyle="None",
               markersize=8, label="FDR q < 0.05"),
    plt.Line2D([0], [0], marker="o", color="grey", linestyle="None",
               markersize=6, markerfacecolor="white", label="Not significant"),
]
ax_forest.legend(handles=legend_elements, loc="lower right", fontsize=8,
                 framealpha=0.9, edgecolor="grey")


# ── Panel B: Heatmap (top right) ────────────────────────────────────────────
ax_heat = fig.add_subplot(gs[0, 1])

cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom", ["#FFFFFF", "#FEE08B", "#FC8D59", "#D73027"], N=256
)
vmax = max(prob_matrix.values.max(), 0.01)

im = ax_heat.imshow(prob_matrix.values, aspect="auto", cmap=cmap, vmin=0, vmax=vmax)

for i in range(prob_matrix.shape[0]):
    for j in range(prob_matrix.shape[1]):
        val = prob_matrix.values[i, j]
        lo = ci_lo_matrix.values[i, j]
        hi = ci_hi_matrix.values[i, j]
        text_color = "white" if val > vmax * 0.65 else "black"
        ax_heat.text(j, i, f"{val:.2f}\n({lo:.2f}–{hi:.2f})",
                     ha="center", va="center", fontsize=7.5, color=text_color,
                     fontweight="bold")

ax_heat.set_xticks(range(len(col_order)))
ax_heat.set_xticklabels([c.replace("_", " ").title() for c in col_order],
                         fontsize=8.5, fontweight="bold", rotation=30, ha="left")
ax_heat.xaxis.set_ticks_position("top")
ax_heat.xaxis.set_label_position("top")
ax_heat.set_yticks(range(len(row_order)))
ax_heat.set_yticklabels([r.replace("_", " ") for r in row_order],
                         fontsize=9, fontweight="bold")
ax_heat.set_title("B. Adjusted HS Rate per 1,000\n(Marginally Standardized)",
                   fontsize=13, fontweight="bold", pad=40)

cbar = fig.colorbar(im, ax=ax_heat, shrink=0.8, pad=0.02)
cbar.set_label("Rate per 1,000", fontsize=9)


# ── Panel C: Grouped bar chart — rates by indication, stratified by TNFi ────
ax_bar = fig.add_subplot(gs[1, 1])

# Filter to the key TNFi agents (exclude "None" and "Multiple_TNFi")
key_agents = ["adalimumab", "infliximab", "etanercept", "certolizumab_pegol", "golimumab"]
key_indications = ["Arthropathies", "IBD", "Psoriasis"]

bar_data = pred_grid[
    pred_grid["tnfi_agent_group"].isin(key_agents) &
    pred_grid["indication_group"].isin(key_indications)
].copy()

# Also get the "None" TNFi baseline for each indication
baseline_data = pred_grid[
    (pred_grid["tnfi_agent_group"] == "None") &
    pred_grid["indication_group"].isin(key_indications)
].copy()

n_agents = len(key_agents)
n_indications = len(key_indications)
x = np.arange(n_indications)
bar_width = 0.13

agent_display = {
    "adalimumab": "ADA", "infliximab": "IFX", "etanercept": "ETN",
    "certolizumab_pegol": "CZP", "golimumab": "GOL",
}
agent_colors_bar = {
    "adalimumab": "#B2182B", "infliximab": "#D6604D", "etanercept": "#2166AC",
    "certolizumab_pegol": "#92C5DE", "golimumab": "#4DAF4A",
}

for i, agent in enumerate(key_agents):
    subset = bar_data[bar_data["tnfi_agent_group"] == agent].copy()
    subset = subset.set_index("indication_group").reindex(key_indications)
    vals = subset["adjusted_prob_per_1000"].values
    ci_lo = subset["ci_lower_per_1000"].values
    ci_hi = subset["ci_upper_per_1000"].values
    err_lo = vals - ci_lo
    err_hi = ci_hi - vals

    bars = ax_bar.bar(x + i * bar_width, vals, bar_width,
                      color=agent_colors_bar[agent], edgecolor="white", zorder=3,
                      label=f"{agent_display[agent]} ({agent.replace('_', ' ')})")
    ax_bar.errorbar(x + i * bar_width, vals,
                    yerr=[err_lo, err_hi],
                    fmt="none", ecolor="#555555", capsize=3, capthick=1, elinewidth=1, zorder=4)

# Baseline reference markers
for j, ind in enumerate(key_indications):
    baseline_row = baseline_data[baseline_data["indication_group"] == ind]
    if len(baseline_row) > 0:
        bval = baseline_row["adjusted_prob_per_1000"].values[0]
        ax_bar.axhline(y=bval, xmin=(j / n_indications) + 0.02,
                       xmax=((j + 1) / n_indications) - 0.02,
                       color="grey", linestyle="--", alpha=0.5, lw=1.2)
        ax_bar.text(x[j] + bar_width * n_agents / 2, bval + 0.02,
                    f"No TNFi: {bval:.2f}", ha="center", fontsize=7.5,
                    color="grey", fontstyle="italic")

ax_bar.set_xticks(x + bar_width * (n_agents - 1) / 2)
ax_bar.set_xticklabels(key_indications, fontsize=11, fontweight="bold")
ax_bar.set_ylabel("Adjusted HS Rate per 1,000", fontsize=11, fontweight="bold")
ax_bar.set_title("C. Adjusted HS Rates by Indication × TNFi Agent",
                  fontsize=13, fontweight="bold")
ax_bar.legend(fontsize=8, loc="upper left", ncol=2, frameon=True)
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.grid(axis="y", alpha=0.2)
ax_bar.set_ylim(0, ax_bar.get_ylim()[1] * 1.15)

plt.savefig("interaction_analysis_combined.png", dpi=300, bbox_inches="tight")
plt.savefig("interaction_analysis_combined.pdf", bbox_inches="tight")
plt.show()

print("✅ Combined interaction figure saved: interaction_analysis_combined.png/.pdf")


# ==============================================================================
# Print summary for manuscript text
# ==============================================================================
print("\n" + "=" * 70)
print("INTERACTION ANALYSIS SUMMARY")
print("=" * 70)

sig_terms = int_results[int_results["p_fdr"] < 0.05]
if len(sig_terms) > 0:
    print(f"\nFDR-significant interaction terms ({len(sig_terms)}):")
    for _, r in sig_terms.iterrows():
        ind, agent = extract_labels(r["term"])
        print(f"  {ind} × {agent}: OR={r['OR']:.2f} "
              f"({r['OR_CI_lower']:.2f}–{r['OR_CI_upper']:.2f}), "
              f"p_raw={r['p_raw']:.2e}, q={r['p_fdr']:.3f}")
else:
    print("\n  No FDR-significant interaction terms.")

nom_sig = int_results[(int_results["p_raw"] < 0.05) & (int_results["p_fdr"] >= 0.05)]
if len(nom_sig) > 0:
    print(f"\nNominally significant (p<0.05, FDR≥0.05) ({len(nom_sig)}):")
    for _, r in nom_sig.iterrows():
        ind, agent = extract_labels(r["term"])
        sparse_flag = " [SPARSE]" if (ind, agent) in sparse_cells_set else ""
        print(f"  {ind} × {agent}: OR={r['OR']:.2f} "
              f"({r['OR_CI_lower']:.2f}–{r['OR_CI_upper']:.2f}), "
              f"p={r['p_raw']:.3f}, q={r['p_fdr']:.3f}{sparse_flag}")

# Highlight the IBD × adalimumab cell (expected strongest signal)
print("\nKey cell — IBD × adalimumab:")
ibd_ada = pred_grid[
    (pred_grid["indication_group"] == "IBD") &
    (pred_grid["tnfi_agent_group"] == "adalimumab")
]
if len(ibd_ada) > 0:
    r = ibd_ada.iloc[0]
    print(f"  Adjusted rate: {r['adjusted_prob_per_1000']:.2f} per 1,000 "
          f"(95% CI: {r['ci_lower_per_1000']:.2f}–{r['ci_upper_per_1000']:.2f})")

# Compare to baseline (None × None)
none_none = pred_grid[
    (pred_grid["indication_group"] == "None") &
    (pred_grid["tnfi_agent_group"] == "None")
]
if len(none_none) > 0:
    r0 = none_none.iloc[0]
    print(f"  Baseline (None × None): {r0['adjusted_prob_per_1000']:.2f} per 1,000")
    if len(ibd_ada) > 0:
        ratio = ibd_ada.iloc[0]["adjusted_prob_per_1000"] / r0["adjusted_prob_per_1000"]
        print(f"  Ratio: {ratio:.1f}×")

## Section 19B — Manuscript Main Figures (revised, data-driven)

Publication-ready composites for the 5-item main set (Table 1 + Figures 1-4). Each figure is a single composite with bold **A/B/C/D** corner labels, JAMA-style conventions (ratios on log scales with point+CI markers; bars only for frequency/rate data), and is exported at 300 dpi **and** as vector PDF. All panels are wired to computed objects — no hardcoded numbers.

Run after Sections 3, 12/12C, 13, 14, 15, 15B-15G, and 16 so the referenced objects exist:
`stats_combined`, `drug_classes`, `T4`, `interaction_table`, `plot_df`, `model_df_full`, `fisher_or_ci`,
`model_score_map`, `enrich_by_model`, `top_risk_enrichment`, `test_preds_raw`, `test_preds_isotonic`,
`shap_vals`, `X_shap`, `stable_features`, `ebm`, `lr_coefs`, `y_test`.

In [ ]:
# ==============================================================================
# Shared figure style
# ==============================================================================
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
mpl.rcParams.update({"figure.dpi":110,"savefig.dpi":300,"font.size":10,
                     "axes.spines.top":False,"axes.spines.right":False,
                     "axes.titlesize":11,"axes.titleweight":"bold"})
C_RISK, C_PROT, C_CLASS, C_NEU = "#B2182B", "#2166AC", "#2166AC", "#4D4D4D"

def panel(ax, letter, title="", loc="left"):
    ax.set_title(f"{letter}  {title}".rstrip(), loc=loc, fontweight="bold")

def save_fig(fig, stem):
    fig.savefig(stem+".png", dpi=300, bbox_inches="tight")
    fig.savefig(stem+".pdf", bbox_inches="tight")   # vector for submission
    print("saved", stem+".png /", stem+".pdf")


In [ ]:
# ==============================================================================
# FIGURE 1 — Drug-associated HS signal landscape (drug-induced stratum)
# Forest of reporting odds ratios by agent, grouped by pharmacologic class.
# Black = individual agents; blue diamond = inverse-variance class-pooled estimate.
# Filled marker = significant after Benjamini-Hochberg FDR (p_fdr_bh < .05).
# ==============================================================================
f1 = stats_combined[(stats_combined["history_group"] == "No HS Indication") &
                    (stats_combined["sex_group"] == "Total")].copy()
f1["drug_l"] = f1["drug_name"].str.lower().str.strip()
drug2class = {d.lower(): c for c, ds in drug_classes.items() for d in ds}
f1["class"] = f1["drug_l"].map(drug2class)
f1 = f1[f1["class"].notna() & (f1["a"] >= 10)].copy()   # >=10 drug-induced HS reports

def _pool(g):
    w = 1.0 / (g["ror_se"] ** 2); lr = np.log(g["ror"])
    m = np.sum(w * lr) / np.sum(w); se = np.sqrt(1.0 / np.sum(w))
    return pd.Series({"ror": np.exp(m), "lo": np.exp(m - 1.96*se),
                      "hi": np.exp(m + 1.96*se), "n": int(g["a"].sum())})
pooled = f1.groupby("class").apply(_pool).reset_index().sort_values("ror")

# build row layout: class header (pooled) then its agents
rows = []
for cls in pooled["class"]:
    pr = pooled[pooled["class"] == cls].iloc[0]
    rows.append(("class", cls, pr["ror"], pr["lo"], pr["hi"], True, pr["n"]))
    ag = f1[f1["class"] == cls].sort_values("ror")
    for _, r in ag.iterrows():
        rows.append(("agent", r["drug_name"], r["ror"], r["ror_ci_lower"],
                     r["ror_ci_upper"], bool(r["p_fdr_bh"] < 0.05), int(r["a"])))
rows = rows[::-1]                       # highest ROR at top
y = np.arange(len(rows))

fig, ax = plt.subplots(figsize=(8, max(5, 0.34*len(rows))))
for yi, (kind, label, ror, lo, hi, sig, n) in zip(y, rows):
    col = C_CLASS if kind == "class" else "black"
    mk  = "D" if kind == "class" else "o"
    ax.plot([lo, hi], [yi, yi], color=col, lw=1.4, zorder=1)
    ax.scatter(ror, yi, marker=mk, s=70 if kind=="class" else 34, color=col,
               facecolor=(col if sig else "white"), edgecolor=col, zorder=2)
ax.axvline(1, color="gray", ls="--", lw=1)
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels([("  "+l if k=="agent" else l.upper()) for k,l,*_ in rows],
                   fontsize=8)
ax.set_xlabel("Reporting odds ratio (95% CI), log scale")
panel(ax, "Figure 1.", "Drug-induced HS reporting signals by agent and class")
ax.text(0.99, -0.06, "Filled = significant after BH-FDR; blue ◆ = class-pooled",
        transform=ax.transAxes, ha="right", fontsize=7, color="gray")
save_fig(fig, "Figure1_ROR_landscape"); plt.show()


In [ ]:
# ==============================================================================
# FIGURE 2 — Indication-aware TNFi risk  (A-D)
# A rate by agent (bar; rates=frequency)  B rate by indication x agent (heatmap)
# C TNFi vs non-TNFi within indication (forest of ORs)  D interaction ORs (forest)
# ==============================================================================
fig = plt.figure(figsize=(13, 10))
gs = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.32)
axA, axB, axC, axD = [fig.add_subplot(gs[i, j]) for i in (0,1) for j in (0,1)]

# --- A: HS rate per 1000 by TNFi agent (bar OK: rate/frequency) ---
t4 = T4.sort_values("Rate_per_1000", ascending=True)
axA.barh(t4["TNFi"], t4["Rate_per_1000"], color=C_RISK, edgecolor="white")
for yi,(rate,orv) in enumerate(zip(t4["Rate_per_1000"], t4["OR_vs_others"])):
    axA.text(rate, yi, f"  OR {orv:.2f}", va="center", fontsize=8)
axA.set_xlabel("HS reporting rate per 1000 users"); panel(axA,"A","TNFi agent-specific HS rate")

# --- B: rate by indication x agent (heatmap; frequency) ---
piv = interaction_table.pivot_table(index="Indication", columns="TNFi",
                                    values="HS_rate_per_1000")
im = axB.imshow(piv.values, cmap="Reds", aspect="auto")
axB.set_xticks(range(len(piv.columns))); axB.set_xticklabels(piv.columns, rotation=45, ha="right", fontsize=8)
axB.set_yticks(range(len(piv.index)));   axB.set_yticklabels(piv.index, fontsize=8)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v=piv.values[i,j]
        if np.isfinite(v): axB.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7)
fig.colorbar(im, ax=axB, fraction=0.046, pad=0.04, label="HS rate/1000")
panel(axB,"B","HS rate by indication × TNFi agent")

# --- C: TNFi vs non-TNFi within indication (computed from model_df_full) ---
tnfi_cols = [c for c in model_df_full.columns if c.startswith("TNFi_")]
tnfi_any = (model_df_full[tnfi_cols] == 1).any(axis=1).values
yv = (model_df_full["outcome_hs"] == "Yes").astype(int).values
indics = [("Hist_Arthropathies","Arthropathies"),("Hist_IBD","IBD"),("Hist_Psoriasis","Psoriasis")]
crows=[]
for col,name in indics:
    if col not in model_df_full.columns: continue
    m = model_df_full[col].values == 1
    a=int(((m)&(tnfi_any)&(yv==1)).sum()); b=int(((m)&(tnfi_any)&(yv==0)).sum())
    c=int(((m)&(~tnfi_any)&(yv==1)).sum());dd=int(((m)&(~tnfi_any)&(yv==0)).sum())
    orv,lo,hi,p = fisher_or_ci(a,b,c,dd)
    crows.append((name,orv,lo,hi,p))
cy=np.arange(len(crows))
for yi,(name,orv,lo,hi,p) in zip(cy,crows):
    axC.plot([lo,hi],[yi,yi],color="black",lw=1.4)
    axC.scatter(orv,yi,s=40,color=(C_RISK if p<0.05 else C_NEU),zorder=2)
    axC.text(hi,yi,f"  OR {orv:.2f}",va="center",fontsize=8)
axC.axvline(1,color="gray",ls="--",lw=1); axC.set_xscale("log")
axC.set_yticks(cy); axC.set_yticklabels([r[0] for r in crows])
axC.set_xlabel("OR for HS, TNFi vs non-TNFi (95% CI), log scale")
panel(axC,"C","TNFi vs non-TNFi by indication")

# --- D: interaction ORs (forest from plot_df) ---
pdd = plot_df.copy()
pdd["cell"]=pdd["Indication"]+" × "+pdd["Agent"]
pdd=pdd.sort_values("OR_display")
dy=np.arange(len(pdd))
for yi,(_,r) in zip(dy,pdd.iterrows()):
    axD.plot([r["OR_lo_display"],r["OR_hi_display"]],[yi,yi],
             color=("silver" if r["sparse"] else "black"),lw=1.2)
    axD.scatter(r["OR_display"],yi,s=36,zorder=2,
                color=(C_PROT if r["fdr_sig"] else ("silver" if r["sparse"] else C_NEU)),
                facecolor=(C_PROT if r["fdr_sig"] else "white"),
                edgecolor=("silver" if r["sparse"] else "black"))
axD.axvline(1,color="gray",ls="--",lw=1); axD.set_xscale("log")
axD.set_yticks(dy); axD.set_yticklabels(pdd["cell"],fontsize=7)
axD.set_xlabel("Interaction OR (95% CI), log scale")
panel(axD,"D","Indication × agent interaction (FDR-flagged; grey=sparse)")

save_fig(fig,"Figure2_TNFi_indication"); plt.show()


In [ ]:
# =====================================================================
# FIGURE 3 (standalone, self-contained) — A precision-recall | B ROC | C PPV | D capture
# Panel B is now ROC (calibration moved to eFigure 8). Builds its own predictions +
# enrichment from the model objects, so it does NOT need Section 15F.
# Needs in memory: final_rf, xgb_model, ebm, lr, enet_cv, X_test_sel, y_test.
# =====================================================================
import numpy as np, matplotlib.pyplot as plt, matplotlib as mpl
from sklearn.metrics import precision_recall_curve, auc, roc_curve, roc_auc_score
mpl.rcParams.update({"savefig.dpi": 300, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
def _save(fig, stem):
    fig.savefig(stem + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(stem + ".pdf", bbox_inches="tight"); print("saved", stem + ".png / .pdf")

y = np.asarray(y_test)

# --- raw test-set scores for each supervised model (predict_proba on the test set) ---
_MODELS = {"Random Forest": final_rf, "XGBoost": xgb_model, "EBM": ebm,
           "Logistic (L2)": lr, "Elastic Net": enet_cv}
scores = {name: mdl.predict_proba(X_test_sel)[:, 1]
          for name, mdl in _MODELS.items() if mdl is not None}
prev = y.mean()

# --- top-risk enrichment (PPV and case capture at top 1/5/10%) ---
def enrich(p, pcts=(1, 5, 10)):
    n = len(y); order = np.argsort(p)[::-1]; total = y.sum(); out = {}
    for pct in pcts:
        k = max(1, int(np.ceil(n * pct / 100))); top = order[:k]
        out[pct] = {"ppv": y[top].mean() * 100, "capture": y[top].sum() / total * 100}
    return out
enr = {name: enrich(p) for name, p in scores.items()}

fig = plt.figure(figsize=(15, 11)); gs = fig.add_gridspec(2, 2, hspace=0.34, wspace=0.28)
axA, axB, axC, axD = [fig.add_subplot(gs[i, j]) for i in (0, 1) for j in (0, 1)]

# A  precision-recall
for name, p in scores.items():
    pr, rc, _ = precision_recall_curve(y, p)
    axA.plot(rc, pr, lw=1.8, label=f"{name} ({auc(rc, pr):.3f})")
axA.axhline(prev, ls="--", c="gray", lw=1, label=f"Prevalence ({prev:.4f})")
axA.set_xlabel("Recall"); axA.set_ylabel("Precision"); axA.legend(fontsize=8, loc="upper right")
axA.set_title("A  Precision-recall (PR-AUC in legend)", loc="left", fontweight="bold")

# B  ROC
for name, p in scores.items():
    fpr, tpr, _ = roc_curve(y, p)
    axB.plot(fpr, tpr, lw=1.8, label=f"{name} ({roc_auc_score(y, p):.3f})")
axB.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
axB.set_xlabel("1 − Specificity (false-positive rate)"); axB.set_ylabel("Sensitivity (true-positive rate)")
axB.legend(fontsize=8, loc="lower right")
axB.set_title("B  Receiver operating characteristic (AUROC in legend)", loc="left", fontweight="bold")

# C  PPV at top-k ; D  case capture at top-k
names = list(scores.keys()); pcts = [1, 5, 10]; w = 0.8 / len(names)
for ax, key, fmt, ttl, ylab in [
    (axC, "ppv", "{:.1f}", "C  Precision (PPV) at top-risk %", "% of flagged reports that are HS"),
    (axD, "capture", "{:.0f}", "D  Case capture at top-risk %", "% of HS cases captured")]:
    for j, n in enumerate(names):
        vals = [enr[n][p][key] for p in pcts]; xpos = np.arange(len(pcts)) + j * w
        ax.bar(xpos, vals, w, label=n)
        for x, v in zip(xpos, vals):
            ax.text(x, v + max(vals) * 0.02, fmt.format(v), ha="center", va="bottom", fontsize=7)
    ax.set_xticks(np.arange(len(pcts)) + w * (len(names) - 1) / 2)
    ax.set_xticklabels([f"Top {p}%" for p in pcts]); ax.set_ylabel(ylab)
    ax.set_title(ttl, loc="left", fontweight="bold")
axD.legend(fontsize=8, ncol=2, loc="upper right")

_save(fig, "Figure3_framework_performance"); plt.show()

In [ ]:
# =====================================================================
# FIGURE 4 (full, 3 rows)
#   Row 1  A  SHAP beeswarm (biological features)
#   Row 2  B  EBM age effect | C  EBM local (true positive) | D  EBM local (false negative)
#   Row 3  E  Cross-model feature-importance concordance (jittered dot plot)
# Needs in memory: shap_vals, X_shap, stable_features, ebm, X_test_sel, y_test,
#                  imp_df, and missing_flags (run the concordance cell first).
# =====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl, re
import shap as _shp
mpl.rcParams.update({"savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
RISK, PROT = "#C0392B", "#1F6FB2"
NONBIO = {"Age_missing", "Sex_Unknown"}

feats = list(stable_features)
keep = [i for i, f in enumerate(feats) if f not in NONBIO]      # biological columns for SHAP
feat_bio = [feats[i] for i in keep]

fig = plt.figure(figsize=(15, 20))
gs = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.05, 1.25], hspace=0.32)

# ---------------- Row 1: SHAP beeswarm ----------------
axA = fig.add_subplot(gs[0])
try:
    plt.sca(axA)
    _shp.summary_plot(shap_vals[:, keep], X_shap[:, keep], feature_names=feat_bio,
                      max_display=15, show=False, plot_size=None, color_bar=True)
    axA.set_title("A  SHAP value distribution (random forest; each point a test report, colored by feature value)",
                  loc="left", fontweight="bold")
except Exception as ex:
    axA.text(0.5, 0.5, f"SHAP beeswarm unavailable\n{ex}", ha="center", transform=axA.transAxes)

# ---------------- Row 2: three EBM plots ----------------
gs2 = gs[1].subgridspec(1, 3, wspace=0.38, width_ratios=[1.3, 1, 1])
# B: EBM age shape + density
gsB = gs2[0, 0].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
axSc = fig.add_subplot(gsB[0, 0]); axDe = fig.add_subplot(gsB[1, 0], sharex=axSc)
try:
    k = feats.index("age") if "age" in feats else 0
    dat = ebm.explain_global().data(k)
    e = np.asarray(dat["names"], float); sc = np.asarray(dat["scores"], float)
    xs = e[:-1] if len(e) == len(sc) + 1 else e[:len(sc)]
    axSc.step(xs, sc, where="post", color=RISK, lw=2); axSc.axhline(0, color="gray", ls="--", lw=1)
    axSc.set_ylabel("EBM contribution (log-odds)")
    axSc.set_title(f"B  EBM feature effect: {feats[k]}", loc="left", fontweight="bold", fontsize=11)
    plt.setp(axSc.get_xticklabels(), visible=False)
    dens = dat.get("density")
    if dens:
        dn = np.asarray(dens["names"], float); dv = np.asarray(dens["scores"], float)
        if len(dn) == len(dv) + 1: axDe.bar(dn[:-1], dv, width=np.diff(dn), align="edge", color="#F4A582")
        else: axDe.bar(range(len(dv)), dv, color="#F4A582")
    axDe.set_yticks([]); axDe.set_xlabel(feats[k]); axDe.set_ylabel("n", fontsize=8)
except Exception as ex:
    axSc.text(0.5, 0.5, f"EBM shape unavailable\n{ex}", ha="center", transform=axSc.transAxes)
# C, D: EBM local explanations for a true-positive and a false-negative report
try:
    ep = ebm.predict_proba(X_test_sel)[:, 1]; pos = np.where(y_test == 1)[0]
    tp = pos[np.argmax(ep[pos])]; fn = pos[np.argmin(ep[pos])]
    loc = ebm.explain_local(X_test_sel[[tp, fn]], y_test[[tp, fn]])
    for c, (idx, lab, letter) in enumerate([(0, f"True positive (pred={ep[tp]:.3f})", "C"),
                                            (1, f"False negative (pred={ep[fn]:.3f})", "D")]):
        axL = fig.add_subplot(gs2[0, c + 1]); d = loc.data(idx)
        nm = [stable_features[int(re.search(r"\d+", str(s)).group())] if str(s).startswith("feature_") else s
              for s in d["names"]]
        sc2 = np.asarray(d["scores"], float); o = np.argsort(np.abs(sc2))[::-1][:12]
        nm = [nm[i] for i in o]; sc2 = sc2[o]; yy = np.arange(len(sc2))[::-1]
        axL.barh(yy, sc2, color=[RISK if s > 0 else PROT for s in sc2])
        axL.set_yticks(yy); axL.set_yticklabels(nm, fontsize=8); axL.axvline(0, color="k", lw=0.8)
        axL.set_xlabel("Contribution to log-odds")
        axL.set_title(f"{letter}  EBM local: {lab}", loc="left", fontweight="bold", fontsize=10)
except Exception as ex:
    ax = fig.add_subplot(gs2[0, 1]); ax.text(0.5, 0.5, f"EBM local unavailable\n{ex}", ha="center", transform=ax.transAxes)

# ---------------- Row 3: concordance dot plot ----------------
axE = fig.add_subplot(gs[2])
bio = imp_df.drop(index=[f for f in imp_df.index if f in NONBIO])
norm = bio / bio.max(axis=0).replace(0, np.nan)
order = norm.mean(axis=1).sort_values(ascending=False).head(12).index[::-1]
models = list(imp_df.columns)
cmap = dict(zip(models, plt.cm.tab10(np.linspace(0, 1, 10))))
off = dict(zip(models, np.linspace(-0.26, 0.26, len(models))))
for yi, f in enumerate(order):
    xs = norm.loc[f]
    axE.plot([xs.min(), xs.max()], [yi, yi], color="#d9d9d9", lw=1, zorder=1)
    for m in models:
        axE.scatter(norm.loc[f, m], yi + off[m], s=55, color=cmap[m], edgecolor="k", lw=0.4,
                    alpha=0.9, zorder=3, label=(m if yi == len(order) - 1 else None))
axE.set_yticks(range(len(order))); axE.set_yticklabels(order, fontsize=10); axE.set_ylim(-0.7, len(order) - 0.3)
axE.set_xlabel("Permutation importance (ROC-AUC decrease, normalized per model)")
axE.set_title("E  Feature-importance concordance across models", loc="left", fontweight="bold", fontsize=12)
axE.legend(fontsize=8, loc="lower right", frameon=True)

fig.savefig("Figure4_full.png", dpi=300, bbox_inches="tight")
fig.savefig("Figure4_full.pdf", bbox_inches="tight"); plt.show()
print("saved Figure4_full.png / .pdf")

## Section 19C — Supplemental Figures (data-driven)

Regenerates the supplemental figures wired to live objects (no hardcoded numbers). Each is exported at 300 dpi + vector PDF.

**Data sources (for provenance):**
- eFig 2 reports by year/age: `demo_std` (age, event_dt) restricted to `ids_case_true`; DW = `ids_case_true & ids_history_true`, DI = the rest.
- eFig 3 top indications: `drug` (PS) × `indications_compositeid` → `indic_table`, computed for DW and DI cohorts.
- eFig 5 mirrored case-count: `stats_combined` (sex="Total"), `a` = exposed HS cases, colored by log2(ROR).
- eFig 7 sex volcano: `stats_combined` (Female/Male).
- eFig 8 ROC / eFig 9 calibration: `model_score_map` and per-model calibrated preds.
- eFig 10 feature stability: `feature_stability`.  eFig 11: `ebm_imp`, `iso_scores`.
- **eFig 4 (comorbidity heatmaps) and eFig 6 (time-to-onset) are already produced data-driven by the existing cells** (comorbidity heatmap cell; TTO-by-class cell) — not duplicated here.

eFigure 1 (study-design flow) is a schematic — build it separately (SVG).

In [ ]:
# eFig setup (self-contained styling + helpers)
import matplotlib.pyplot as plt, matplotlib as mpl, matplotlib.colors as mcolors, numpy as np, pandas as pd
mpl.rcParams.update({"savefig.dpi":300,"font.size":10,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.titlesize":11,"axes.titleweight":"bold"})
C_RISK,C_PROT,C_NEU = "#B2182B","#2166AC","#4D4D4D"
def panel(ax,letter,title="",loc="left"): ax.set_title(f"{letter}  {title}".rstrip(),loc=loc,fontweight="bold")
def save_fig(fig,stem):
    fig.savefig(stem+".png",dpi=300,bbox_inches="tight"); fig.savefig(stem+".pdf",bbox_inches="tight")
    print("saved",stem+".png /",stem+".pdf")
def reliability(y,p,nb=10):
    q=np.unique(np.quantile(p,np.linspace(0,1,nb+1))); idx=np.clip(np.digitize(p,q[1:-1]),0,len(q)-2)
    xs,ys=[],[]
    for b in range(len(q)-1):
        m=idx==b
        if m.sum(): xs.append(p[m].mean()); ys.append(y[m].mean())
    return np.array(xs),np.array(ys)
print("eFig helpers ready.")


### eFigure 1 — Study design and analytic cohorts

In [ ]:
# =====================================================================
# eFigure 1 (v2) — same figure, re-rendered from the ORIGINAL source so
# every glyph is byte-identical & sharp. Only the geometry changes:
#   * top FAERS box untouched
#   * all other boxes shifted DOWN ~50 px
#   * the three left lower boxes shifted an EXTRA ~40 px (branch breathing room)
#   * clean orthogonal branch + vertical arrows, tips ending ~8-10 px above boxes
#   * extra white canvas at bottom
# Axis 0-1 maps to an 8.47-in axes at 300 dpi -> 1 axis unit ~= 2541 px,
# so 50 px ~= 0.0197, 40 px ~= 0.0157, 10 px ~= 0.0039.
# =====================================================================
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({"font.family": "DejaVu Sans"})
fig, ax = plt.subplots(figsize=(12, 11))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
LW = 1.1
GAP = 0.0039          # ~10 px clear gap between every arrowhead tip and box top
SHIFT = 0.0197        # ~50 px general downward shift
EXTRA = 0.0157        # ~40 px additional shift for the 3 left lower boxes

def box(cx, cy, text, w=0.24, bold=False, fs=9):
    nl = text.count("\n") + 1
    h = 0.026 * nl + 0.028
    ax.add_patch(FancyBboxPatch((cx - w/2, cy - h/2), w, h,
        boxstyle="round,pad=0.004,rounding_size=0.010",
        linewidth=LW, edgecolor="black", facecolor="white"))
    ax.text(cx, cy, text, ha="center", va="center", fontsize=fs,
            fontweight=("bold" if bold else "normal"))
    return h

def arrow(x1, y1, x2, y2):
    # straight (vertical) connector; tip lands exactly at (x2,y2)
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="black", lw=LW,
                                shrinkA=0, shrinkB=0))
def vline(x, y1, y2): ax.plot([x, x], [y1, y2], color="black", lw=LW)
def hline(y, x1, x2): ax.plot([x1, x2], [y, y], color="black", lw=LW)

# ---- top box (UNCHANGED) ----
hF = box(0.5, 0.955, "FDA Adverse Event Reporting System\n(FAERS), 2004–2023", w=0.44, fs=10)

# ---- split from FAERS into two independent branches (labels + junction fixed) ----
LX, RX = 0.25, 0.74
ax.text(LX, 0.893, "Disproportionality cohort", ha="center", fontsize=9.5, fontweight="bold")
ax.text(RX, 0.893, "Machine-learning cohort", ha="center", fontsize=9.5, fontweight="bold")
yj = 0.872
vline(0.5, 0.955 - hF/2 - GAP, yj)
hline(yj, LX, RX)

# ================= LEFT branch (disproportionality) =================
cyL1 = 0.815 - SHIFT
h = box(LX, cyL1, "Reports screened for a MedDRA preferred\nterm containing “hidradenitis”", w=0.32, fs=8.5)
arrow(LX, yj, LX, cyL1 + h/2 + GAP)
top = cyL1 - h/2

cyL2 = 0.700 - SHIFT
h = box(LX, cyL2, "5529 hidradenitis suppurativa (HS) reports", w=0.32, bold=True, fs=9)
arrow(LX, top - GAP, LX, cyL2 + h/2 + GAP)
bot5529 = cyL2 - h/2

# two-way split with extra breathing room
La, Lb = 0.13, 0.37
yj2 = bot5529 - 0.028                       # trunk before the split (starts below box)
vline(LX, bot5529 - GAP, yj2); hline(yj2, La, Lb)

cySub = 0.545 - SHIFT - EXTRA
hA = box(La, cySub, "Drug-worsened\n(HS listed as indication)\nn = 3725", w=0.21, fs=8)
hB = box(Lb, cySub, "Drug-induced\n(no HS indication)\nn = 1804", w=0.21, fs=8)
arrow(La, yj2, La, cySub + hA/2 + GAP)
arrow(Lb, yj2, Lb, cySub + hB/2 + GAP)

subBot = cySub - hA/2
yj3 = subBot - 0.034
vline(La, subBot - GAP, yj3); vline(Lb, subBot - GAP, yj3); hline(yj3, La, Lb)
cyL4 = 0.360 - SHIFT - EXTRA
h = box(LX, cyL4, "Disproportionality (reporting odds ratios,\nBH-FDR), comorbidity, indication, and\ntime-to-onset analyses", w=0.36, fs=8.5)
arrow(LX, yj3, LX, cyL4 + h/2 + GAP)

# ================= RIGHT branch (machine learning) =================
cyR1 = 0.808 - SHIFT
h = box(RX, cyR1, "All TNF-inhibitor–exposed reports\n(after excluding HS as a reported indication):\nn = 1,451,950", w=0.42, fs=8.5)
arrow(RX, yj, RX, cyR1 + h/2 + GAP); bot = cyR1 - h/2

cyR2 = 0.688 - SHIFT
h = box(RX, cyR2, "HS cases (positive class): n = 953 (0.07%)", w=0.42, fs=8.5)
arrow(RX, bot - GAP, RX, cyR2 + h/2 + GAP); bot = cyR2 - h/2

cyR3 = 0.575 - SHIFT
h = box(RX, cyR3, "70/30 stratified split: training n = 1,016,365;\ntest n = 435,585 (286 test cases)", w=0.42, fs=8.5)
arrow(RX, bot - GAP, RX, cyR3 + h/2 + GAP); bot = cyR3 - h/2

cyR4 = 0.478 - SHIFT
h = box(RX, cyR4, "Feature selection: 29 of 62 candidates (10-fold CV)", w=0.42, fs=8.5)
arrow(RX, bot - GAP, RX, cyR4 + h/2 + GAP); bot = cyR4 - h/2

cyR5 = 0.350 - SHIFT
h = box(RX, cyR5, "Model comparison: elastic net, ridge logistic\nregression, random forest, XGBoost, explainable\nboosting machine, and isolation forest (anomaly detection)", w=0.46, fs=8.5)
arrow(RX, bot - GAP, RX, cyR5 + h/2 + GAP); bot = cyR5 - h/2

cyR6 = 0.200 - SHIFT
h = box(RX, cyR6, "Calibrated rare-event risk ranking; PR-AUC,\nenrichment, and calibration; SHAP/EBM interpretation;\nindication × TNFi-agent interaction", w=0.46, fs=8.5)
arrow(RX, bot - GAP, RX, cyR6 + h/2 + GAP)

# extra white canvas (pad_inches ~0.45 in ~= 135 px) so nothing is clipped
plt.savefig("eFigure1_study_design.png", dpi=300, bbox_inches="tight", pad_inches=0.45,
            facecolor="white")
plt.savefig("eFigure1_study_design.pdf", bbox_inches="tight", pad_inches=0.45,
            facecolor="white")
plt.savefig("eFigure1_study_design.svg", bbox_inches="tight", pad_inches=0.45,
            facecolor="white")
print("saved eFigure1_study_design.png / .pdf / .svg")


In [ ]:
# eFigure 2 — HS reports by year and age (drug-worsened vs drug-induced)
dw_ids = set(ids_case_true) & set(ids_history_true)
dsc = demo_std[demo_std["compositeid"].isin(ids_case_true)].copy()
dsc["stratum"] = np.where(dsc["compositeid"].isin(dw_ids), "Drug-worsened", "Drug-induced")
dsc["year"] = pd.to_numeric(dsc["event_dt"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
dsc["age_num"] = pd.to_numeric(dsc["age"], errors="coerce")

fig,(axA,axB)=plt.subplots(1,2,figsize=(13,5))
yr=dsc.dropna(subset=["year"]); yr=yr[(yr.year>=2004)&(yr.year<=2023)]
pt=yr.groupby(["year","stratum"]).size().unstack(fill_value=0)
for s,c in [("Drug-worsened",C_PROT),("Drug-induced",C_RISK)]:
    if s in pt.columns: axA.plot(pt.index,pt[s],"o-",color=c,label=s,ms=4)
axA.set_xlabel("Report year"); axA.set_ylabel("No. of HS reports"); axA.legend()
panel(axA,"A","HS reports by year")
ag=dsc.dropna(subset=["age_num"]); ag=ag[(ag.age_num>=0)&(ag.age_num<=100)]
for s,c in [("Drug-worsened",C_PROT),("Drug-induced",C_RISK)]:
    axB.hist(ag[ag.stratum==s]["age_num"],bins=20,alpha=0.5,color=c,label=s)
axB.set_xlabel("Age at report (years)"); axB.set_ylabel("No. of reports"); axB.legend()
panel(axB,"B","Age distribution at report")
save_fig(fig,"eFigure2_year_age"); plt.show()


In [ ]:
# eFigure 3 — Top indications for primary suspect drug (A drug-worsened, B drug-induced)
# detect the indication-name column in indic_table
_namecol=None
for c in indic_table.columns:
    if c!="indication_concept_id" and indic_table[c].dtype==object: _namecol=c; break
print("Indication name column detected:", _namecol)

def top_indications(id_set,n=15):
    ps=drug.loc[drug["compositeid"].isin(id_set)&drug["role_cod"].eq("PS"),
                ["compositeid","drug_seq","drug_name"]].dropna(subset=["compositeid","drug_seq"])
    mi=ps.merge(indications_compositeid,left_on=["compositeid","drug_seq"],
                right_on=["compositeid","indi_drug_seq"],how="inner")[["compositeid","indication_concept_id"]].dropna()
    cnt=mi.groupby("indication_concept_id").size().reset_index(name="N")
    cnt=cnt.merge(indic_table,on="indication_concept_id",how="left")
    cnt["name"]=(cnt[_namecol].astype(str) if _namecol else cnt["indication_concept_id"].astype(str))
    cnt["pct"]=cnt["N"]/max(len(id_set),1)*100
    return cnt.sort_values("N",ascending=False).head(n)

dw=top_indications(set(ids_case_true)&set(ids_history_true))
di=top_indications(set(ids_case_true)-set(ids_history_true))
fig,(axA,axB)=plt.subplots(1,2,figsize=(15,6))
for ax,dfi,lab,L in [(axA,dw,"Drug-worsened","A"),(axB,di,"Drug-induced","B")]:
    dfi=dfi.iloc[::-1]
    ax.barh(dfi["name"],dfi["N"],color=C_RISK,edgecolor="white")
    for yi,(nn,pc) in enumerate(zip(dfi["N"],dfi["pct"])): ax.text(nn,yi,f" {pc:.1f}%",va="center",fontsize=7)
    ax.set_xlabel("No. of reports"); panel(ax,L,lab)
save_fig(fig,"eFigure3_top_indications"); plt.show()


### eFigure 5 — Mirrored HS case-count (in-memory, left-axis fixed)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.colors as mcolors
mpl_ok=True
# eFigure 5 — Mirrored HS case-count (top 50 by drug-induced count) [counts beside bars, bigger names]
# ---------------------------------------------------------------------
sc=stats_combined[stats_combined["sex_group"]=="Total"].copy()
di_s=sc[sc.history_group=="No HS Indication"].set_index("drug_name")
dw_s=sc[sc.history_group=="Has HS Indication"].set_index("drug_name")
top=di_s["a"].sort_values(ascending=False).head(50).index.tolist()
norm=mcolors.TwoSlopeNorm(vmin=-3,vcenter=0,vmax=3); cmap=plt.cm.RdBu_r
def bcol(sub,dr):
    if dr in sub.index and np.isfinite(sub.loc[dr,"ror"]) and sub.loc[dr,"p_value"]<0.05:
        return cmap(norm(np.log2(sub.loc[dr,"ror"])))
    return "lightgrey"
order=top[::-1]
fig,ax=plt.subplots(figsize=(11,max(10,0.34*len(order))))
for yi,dr in enumerate(order):
    dc=int(di_s.loc[dr,"a"]) if dr in di_s.index else 0
    wc=int(dw_s.loc[dr,"a"]) if dr in dw_s.index else 0
    ax.barh(yi,-dc,color=bcol(di_s,dr)); ax.barh(yi,wc,color=bcol(dw_s,dr))
    if dc>0: ax.text(-dc,yi,f"{dc} ",va="center",ha="right",fontsize=8.5)
    if wc>0: ax.text(wc,yi,f" {wc}",va="center",ha="left",fontsize=8.5)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order,fontsize=10)
ax.axvline(0,color="black",lw=1); ax.set_xscale("symlog")
# extra room on the left so the longest bar's count label clears the drug names
ax.set_xlim(-di_s["a"].max()*4, dw_s["a"].max()*3)
ax.tick_params(axis="y", pad=6)
ax.set_xlabel("← Drug-induced      HS exposed case count (symlog)      Drug-worsened →",fontsize=12)
ax.set_title("eFigure 5.  Mirrored HS case-count by indication history (top 50 drugs)",loc="left",fontweight="bold",fontsize=13)
sm=plt.cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
fig.colorbar(sm,ax=ax,fraction=0.04,pad=0.02,label="log2(ROR); grey = nonsignificant")
plt.tight_layout()
fig.savefig("eFigure5_mirrored_case_count.png",dpi=300,bbox_inches="tight")
fig.savefig("eFigure5_mirrored_case_count.pdf",bbox_inches="tight"); plt.show()

In [ ]:
# eFigure 7 — Sex-stratified volcano (risk signals, ROR>1), faceted by HS history
v=stats_combined[stats_combined["ror"]>1].copy()
v["log2ror"]=np.log2(v["ror"]); v["neglog10p"]=-np.log10(v["p_value"].clip(lower=1e-300))
v["x"]=np.where(v["sex_group"]=="Female",-v["log2ror"],v["log2ror"])
groups=[g for g in v["history_group"].unique() if g in ("No HS Indication","Has HS Indication")]
fig,axes=plt.subplots(1,len(groups),figsize=(15,7),squeeze=False)
for j,g in enumerate(groups):
    ax=axes[0][j]; gd=v[(v.history_group==g)&(v.sex_group.isin(["Female","Male"]))].copy()
    for sex,c in [("Female","#E41A1C"),("Male","#377EB8")]:
        s=gd[gd.sex_group==sex]; ax.scatter(s.x,s.neglog10p,c=c,alpha=0.5,s=28,label=sex)
    gd["score"]=gd.neglog10p*gd.log2ror.abs()
    for _,r in gd.sort_values("score",ascending=False).groupby("sex_group").head(10).iterrows():
        ax.annotate(str(r["drug_name"]),(r.x,r.neglog10p),xytext=(4,4),textcoords="offset points",fontsize=6)
    ax.axvline(0,color="black",lw=1); ax.set_xlabel("← Female     log2(ROR)     Male →")
    ax.set_ylabel("-log10(P)"); ax.set_title(g,fontweight="bold"); ax.legend(fontsize=7)
axes[0][0].text(0.0,1.04,"eFigure 7.  Sex-stratified volcano (ROR>1)",transform=axes[0][0].transAxes,fontweight="bold")
save_fig(fig,"eFigure7_sex_volcano"); plt.show()


### eFigure 9 — Calibration, all models

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

# same colors as Figure 3
MODEL_COLORS = {"Random Forest": "#1f77b4", "XGBoost": "#ff7f0e", "EBM": "#2ca02c",
                "Ridge (L2)": "#d62728", "Elastic Net": "#9467bd"}
MODELS = {"Random Forest": final_rf, "XGBoost": xgb_model, "EBM": ebm,
          "Ridge (L2)": lr, "Elastic Net": enet_cv}

y = np.asarray(y_test)
cal_idx, ev_idx = train_test_split(np.arange(len(y)), test_size=0.5, stratify=y, random_state=0)

def reliability(yy, p, nb=10):
    q = np.unique(np.quantile(p, np.linspace(0, 1, nb + 1)))
    if len(q) < 3: return np.array([]), np.array([])
    idx = np.clip(np.digitize(p, q[1:-1]), 0, len(q) - 2)
    xs, ys = [], []
    for b in range(len(q) - 1):
        m = idx == b
        if m.sum(): xs.append(p[m].mean()); ys.append(yy[m].mean())
    return np.array(xs), np.array(ys)

plt.rcParams.update({"savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(7, 6)); zmax = 0
for name, mdl in MODELS.items():
    if mdl is None: continue
    raw = mdl.predict_proba(X_test_sel)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip").fit(raw[cal_idx], y[cal_idx])   # calibrate on half
    cal = iso.transform(raw[ev_idx])                                               # evaluate on the other half
    xs, ys = reliability(y[ev_idx], cal, nb=10)
    if len(xs):
        ax.plot(xs, ys, "o-", ms=4, lw=1.6, color=MODEL_COLORS.get(name), label=name)
        zmax = max(zmax, xs.max(), ys.max())
zmax = min(zmax, 0.006) if zmax > 0 else 0.006
ax.plot([0, zmax], [0, zmax], "k--", lw=1, alpha=0.5)
ax.set_xlim(0, zmax); ax.set_ylim(0, zmax)
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed HS rate")
ax.set_title("eFigure 9.  Calibration of isotonic-scaled probabilities across models",
             loc="left", fontweight="bold", fontsize=12)
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
fig.savefig("eFigure9_calibration_all.png", dpi=300, bbox_inches="tight")
fig.savefig("eFigure9_calibration_all.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Feature-selection stability (10-fold CV) — reported as eTable 12 in the manuscript (not an eFigure)
fs=feature_stability.sort_values("folds_selected").tail(25)
fig,ax=plt.subplots(figsize=(8,max(6,0.3*len(fs))))
cols=[C_RISK if v>=5 else C_NEU for v in fs["folds_selected"]]
ax.barh(fs["feature"],fs["folds_selected"],color=cols,edgecolor="white")
ax.axvline(5,color="gray",ls="--",lw=1,label="Stability threshold (5 of 10)")
ax.set_xlabel("Folds selected (of 10)"); ax.legend(loc="lower right")
panel(ax,"","Feature-selection stability (folds selected per feature; manuscript eTable 12)")
save_fig(fig,"feature_selection_stability"); plt.show()


In [ ]:
# eFigure 8 — EBM global importances (A) and Isolation Forest anomaly scores (B)
fig,(axA,axB)=plt.subplots(1,2,figsize=(13,5.5))
ei=ebm_imp.sort_values("importance").tail(15)
axA.barh(ei["feature"],ei["importance"],color="#762A83",edgecolor="white")
axA.set_xlabel("EBM global term importance"); panel(axA,"A","EBM feature importances")
axB.hist(iso_scores[y_test==0],bins=40,alpha=0.5,color=C_NEU,density=True,label="Controls")
axB.hist(iso_scores[y_test==1],bins=40,alpha=0.6,color=C_RISK,density=True,label="HS cases")
axB.set_xlabel("Isolation Forest anomaly score"); axB.set_ylabel("Density"); axB.legend()
panel(axB,"B","Anomaly score by outcome")
save_fig(fig,"eFigure8_ebm_isoforest"); plt.show()


In [ ]:
# =====================================================================
# SUPPLEMENT — 4 CONCORDANCE HEATMAPS
# A Spearman (all features)   B Spearman (top-10 features)
# C Top-10 overlap (of 10)    D Top-5 overlap (of 5)
# Run after cell_crossmodel_concordance.py (uses imp_df).
# =====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

models = list(imp_df.columns)

def spearman_mat(df):
    M = pd.DataFrame(index=models, columns=models, dtype=float)
    for a in models:
        for b in models:
            M.loc[a, b] = spearmanr(df[a], df[b]).correlation
    return M

def overlap_mat(df, K):
    top = {n: set(df[n].sort_values(ascending=False).head(K).index) for n in models}
    M = pd.DataFrame(index=models, columns=models, dtype=float)
    for a in models:
        for b in models:
            M.loc[a, b] = len(top[a] & top[b])
    return M

S_all  = spearman_mat(imp_df)
union10 = sorted(set().union(*[set(imp_df[n].sort_values(ascending=False).head(10).index) for n in models]))
S_top  = spearman_mat(imp_df.loc[union10])
O10    = overlap_mat(imp_df, 10)
O5     = overlap_mat(imp_df, 5)

panels = [("A  Spearman ρ — all features",   S_all, "Blues",  0, 1,  "{:.2f}"),
          ("B  Spearman ρ — top-10 features", S_top, "Blues",  0, 1,  "{:.2f}"),
          ("C  Top-10 overlap (of 10)",                O10,   "Greens", 0, 10, "{:.0f}"),
          ("D  Top-5 overlap (of 5)",                  O5,    "Greens", 0, 5,  "{:.0f}")]

fig, axes = plt.subplots(2, 2, figsize=(13, 12))
for ax, (title, M, cmap, vmin, vmax, fmt) in zip(axes.ravel(), panels):
    V = M.values.astype(float)
    im = ax.imshow(V, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(models))); ax.set_xticklabels(models, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(models))); ax.set_yticklabels(models, fontsize=8)
    for i in range(len(models)):
        for j in range(len(models)):
            ax.text(j, i, fmt.format(V[i, j]), ha="center", va="center", fontsize=8,
                    color=("white" if V[i, j] > vmax * 0.6 else "black"))
    ax.set_title(title, loc="left", fontweight="bold", fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("eFigure 10. Cross-model feature-importance concordance", fontweight="bold", fontsize=13, y=1.00)
plt.tight_layout()
fig.savefig("eFigure10_concordance_heatmaps.png", dpi=300, bbox_inches="tight")
fig.savefig("eFigure10_concordance_heatmaps.pdf", bbox_inches="tight"); plt.show()
print("saved eFigure10_concordance_heatmaps.png / .pdf")

In [ ]:
# =====================================================================
# FINAL EXPORT — run LAST, after all main + supplemental figure cells.
# Collects every saved figure, relabels by manuscript number + title,
# writes vector PDF (submission quality) + PNG into one folder, copies to
# Drive, zips it, and prints a manifest (flags anything missing).
# =====================================================================
import os, re, shutil

OUT   = "Manuscript_Figures_Final"
DRIVE = "/content/drive/MyDrive/FAERS Files/Manuscript_Figures_Final"
os.makedirs(OUT, exist_ok=True)
try: os.makedirs(DRIVE, exist_ok=True); DRIVE_OK = True
except Exception: DRIVE_OK = False

# (file stem written by the figure cell, manuscript label, manuscript title)
FIGS = [
    # --- Main text ---
    ("Figure1_ROR_landscape",        "Figure 1",  "Drug-induced HS reporting signals by agent and class"),
    ("Figure2_TNFi_indication",      "Figure 2",  "Indication-aware TNFi-associated HS risk"),
    ("Figure3_framework_performance","Figure 3",  "Rare-event detection framework: model performance"),
    ("Figure4_full",                 "Figure 4",  "Model interpretation across models"),
    # --- Supplement ---
    ("eFigure1_study_design",        "eFigure 1", "Study design and analytic cohorts"),
    ("eFigure2_year_age",            "eFigure 2", "HS reports by year and age, by indication history"),
    ("eFigure3_top_indications",     "eFigure 3", "Top indications for the primary suspect drug"),
    ("heatmap_sidebyside_comparison","eFigure 4", "Comorbidity-profile heatmaps"),
    ("eFigure5_mirrored_case_count", "eFigure 5", "Mirrored HS case-count signal plot"),
    ("tto_by_drug_class",            "eFigure 6", "Time-to-onset by drug class"),
    ("eFigure7_sex_volcano",         "eFigure 7", "Sex-stratified volcano plot"),
    ("eFigure8_ebm_isoforest",        "eFigure 8", "EBM importances and Isolation Forest scores"),
    ("eFigure9_calibration_all",      "eFigure 9", "Calibration, all models"),
    ("eFigure10_concordance_heatmaps","eFigure 10","Cross-model feature-importance concordance"),
    # feature-selection stability is manuscript eTable 12, not an eFigure
]

def _clean(t): return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")

rows = []
for stem, label, title in FIGS:
    base = f"{label.replace(' ', '_')}_-_{_clean(title)}"
    got = []
    for ext in ("pdf", "svg", "tif", "tiff", "png"):   # PDF/vector first (best quality)
        src = f"{stem}.{ext}"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(OUT, f"{base}.{ext}"))
            if DRIVE_OK: shutil.copy2(src, os.path.join(DRIVE, f"{base}.{ext}"))
            got.append(ext)
    rows.append((label, ",".join(got) if got else "*** MISSING ***", title))

print(f"{'Label':11}{'Formats':16}Title")
print("-"*90)
for label, fmts, title in rows:
    print(f"{label:11}{fmts:16}{title}")

zip_path = shutil.make_archive(OUT, "zip", OUT)
print("\nSaved to:", os.path.abspath(OUT))
if DRIVE_OK: print("Drive copy:", DRIVE)
print("Zip:", zip_path)
print("\nNote: PDFs are vector (resolution-independent) = highest quality for submission; PNGs are 300 dpi for review.")
print("Missing rows just mean that figure cell wasn't run yet. eFigure 1 (flow diagram) is produced by its own cell above.")


In [ ]:
import sys
import importlib.metadata

packages = ['scikit-learn', 'xgboost', 'interpret', 'statsmodels', 'shap']

print(f"Python version: {sys.version.split()[0]}")
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg} version: NOT FOUND")

In [ ]:
import os, shutil
SRC  = "/content"
DEST = "/content/drive/MyDrive/FAERS Files/notebook_outputs"
EXCLUDE = {"/content/drive", "/content/FAERS_Files", "/content/sample_data", "/content/.config"}
KEEP_EXT = {".csv",".xlsx",".xls",".png",".pdf",".svg",".tif",".tiff",".jpg",".jpeg",
            ".json",".txt",".html",".docx",".pptx",".pkl",".parquet"}

os.makedirs(DEST, exist_ok=True)
n = tot = 0
for dp, dns, fns in os.walk(SRC):
    dns[:] = [d for d in dns if os.path.join(dp, d) not in EXCLUDE]   # prune before descending
    for fn in fns:
        if os.path.splitext(fn)[1].lower() not in KEEP_EXT:
            continue
        src = os.path.join(dp, fn)
        try:
            if os.path.getsize(src) > 300e6:      # skip anything >300 MB (not an output)
                continue
            rel = os.path.relpath(dp, SRC)
            outdir = DEST if rel == "." else os.path.join(DEST, rel)
            os.makedirs(outdir, exist_ok=True)
            shutil.copy2(src, outdir)
            n += 1; tot += os.path.getsize(src)
        except Exception as e:
            print("skip", src, e)
print(f"Copied {n} files ({tot/1e6:.1f} MB) -> {DEST}")


## Checkpoint Save (run once, after a full clean run)
Both the dataframes and the fitted-model objects are written here so future sessions can skip the pipeline.

In [ ]:
###############################################################################
# CELL 1: COMPREHENSIVE CHECKPOINT SAVE (replaces your current Section 25)
# Run this ONCE after all analyses complete successfully.
###############################################################################

# ==============================================================================
# COMPREHENSIVE CHECKPOINT SAVE — Run after all analyses are complete
# ==============================================================================
import pickle, os, glob
import numpy as np
import pandas as pd

save_dir = "/content/drive/MyDrive/FAERS Files/Checkpoints"
os.makedirs(save_dir, exist_ok=True)

# ==============================================================================
# A. DATAFRAMES — all intermediate tables needed for figures/tables/exports
# ==============================================================================
print("=" * 60)
print("SAVING DATAFRAMES")
print("=" * 60)

df_names = [
    # Core ML dataframes
    "demo_std", "comor_matrix", "model_df_full", "model_df_model",
    # Feature importance
    "perm_importance", "shap_importance", "feature_stability",
    # ROR statistics (needed for volcano plots)
    "stats_combined",
    # Interaction regression
    "int_df", "results_df", "int_results", "pred_grid",
    # TTO analysis
    "tto_data", "tto_summary", "tnfi_tto", "drug_tto_summary",
    # Enrichment tables
    "enrich_rf_iso", "enrich_rf_raw", "enrich_xgb", "enrich_lr",
    "enrich_combined", "enrich_combined_rf", "enrich_combined_all",
    "enrich_all_models",
    # Model comparison tables
    "three_model_table", "three_model_comparison_partial",
    "ds_comparison", "cal_comparison",
    # LR coefficients
    "lr_coefs", "df_forest",
    # XGBoost SHAP
    "xgb_shap_importance", "xgb_cal_df",
    # Calibration comparison
    "threshold_table",
]

df_checkpoint = {}
for name in df_names:
    obj = globals().get(name)
    if obj is not None and hasattr(obj, 'shape'):
        df_checkpoint[name] = obj
        print(f"  ✅ {name}: {obj.shape}")
    elif obj is not None and isinstance(obj, pd.DataFrame):
        df_checkpoint[name] = obj
        print(f"  ✅ {name}: {obj.shape}")

with open(f"{save_dir}/checkpoint_dataframes_v2.pkl", "wb") as f:
    pickle.dump(df_checkpoint, f)
print(f"\n  → Saved {len(df_checkpoint)} dataframes to checkpoint_dataframes_v2.pkl")

# ==============================================================================
# B. MODEL OBJECTS + ARRAYS — everything needed to regenerate predictions
# ==============================================================================
print("\n" + "=" * 60)
print("SAVING MODELS & ARRAYS")
print("=" * 60)

model_names = [
    # Core RF model + calibration
    "final_rf", "calibration_model", "iso_model",
    # Train/test data
    "X_train_sel", "X_test_sel", "y_train", "y_test",
    # RF predictions (all calibration variants)
    "test_preds_raw", "test_preds_final", "test_preds_isotonic",
    "test_preds_platt_bal", "test_preds_beta",
    "oof_probs", "oof_probs_clamped",
    # Feature tracking
    "stable_features", "feature_names",
    # Cohort ID sets
    "ids_case_true", "ids_history_true",
    # SHAP values
    "shap_vals", "X_shap",
    # Platt + Beta calibration models
    "platt_balanced", "beta_cal",
    # Logistic regression
    "lr", "lr_preds",
    # XGBoost model + predictions
    "xgb_model", "xgb_test_preds_raw", "xgb_test_preds_iso", "xgb_test_preds_beta",
    "xgb_iso_model", "xgb_beta_cal", "xgb_shap_values",
    # Interaction regression model
    "interaction_model",
    # Sensitivity analysis (downsampled)
    "ds_rf", "ds_iso", "ds_test_raw", "ds_test_iso",
    # Thresholds (save as dict)
]

model_checkpoint = {}
for name in model_names:
    obj = globals().get(name)
    if obj is not None:
        model_checkpoint[name] = obj
        if hasattr(obj, 'shape'):
            print(f"  ✅ {name}: {obj.shape}")
        elif isinstance(obj, (set, list)):
            print(f"  ✅ {name}: {len(obj)} items")
        else:
            print(f"  ✅ {name}: {type(obj).__name__}")

# Save derived thresholds so we don't need to recompute
try:
    model_checkpoint["_thresholds"] = {
        "youden_thresh": youden_thresh,
        "youden_sens": youden_sens,
        "youden_spec": youden_spec,
    }
    print(f"  ✅ _thresholds: youden_thresh={youden_thresh:.4f}")
except NameError:
    print("  ⚠️  Threshold variables not found — skipping")

with open(f"{save_dir}/checkpoint_models_v2.pkl", "wb") as f:
    pickle.dump(model_checkpoint, f)
print(f"\n  → Saved {len(model_checkpoint)} objects to checkpoint_models_v2.pkl")

# ==============================================================================
# C. RAW FAERS TABLES — needed for TTO, volcano plots, comorbidity heatmaps
# ==============================================================================
print("\n" + "=" * 60)
print("SAVING RAW FAERS TABLES (parquet for speed)")
print("=" * 60)

faers_dir = f"{save_dir}/faers_tables"
os.makedirs(faers_dir, exist_ok=True)

for name in ["demo", "drug", "ther", "indic", "outcome", "indications_compositeid"]:
    obj = globals().get(name)
    if obj is not None and hasattr(obj, 'to_parquet'):
        path = f"{faers_dir}/{name}.parquet"
        obj.to_parquet(path, index=False)
        print(f"  ✅ {name}: {obj.shape} → {path}")

# ==============================================================================
# D. FIGURES — copy all PNGs to Drive
# ==============================================================================
fig_dir = "/content/drive/MyDrive/FAERS Files/Figures"
os.makedirs(fig_dir, exist_ok=True)
for png in glob.glob("*.png"):
    os.system(f'cp "{png}" "{fig_dir}/"')
    print(f"  📊 {png}")

print(f"\n{'=' * 60}")
print(f"✅ COMPREHENSIVE CHECKPOINT COMPLETE")
print(f"  Dataframes: {save_dir}/checkpoint_dataframes_v2.pkl")
print(f"  Models:     {save_dir}/checkpoint_models_v2.pkl")
print(f"  FAERS:      {faers_dir}/")
print(f"  Figures:    {fig_dir}/")
print(f"{'=' * 60}")

In [ ]:
# =====================================================================
# CELL 3 (RUN ONCE, after CELL 2) — SAVE AN AUGMENTED CHECKPOINT
# Bundles the fitted models + test arrays into one file so future sessions
# skip re-fitting (CELL 2). Writes LOCALLY first, then copies to Drive
# (avoids the FUSE hang that truncated the old models_v2 checkpoint).
# =====================================================================
import pickle, os, shutil
from pathlib import Path

# everything the concordance + Figure 4 cells need (skip the big training matrix)
WANT = ["final_rf", "lr", "xgb_model", "ebm", "enet_cv",
        "enet_coefs", "ebm_imp", "xgb_shap_importance", "xgb_shap_values", "lr_coefs",
        "X_test_sel", "y_test", "stable_features", "shap_vals", "X_shap",
        "test_preds_raw", "test_preds_final"]

blob, skipped = {}, []
for n in WANT:
    obj = globals().get(n, None)
    if obj is None:
        skipped.append(n); continue
    try:
        pickle.dumps(obj); blob[n] = obj
    except Exception as e:
        skipped.append(f"{n} (unpicklable: {str(e)[:40]})")

local = Path("/content/checkpoint_models_full.pkl")
with open(local, "wb") as f:
    pickle.dump(blob, f, protocol=pickle.HIGHEST_PROTOCOL)
size_mb = local.stat().st_size / 1e6
print(f"wrote local {local} ({size_mb:.0f} MB) with {len(blob)} objects")
if skipped: print("  not saved (missing/unpicklable):", skipped)

# verify the local file loads before copying (guards against truncation)
with open(local, "rb") as f:
    check = pickle.load(f)
assert set(check) == set(blob), "local file failed verification!"
print("  local file verified OK")

# copy to Drive
drive_dir = Path("/content/drive/MyDrive/FAERS Files/Checkpoints")
drive_dir.mkdir(parents=True, exist_ok=True)
dst = drive_dir / "checkpoint_models_full.pkl"
shutil.copy(local, dst)
print(f"✅ copied to Drive: {dst} ({dst.stat().st_size/1e6:.0f} MB)")
print("\nNext session: run CELL 1 (it will auto-load this file) and skip CELL 2.")

## Resume Point — reload on a fresh kernel
Run this instead of the pipeline to restore every object saved above, then jump to the figure cells.

In [ ]:
# =====================================================================
# CELL 1 — SETUP + CHECKPOINT LOAD  (run first, on a fresh kernel)
# Installs the packages the later cells need (not all preinstalled in Colab),
# mounts Drive, and restores the saved dataframes + models + arrays:
#   final_rf, lr, X_train_sel, X_test_sel, y_train, y_test,
#   stable_features, shap_vals, X_shap, and more.
# Replaces the need to run the pipeline's Section 0A/0C.
# =====================================================================
import importlib, subprocess, sys
for pkg in ("interpret", "betacal", "xgboost", "shap"):
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import pickle, os
from pathlib import Path
import pandas as pd

# mount Drive if needed
if not Path("/content/drive").exists():
    from google.colab import drive; drive.mount("/content/drive")

save_dir   = Path("/content/drive/MyDrive/FAERS Files/Checkpoints")
model_path = save_dir / "checkpoint_models_full.pkl"      # written by the checkpoint-save cell above
df_path    = save_dir / "checkpoint_dataframes_v2.pkl"  # written by the checkpoint-save cell above

# Only the models checkpoint is required (final_rf, lr, X_*_sel, y_*, stable_features,
# shap_vals, X_shap). Skip the 2.7 GB dataframes file that was making setup take >10 min.
LOAD_BIG_DATAFRAMES = True

targets = [("models", model_path)]
if LOAD_BIG_DATAFRAMES:
    targets.append(("dataframes", df_path))

for label, path in targets:
    if path.exists():
        print(f"loading {label} ({path.stat().st_size/1e6:.0f} MB) ...")
        with open(path, "rb") as f:
            blob = pickle.load(f)
        for name, obj in blob.items():
            globals()[name] = obj
        print(f"✅ loaded {label}: {len(blob)} objects")
    else:
        print(f"⚠️  MISSING {label} checkpoint at {path}")

# what CELL 2 + the concordance work need to already be present
need = ["final_rf", "lr", "X_train_sel", "X_test_sel", "y_train", "y_test",
        "stable_features", "shap_vals", "X_shap"]
status = {v: v in globals() for v in need}
print("\nReady:", status)
missing = [v for v, ok in status.items() if not ok]
print("All pipeline objects restored — you can jump straight to the figure cells." if not missing else f"Still missing (checkpoint incomplete): {missing}")